# B7 · Du notebook à la production — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b7_production/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B7.Q1 à B7.Q11, B7.R1 à B7.R3, B7.1 à B7.4 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B7.9 | Du notebook au paquet : data.py, model.py, train.py, pyproject.toml et installation éditable | 📦 | ★★★ | 60 |
|  | B7.10 | Une interface en ligne de commande : argparse, logging et codes de sortie | 📦 | ★★ | 30 |
|  | B7.11 | Configuration : dataclass + YAML, surcharges en ligne de commande, validation | 📦 | ★★ | 30 |
|  | B7.12 | Prédire : même graine, puis num_workers = 4, puis GPU — résultats identiques ? | 🔮 | ★ | 10 |
|  | B7.13 | Mesurer le non-déterminisme : graines fixées ou non, algorithmes déterministes, cinq exécutions | 🔬 | ★★ | 30 |
|  | B7.14 | Suivi d'expériences maison : identifiant de run, configuration, métriques et artefacts en JSONL | 📦 | ★★ | 30 |
|  | B7.15 | Tests pytest pour le ML : formes, overfitting d'un mini-batch, prétraitement, schéma des données | 📦 | ★★★ | 45 |
| B | B7.16 | Le modèle qui marche dans le notebook et pas en production : cinq écarts à retrouver | 🐛 | ★★ | 30 |
| B | B7.17 à B7.21 | le service (suite) : l'artefact, l'API, la latence, l'accélération sur CPU et l'export (*prochaine session de génération*) | | | |
| C | B7.22 à B7.25 | l'exploitation : la dérive, l'intégration continue, le Dockerfile et le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Restructurer un notebook en paquet Python installable, avec une ligne de commande, du logging et une configuration validée.
- Rendre un entraînement reproductible et traçable (graines, déterminisme, versions, empreinte des données, journal des runs), et rapporter un score sur plusieurs graines avec son intervalle.
- Écrire des tests utiles à un projet de ML et les automatiser dans une intégration continue.
- Empaqueter un modèle avec son prétraitement, le servir par une API HTTP et mesurer sa latence.
- Exporter et accélérer l'inférence sur CPU en vérifiant l'équivalence numérique ; détecter une dérive des données et décider d'une action.

**Rappel express.** Intervalle sur $n$ graines : $\bar a \pm t_{0{,}975;\,n-1}\,\frac{s}{\sqrt n}$ ($s$ divisé par $n - 1$). Percentile de `np.percentile` : position $h = \frac{q}{100}(n - 1)$ dans les mesures triées, interpolation linéaire. Débit $= \frac{\text{taille du batch}}{\text{latence du batch}}$ ; loi de Little $L = \lambda W$. Taille des poids $= N \times$ octets par paramètre (4, 2, 1). $\operatorname{PSI} = \sum_k (p_k - q_k)\ln\frac{p_k}{q_k} = D_{\text{KL}}(p\,\|\,q) + D_{\text{KL}}(q\,\|\,p)$ ; $D = \sup_x |F_{\text{réf}}(x) - F_{\text{prod}}(x)|$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B7.1 à B7.4)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres ensemble, entre guillemets, dans n'importe quel ordre (`"EF"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », l'estimation 🧮 B7.5, la réflexion (🗣️ B7.6, ⚖️ B7.7, 📄 B7.8) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np

MB, MIB = 1e6, 2 ** 20                  # 1 Mo = 10**6 bytes, 1 Mio = 2**20 bytes (fiche, 207.8)

# quiz Q1: the cells x = 2 ; x = x * 10 ; print(x + 1), run in the order 1, 2, 3, 2, 3
Q1A = 2 * 10 * 10 + 1
# quiz Q3: basicConfig(level=WARNING), then 1 debug, 3 info, 1 warning and 1 error messages
LEVELS_Q3 = ["DEBUG"] + ["INFO"] * 3 + ["WARNING", "ERROR"]
Q3C = sum(level in ("WARNING", "ERROR", "CRITICAL") for level in LEVELS_Q3)
# quiz Q7: the ML Test Score is the minimum of the four section sums
SECTIONS_Q7 = {"data": 3.0, "model": 2.5, "infrastructure": 4.0, "monitoring": 1.0}
Q7E = min(SECTIONS_Q7.values())
# quiz Q9: 900 requests of 10 ms and 100 of 200 ms ; Little's law ; a throughput
LAT_Q9 = np.array([10.0] * 900 + [200.0] * 100)
Q9A, Q9B = float(LAT_Q9.mean()), float(np.percentile(LAT_Q9, 95))
Q9D = 60 * 0.150                         # L = lambda W, W in seconds
Q9E = 64 / 0.4
# recall R1: a JSONL log of 7 runs, 2 runs added (append, never rewrite)
R1D = 7 + 2

# ✏️ B7.1: ConvNeXt-Tiny (torchvision) and the VGG of the MP5 solution
N_B71, LIN_W_B71, LIN_B_B71 = 28_589_128, 26_646_528, 34_120
F32_B71 = 4 * N_B71 / MB
F32_MIB_B71 = 4 * N_B71 / MIB
BF16_B71 = 2 * N_B71 / MB
INT8_B71 = N_B71 / MB
DYNQ_B71 = (LIN_W_B71 + 4 * (N_B71 - LIN_W_B71)) / MB          # dense matrices in int8, everything else float32
RATIO_B71 = F32_B71 / DYNQ_B71
PEAK_B71 = 2 * F32_MIB_B71                                     # random weights + the copy read by torch.load
VGG_PARAMS_B71, VGG_LINEAR_B71, VGG_FILE_B71 = 72_954, 650, 296_464
VGG_BN_CHANNELS_B71 = (16, 16, 32, 32, 64, 64)
VGG_FLOAT_BUFFERS_B71 = 2 * sum(VGG_BN_CHANNELS_B71)            # running_mean and running_var
VGG_BYTES_B71 = 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71) + 8 * len(VGG_BN_CHANNELS_B71)
HEADER_B71 = VGG_FILE_B71 - VGG_BYTES_B71
LIN_SHARE_B71 = 100 * VGG_LINEAR_B71 / VGG_PARAMS_B71

# ✏️ B7.2: five seeds, Student t with n - 1 = 4 degrees of freedom (the table of 207.4)
A_B72 = np.array([64.2, 65.0, 63.6, 64.8, 64.4])
B_B72 = np.array([64.9, 65.3, 64.5, 65.6, 64.7])
T4_B72 = 2.776


def interval_b72(values, t=T4_B72, ddof=1):
    """Mean, standard deviation, standard error and half-width of the Student interval."""
    m, s = float(np.mean(values)), float(np.std(values, ddof=ddof))
    se = s / math.sqrt(len(values))
    return m, s, se, t * se


MEAN_A_B72, S_A_B72, SE_A_B72, HW_A_B72 = interval_b72(A_B72)
LOW_A_B72 = MEAN_A_B72 - HW_A_B72
MEAN_B_B72, _, _, HW_B_B72 = interval_b72(B_B72)
OVERLAP_B72 = bool(MEAN_A_B72 + HW_A_B72 >= MEAN_B_B72 - HW_B_B72)
MEAN_D_B72, S_D_B72, SE_D_B72, HW_D_B72 = interval_b72(B_B72 - A_B72)
ZERO_IN_D_B72 = bool(MEAN_D_B72 - HW_D_B72 <= 0 <= MEAN_D_B72 + HW_D_B72)

# ✏️ B7.3: twenty latencies of batch 1 (ms), np.percentile's default (method="linear")
LAT_B73 = np.array([21.3, 19.8, 20.4, 22.1, 19.6, 20.9, 20.1, 35.7, 20.6, 19.9,
                    21.0, 20.2, 23.4, 19.7, 20.5, 47.9, 20.8, 20.3, 21.6, 20.0])
MEAN_B73 = float(LAT_B73.mean())
P50_B73, P95_B73 = (float(v) for v in np.percentile(LAT_B73, [50, 95]))
ABOVE_B73 = int((LAT_B73 > MEAN_B73).sum())
P50_32_B73, P95_32_B73 = 118.0, 131.0                          # batch 32, given by the statement
THR1_B73 = 1000 / P50_B73
THR32_B73 = 32 * 1000 / P50_32_B73
RATIO_B73 = THR32_B73 / THR1_B73
PER_IMAGE_B73 = P50_32_B73 / 32
SLO_B73 = bool(P95_32_B73 <= 100)
PAGE_B73 = 1 - 0.95 ** 8

# ✏️ B7.4: PSI on four intervals (the quartiles of the reference), natural log, epsilon added to every share
Q_B74 = np.full(4, 0.25)
COUNTS_B74 = {1: [38, 52, 54, 56], 2: [96, 62, 30, 12], 3: [120, 56, 24, 0]}
P_B74 = {week: np.array(c) / 200 for week, c in COUNTS_B74.items()}


def psi_terms_b74(p, q=Q_B74, eps=0.0, log=np.log):
    """The four terms (p_k - q_k) log(p_k / q_k), eps added to every share of p and q (without renormalizing)."""
    p, q = np.asarray(p, float) + eps, np.asarray(q, float) + eps
    return (p - q) * log(p / q)


C1_W1_B74 = float(psi_terms_b74(P_B74[1])[0])
PSI_W1_B74 = float(psi_terms_b74(P_B74[1]).sum())
PSI_W2_B74 = float(psi_terms_b74(P_B74[2]).sum())
TOP_W2_B74 = "ABCD"[int(np.argmax(psi_terms_b74(P_B74[2])))]
KL_PQ_W2_B74 = float(np.sum(P_B74[2] * np.log(P_B74[2] / Q_B74)))
KL_QP_W2_B74 = float(np.sum(Q_B74 * np.log(Q_B74 / P_B74[2])))
PSI_W3_B74 = float(psi_terms_b74(P_B74[3], eps=1e-4).sum())
PSI_W3_EPS3_B74 = float(psi_terms_b74(P_B74[3], eps=1e-3).sum())
# a classic mistake for i): epsilon added to the production shares only (or to the empty share only: same at 3 decimals)
PSI_W3_EPS3_PART_B74 = float(((P_B74[3] + 1e-3 - Q_B74) * np.log((P_B74[3] + 1e-3) / Q_B74)).sum())

# The same answers from the libraries (no download): NumPy, SciPy, torchvision on the `meta` device, safetensors,
# PyYAML, logging, argparse, subprocess, DataLoader, torch.export, torch.load and scikit-learn
import io
import json
import logging
import subprocess
import sys
import tempfile
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import yaml
from scipy import stats

# quiz Q1: run the three cells in the order 1, 2, 3, 2, 3 and keep what the last one prints
cells, namespace, printed = {1: "x = 2", 2: "x = x * 10", 3: "shown = x + 1"}, {}, None
for k in (1, 2, 3, 2, 3):
    exec(cells[k], namespace)
assert namespace["shown"] == Q1A

# quiz Q3: a WARNING level lets through WARNING and ERROR only ; an exception swallowed by print ends with code 0
records = []
logger = logging.getLogger("wb_chB7_q3")
logger.propagate = False
handler = logging.Handler(level=logging.WARNING)
handler.emit = records.append
logger.addHandler(handler)
logger.setLevel(logging.DEBUG)
for level in LEVELS_Q3:
    logger.log(getattr(logging, level), "message")
logger.removeHandler(handler)
assert len(records) == Q3C
swallow = "try:\n    raise RuntimeError('epoch 2: loss is nan')\nexcept Exception as err:\n    print(err)\n"
assert subprocess.run([sys.executable, "-c", swallow], capture_output=True).returncode == 0
unknown = "import argparse; p = argparse.ArgumentParser(); p.add_argument('--lr'); p.parse_args(['--lerning-rate', '1'])"
assert subprocess.run([sys.executable, "-c", unknown], capture_output=True).returncode == 2

# quiz Q4: PyYAML 6.0 reads 5.0e-4 as a float, 5e-4 and "5.0e-4" as strings ; a dataclass does not check types
assert yaml.safe_load("lr: 5.0e-4")["lr"] == 0.0005
assert yaml.safe_load("lr: 5e-4")["lr"] == "5e-4" and yaml.safe_load('lr: "5.0e-4"')["lr"] == "5.0e-4"


@dataclass(frozen=True)
class FrozenOnly:
    lr: float = 0.05


assert FrozenOnly(lr="1e-3").lr == "1e-3"                       # accepted: no check of the annotation

# quiz Q5: float addition is not associative ; without a generator, an extra draw moves the DataLoader's order
assert ((0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3)) is False
import torch
from torch.utils.data import DataLoader


def first_order(extra_draw):
    torch.manual_seed(0)
    if extra_draw:
        torch.rand(1)
    return next(iter(DataLoader(list(range(10)), batch_size=10, shuffle=True))).tolist()


assert first_order(False) != first_order(True)

# quiz Q7, Q9 and recall R1
assert Q7E == 1.0 and Q9A == 29.0 and Q9B == 200.0 and math.isclose(Q9D, 9.0) and math.isclose(Q9E, 160.0)
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "runs.jsonl"
    for seed in list(range(7)) + [10, 11]:                     # 7 runs, then 2 runs again with other seeds
        with path.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"seed": seed}) + "\n")
    assert len(path.read_text(encoding="utf-8").splitlines()) == R1D
assert datetime.strptime("20261104-161731", "%Y%m%d-%H%M%S").strftime("%H:%M:%S") == "16:17:31"

# quiz Q8: load_state_dict refuses tensors of another shape ; safetensors metadata must be strings
narrow, wide = torch.nn.Linear(12, 3), torch.nn.Linear(16, 3)
try:
    wide.load_state_dict(narrow.state_dict())
except RuntimeError as err:
    assert "size mismatch" in str(err)
else:
    raise AssertionError("load_state_dict accepted tensors of another shape")
from safetensors.torch import save as save_safetensors

try:
    save_safetensors({"w": torch.zeros(2)}, metadata={"version": 1})
except TypeError:
    pass
else:
    raise AssertionError("safetensors accepted a metadata value that is not a string")

# quiz Q10: an export without dynamic_shapes refuses another batch size ; a data-dependent if does not export
small = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(12, 3)).eval()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    exported = torch.export.export(small, (torch.randn(8, 3, 2, 2),)).module()
    try:
        exported(torch.randn(3, 3, 2, 2))
    except Exception as err:                                    # « Guard failed: input.size()[0] == 8 »
        assert "8" in str(err)
    else:
        raise AssertionError("the exported model accepted a batch of 3")


class DataDependent(torch.nn.Module):
    def forward(self, x):
        if x.mean() > 0:
            return 2 * x
        return x - 1


import contextlib

torch_loggers = ("torch", "torch.fx.experimental.symbolic_shapes", "torch._dynamo", "torch.export")
saved_levels = {name: logging.getLogger(name).level for name in torch_loggers}
try:                                                            # torch.export prints the trace that failed
    for name in torch_loggers:
        logging.getLogger(name).setLevel(logging.CRITICAL)
    with contextlib.redirect_stderr(io.StringIO()), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.export.export(DataDependent(), (torch.randn(4, 3),))
except Exception as err:
    assert "data-dependent" in str(err), str(err)[:200]
else:
    raise AssertionError("a data-dependent if was exported")
finally:
    for name, level in saved_levels.items():
        logging.getLogger(name).setLevel(level)

# recall R2: weights_only=True refuses an arbitrary Python object


class Trainer:
    pass


buffer = io.BytesIO()
torch.save({"trainer": Trainer()}, buffer)
buffer.seek(0)
try:
    torch.load(buffer, weights_only=True)
except Exception:
    pass
else:
    raise AssertionError("weights_only=True loaded a Python object")

# recall R3: the warning of scikit-learn when a model was written by another version
from sklearn.exceptions import InconsistentVersionWarning

assert issubclass(InconsistentVersionWarning, UserWarning)

# ✏️ B7.1: ConvNeXt-Tiny counted on the `meta` device (no memory, no download) ; the MP5 VGG and its safetensors
from torchvision.models import convnext_tiny

with torch.device("meta"):
    convnext = convnext_tiny(weights=None)
linear = [m for m in convnext.modules() if isinstance(m, torch.nn.Linear)]
assert sum(p.numel() for p in convnext.parameters()) == N_B71 and not list(convnext.buffers())
assert sum(m.weight.numel() for m in linear) == LIN_W_B71 and sum(m.bias.numel() for m in linear) == LIN_B_B71


def vgg_mp5(widths=(16, 32, 64)):
    """The VGG of the MP5 solution (cifarkit.models.vgg with downsample="pool" and batchnorm)."""
    layers, c = [], 3
    for i, w in enumerate(widths):
        for _ in range(2):
            layers += [torch.nn.Conv2d(c, w, 3, padding=1, bias=False), torch.nn.BatchNorm2d(w), torch.nn.ReLU()]
            c = w
        if i < len(widths) - 1:
            layers.append(torch.nn.MaxPool2d(2))
    layers += [torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten(), torch.nn.Linear(c, 10)]
    return torch.nn.Sequential(*layers)


vgg = vgg_mp5()
state = vgg.state_dict()
assert sum(p.numel() for p in vgg.parameters()) == VGG_PARAMS_B71
assert sum(t.numel() * t.element_size() for t in state.values()) == VGG_BYTES_B71
assert len(save_safetensors(state)) == VGG_FILE_B71
assert sum(p.numel() for p in vgg[-1].parameters()) == VGG_LINEAR_B71

# ✏️ B7.2: the Student intervals of SciPy (exact quantile 2.7764 instead of the 2.776 of the table)
assert math.isclose(stats.t.ppf(0.975, 4), T4_B72, abs_tol=5e-4)
ci_a = stats.ttest_1samp(A_B72, 0).confidence_interval()
ci_d = stats.ttest_rel(B_B72, A_B72).confidence_interval()
assert round(float(ci_a.low), 2) == round(LOW_A_B72, 2)
assert round(float(ci_d.high - ci_d.low) / 2, 2) == round(HW_D_B72, 2) and (float(ci_d.low) > 0) == (not ZERO_IN_D_B72)

# ✏️ B7.3: the other conventions of np.percentile, for the record
assert np.percentile(LAT_B73, 95, method="inverted_cdf") == 35.7 and np.percentile(LAT_B73, 50, method="inverted_cdf") == 20.5

# ✏️ B7.4: PSI = KL(p || q) + KL(q || p), with scipy.stats.entropy
for week in (1, 2):
    kl = stats.entropy(P_B74[week], Q_B74) + stats.entropy(Q_B74, P_B74[week])
    assert math.isclose(kl, float(psi_terms_b74(P_B74[week]).sum()))
assert math.isclose(stats.entropy(Q_B74, P_B74[2]), KL_QP_W2_B74)
with np.errstate(divide="ignore", invalid="ignore"):
    assert np.isinf(psi_terms_b74(P_B74[3]).sum())
print("✅ the paper answers agree with NumPy, SciPy, torchvision, safetensors, PyYAML, logging, subprocess and torch")

✅ the paper answers agree with NumPy, SciPy, torchvision, safetensors, PyYAML, logging, subprocess and torch


**B7.Q1 — Les pièges du notebook : état caché et ordre d'exécution**

In [3]:
wb.record("B7.Q1a", Q1A, mistakes={"le noyau garde l'ordre où tu as exécuté les cellules, pas celui où elles sont écrites : refais la suite 1, 2, 3, 2, 3": 21,
          "compte les exécutions de la cellule 2 dans la suite 1, 2, 3, 2, 3": 2001,
          "relis la cellule 3 : elle affiche x + 1": 200})
wb.record("B7.Q1b", "A", mistakes={"« Redémarrer » vide le noyau : d'où viendrait encore scale ?": "B", "Jupyter n'exécute que les cellules du fichier : relis « Ce que cache un notebook » (§207.1)": "C"})
wb.record("B7.Q1c", False, mistakes={"un .ipynb range code, sorties et images en JSON : relis « Ce que cache un notebook » (§207.1)": True})
wb.record("B7.Q1d", "C", mistakes={"un module déjà importé est-il relu par le noyau ? Relis la fin de « pyproject.toml et installation éditable »": "A", "le lien éditable existe déjà : est-ce l'installation ou le noyau qui garde l'ancienne version ?": "B"})
wb.record("B7.Q1e", True, mistakes={"supprimer une cellule touche-t-il au noyau ? Relis « Ce que cache un notebook » (§207.1)": False})

WB_ANSWER {"hash": "c60b08e131923f38ef20943d6b7a1b80374e12ff8cc80644acb9d63e54cc97d1", "id": "B7.Q1a", "kind": "int", "magnitude": 2, "mistakes": {"949a78b1c4a3833b78c74333fd5106725777ba5f3d00068d1f08f9ff77b95a62": "relis la cellule 3 : elle affiche x + 1", "a77b3ed193a23dbbff0a1e26a855f366e7d9c0939abd011378dc1763a26a9274": "compte les exécutions de la cellule 2 dans la suite 1, 2, 3, 2, 3", "bd2f1ef5e7c54a0d32ef18b594977477a0a049b9e3a47d97e2d0733b28bb569d": "le noyau garde l'ordre où tu as exécuté les cellules, pas celui où elles sont écrites : refais la suite 1, 2, 3, 2, 3"}, "sign": 1}
📝 Ex B7.Q1a : réponse enregistrée (int).
WB_ANSWER {"hash": "f0285ae738d3da1a2fbf9ad60d0577d9d68870fb765ab2cda5304442cf59b85d", "id": "B7.Q1b", "kind": "str", "mistakes": {"1ec5ee77a9be8d1ab65a86b22ae9c25d62f3da1345f83b095c697d1eea81c4c1": "Jupyter n'exécute que les cellules du fichier : relis « Ce que cache un notebook » (§207.1)", "5573f62f405a12424a023bd031c0966578c56f98a6e021221f1e68af77d8b4ae": "

**B7.Q2 — src/, tests/, configs/, scripts/ : où va quoi ?**

In [4]:
wb.record("B7.Q2a", "A", mistakes={"un fichier de tests vérifie le code ; l'API, elle, importe cette fonction": "B", "configs/ ne contient que des réglages, pas du code": "C", "scripts/ range ce qu'on lance à la main ; cette fonction est importée par l'entraînement et l'API": "D"})
wb.record("B7.Q2b", "D", mistakes={"src/cifarprod/ ne contient que le code qu'importent l'entraînement et le service : ce script, on le lance": "A", "ce script ne vérifie pas le code, il prépare les données": "B", "configs/ ne contient que des réglages, pas du code": "C"})
wb.record("B7.Q2c", "C", mistakes={"un fichier YAML de réglages n'est pas du code importable": "A", "ce fichier décrit un entraînement ; il ne teste rien lui-même": "B", "scripts/ range du code à lancer ; un fichier YAML est un réglage": "D"})
wb.record("B7.Q2d", "B", mistakes={"src/cifarprod/ ne contient que le code qui sert en production": "A", "configs/ ne contient que des réglages": "C", "où pytest cherche-t-il les tests d'un dépôt ? Relis l'arborescence de 207.1": "D"})
wb.record("B7.Q2e", False, mistakes={"relis le paragraphe qui suit l'arborescence (§207.1) : les données et les sorties des runs vont-elles dans git ?": True})
wb.record("B7.Q2f", "B", mistakes={"relis l'avantage du layout src/ (§207.1) : le dossier courant suffit-il pour importer cifarprod ?": "A", "pytest installe-t-il quoi que ce soit ? Relis le layout src/ (§207.1)": "C"})
wb.record("B7.Q2g", True, mistakes={"relis [project.scripts] (§207.1) : que crée cette ligne à l'installation ?": False})

WB_ANSWER {"hash": "9d95322120c26b2afecc429aee86d7dfb20ed23162239e00beb4e291a0f3c6ad", "id": "B7.Q2a", "kind": "str", "mistakes": {"6946269fef036f59ae3dcc288c370126f6e301c3910a40d3fb47cbafe21fc46a": "un fichier de tests vérifie le code ; l'API, elle, importe cette fonction", "c2a337d958c229641b498eb4d4f5bdd0bcab5766e69620615c671d2eaa710228": "scripts/ range ce qu'on lance à la main ; cette fonction est importée par l'entraînement et l'API", "f4fe70d4259a06e0559463b0ff67b84c8e98fa5679b1f5cbf290622dfa08be21": "configs/ ne contient que des réglages, pas du code"}}
📝 Ex B7.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "514e71f81aa6170e1155b0c7d2842489151d6a6bf6a9d9c9d78561625bf41552", "id": "B7.Q2b", "kind": "str", "mistakes": {"71c71ac893b0e46a8ecf90f3fb62abb5a7d78425ade56d35208c7a61065cea72": "ce script ne vérifie pas le code, il prépare les données", "c48c5cd96af9c5e35ec0b9fa010485a39cf273ce87b51ef7a40ed578081c9531": "src/cifarprod/ ne contient que le code qu'importent l'entraînem

**B7.Q3 — print ou logging : niveaux, destinations et messages utiles**

In [5]:
wb.record("B7.Q3a", "B", mistakes={"DEBUG sert au détail pour chercher un bug : l'écriture de l'artefact en est-elle un ?": "A", "WARNING signale l'inattendu : écrire l'artefact à la fin d'un run réussi l'est-il ?": "C", "ERROR dit qu'une opération a échoué : relis les niveaux (§207.2)": "D"})
wb.record("B7.Q3b", "C", mistakes={"un entraînement dix fois plus lent que prévu : un simple détail de débogage ?": "A", "est-ce une étape normale, ou quelque chose d'inattendu qui n'empêche pas de continuer ?": "B", "le programme continue : une opération a-t-elle échoué ?": "D"})
wb.record("B7.Q3c", Q3C, mistakes={"un niveau WARNING laisse-t-il passer les messages INFO ? Relis les niveaux (§207.2)": 5,
          "un message ERROR est plus grave qu'un WARNING : passe-t-il le niveau choisi ?": 1,
          "un niveau WARNING laisse-t-il passer DEBUG et INFO ?": 6})
wb.record("B7.Q3d", False, mistakes={"relis « Le logging plutôt que print » (§207.2) : où configure-t-on les handlers, et combien de fois ?": True})
wb.record("B7.Q3e", "C", mistakes={"quel fait vérifiable ce message donne-t-il ? Relis la fin de « Le logging plutôt que print »": "A", "un message utile donne des valeurs : lequel en donne ?": "B"})
wb.record("B7.Q3f", 0, mistakes={"le except affiche l'erreur, puis le script va au bout : comment finit-il ? Relis « Les codes de sortie » (§207.2)": 1,
          "2 est le code d'argparse devant une mauvaise option : relis « Les codes de sortie »": 2})

WB_ANSWER {"hash": "0bed5eeb746dc46f5044971c9c82cc3839eb68b565284af89486f9928db10f51", "id": "B7.Q3a", "kind": "str", "mistakes": {"789ae652c581b62fa26ccb3cf0e3c07491b9d4caeb7cc1795703636d58136da4": "DEBUG sert au détail pour chercher un bug : l'écriture de l'artefact en est-elle un ?", "84066a21f5ba6b074d6966c8713fc9d534f81766e25be343a986447cb263022a": "ERROR dit qu'une opération a échoué : relis les niveaux (§207.2)", "bf34158db617480ac67bea97c5e3546ec65cf1f3bc0e2a36285890fd09aabe7f": "WARNING signale l'inattendu : écrire l'artefact à la fin d'un run réussi l'est-il ?"}}
📝 Ex B7.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "289569a0791c35ab21c0a9dc7ae10c60c244ce87412985c1f9e044267d36571a", "id": "B7.Q3b", "kind": "str", "mistakes": {"4d9704d3e7e14f4c62f15ce7d7f0601a3cf8001686806b15a9e245672fda1f4f": "est-ce une étape normale, ou quelque chose d'inattendu qui n'empêche pas de continuer ?", "7ffc12c4beb2f76368b6c2acaf80e4617884fd6f93d9e33a21a089da9b1aa598": "un entraînement dix 

**B7.Q4 — Ce qui va dans la configuration, et ce qui ne doit jamais y aller**

In [6]:
wb.record("B7.Q4a", "ACD", mistakes={"il manque au moins une lettre : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'A', 'AD', 'D', 'CD', 'AC', 'C'}, "au moins une des lettres choisies n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)": {'ABD', 'BCD', 'BD', 'ABC', 'AB', 'B', 'BC'}})
wb.record("B7.Q4b", "A", mistakes={"relis le rappel outil sur PyYAML (§207.3) : que faut-il pour qu'un nombre en notation scientifique soit lu comme un float ?": "B", "des guillemets donnent-ils un nombre ou une chaîne en YAML ?": "C"})
wb.record("B7.Q4c", 2, mistakes={"relis main de la fiche (§207.2) : que rend-il quand load_config lève une ValueError ?": 0,
          "1 est le code d'une erreur générale : relis le except de main (§207.2)": 1})
wb.record("B7.Q4d", True, mistakes={"relis « Variables d'environnement et secrets » (§207.9) : que garde une image de chacune de ses couches ?": False})
wb.record("B7.Q4e", "C", mistakes={"un commit suivant efface-t-il le précédent de l'historique ?": "A", "un dépôt privé se clone, se partage et peut devenir public : relis 207.9": "B"})
wb.record("B7.Q4f", False, mistakes={"relis le commentaire de __post_init__ dans TrainConfig (§207.3) : une dataclass vérifie-t-elle les types ?": True})

WB_ANSWER {"hash": "f9e9974f6deb83829f93e8c1c6a7e5cbad64b77222b268d3521250feede0727e", "id": "B7.Q4a", "kind": "str", "mistakes": {"1190f13b051eaaab8197b43aba801b77a6624a80264b61a92616321463fc30d9": "il manque au moins une lettre, et une autre n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)", "397b7fe7a842664f1ff1b78876b3da18ff28bec84a3ba529f9abc72182693cd3": "au moins une des lettres choisies n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)", "48f4a199796cee35ae33d9aa3c18d391f7c9e2ee8c59d70ef3b651bcccd51dc0": "il manque au moins une lettre, et une autre n'a pas sa place : relis 207.3 et « Variables d'environnement et secrets » (§207.9)", "4d75aad1bc629775119b42eb39b19f6df248fdefffc87d66e26e491c3e5bcbb6": "il manque au moins une lettre : relis 207.3 et « Variables d'environnement et secrets » (§207.9)", "a13a92ca1adbb165b97d9cad27fcc42ed748c48a026955b2d1fca48297a97879": "il manque au moins une lettre, et une autre n

**B7.Q5 — Même graine, résultats différents : les sources de non-déterminisme**

In [7]:
wb.record("B7.Q5a", "C", mistakes={"l'initialisation de nn.Linear tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?": "A", "le dropout tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?": "B"})
wb.record("B7.Q5b", True, mistakes={"relis le paragraphe qui suit seed_everything (§207.4) : l'addition des flottants est-elle associative ?": False})
wb.record("B7.Q5c", "C", mistakes={"c'est le comportement sans warn_only : relis l'encadré 🕰️ de 207.4": "A", "PyTorch ne remplace pas une opération par une autre : relis l'encadré 🕰️ de 207.4": "B"})
wb.record("B7.Q5d", False, mistakes={"calcule les deux sommes en Python : l'addition des flottants est-elle associative ?": True})
wb.record("B7.Q5e", "B", mistakes={"sans generator, où le DataLoader tire-t-il son ordre ? Relis le paragraphe qui suit seed_everything (§207.4)": "A", "un tirage de plus n'est pas une erreur : que change-t-il au générateur global ?": "C"})

WB_ANSWER {"hash": "69f94eaf2a8f07454de325c9483a70d22f84c496048c50d6d99823cf5d7db2c1", "id": "B7.Q5a", "kind": "str", "mistakes": {"0a6368ccdd96c4ae4d04088493c16d0cd011bc282379c2850f78554e374be681": "l'initialisation de nn.Linear tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?", "ff884821df42b00a71e006c3dc5fae47012810edda2149029e465df147f9d50a": "le dropout tire dans le générateur de PyTorch : torch.manual_seed le fixe-t-il ?"}}
📝 Ex B7.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "2eae29837880b4b31d6d37273204134e0bcd01650bc9b0939a75a089398643eb", "id": "B7.Q5b", "kind": "bool", "mistakes": {"6a16c1c5db501a5947f26c33a99165af9355999272705134bf468fe6b74cc17f": "relis le paragraphe qui suit seed_everything (§207.4) : l'addition des flottants est-elle associative ?"}}
📝 Ex B7.Q5b : réponse enregistrée (bool).
WB_ANSWER {"hash": "b941a1b6bb8becf159c97f62720b7f1a5bdf675abab61fcf8fae643125ae97b8", "id": "B7.Q5c", "kind": "str", "mistakes": {"aa2060dc6e2e77556af1ff

**B7.Q6 — requirements.txt, fichier de verrouillage, pyproject.toml : qui fige quoi ?**

In [8]:
wb.record("B7.Q6a", "C", mistakes={"une borne >= dit-elle une version exacte ? Relis « pyproject.toml et installation éditable » (§207.1)": "A", "Colab installe ses propres versions : que dit une borne inférieure ?": "B"})
wb.record("B7.Q6b", "A", mistakes={"pyproject.toml donne des bornes : relis « Les versions figées et l'empreinte des données » (§207.4)": "B", "le cache de pip garde des fichiers téléchargés, pas ce qu'un projet a installé : relis « Les versions figées et l'empreinte des données » (§207.4)": "C"})
wb.record("B7.Q6c", False, mistakes={"relis « Les versions figées » (§207.4) : une bibliothèque qu'on publie garde-t-elle des versions exactes ou des bornes ?": True})
wb.record("B7.Q6d", "A", mistakes={"le run a tourné avec du code modifié : ce code est-il tout entier dans a1b2c3d ?": "B", "un commit enregistré reste valide : que manque-t-il au run ?": "C"})
wb.record("B7.Q6e", True, mistakes={"un fichier de verrouillage fige-t-il des versions exactes ou des bornes ? Relis « Les versions figées » (§207.4)": False})

WB_ANSWER {"hash": "e9ea2cb58b3f0684a6a7febacbf7b8996866ab4ba2fb55e72a0a5631e18ff2f7", "id": "B7.Q6a", "kind": "str", "mistakes": {"2ca5919aec7b19fec472e868a6c7cd09e35f87a2751da12334900430c6c1b9b6": "une borne >= dit-elle une version exacte ? Relis « pyproject.toml et installation éditable » (§207.1)", "bcb3be021a46ad62a214399535d7e8ada46a2f21f393e44fe7b06c4732813d60": "Colab installe ses propres versions : que dit une borne inférieure ?"}}
📝 Ex B7.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "7925241a34b466d33c22605ef5c86d64f31120d19be855b267cfa1a5a0c89a4f", "id": "B7.Q6b", "kind": "str", "mistakes": {"6eb84f0d8d155c1afd728becac1ae5779aec58be3b515e3621d700d4a0d0d863": "pyproject.toml donne des bornes : relis « Les versions figées et l'empreinte des données » (§207.4)", "92d8498e61a46696b3833392b90e661fbfb9450e2e144fe968af4112450d76bd": "le cache de pip garde des fichiers téléchargés, pas ce qu'un projet a installé : relis « Les versions figées et l'empreinte des données » (§207

**B7.Q7 — Que tester dans un projet de machine learning ?**

In [9]:
wb.record("B7.Q7a", "B", mistakes={"la forme de la sortie dépend-elle de zero_grad ? Relis « Quatre familles de tests » (§207.6)": "A", "le schéma des données ne voit pas la boucle d'entraînement : relis « Quatre familles de tests »": "C"})
wb.record("B7.Q7b", "C", mistakes={"l'entraînement fumée vérifie que tout va au bout, pas le contenu des données": "A", "apprendre 16 images par cœur dit-il quelque chose du recouvrement des jeux ?": "B"})
wb.record("B7.Q7c", "A", mistakes={"relis la puce « bout en bout » (§207.6) : l'entraînement fumée juge-t-il l'accuracy ?": "B", "l'entraînement fumée lance un entraînement, pas l'API": "C"})
wb.record("B7.Q7d", False, mistakes={"relis la puce « bout en bout » (§207.6) : que vérifie l'entraînement fumée, et que ne juge-t-il pas ?": True})
wb.record("B7.Q7e", Q7E, decimals=1, mistakes={"relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par leur moyenne ?": float(np.mean(list(SECTIONS_Q7.values()))),
          "relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par leur total ?": float(sum(SECTIONS_Q7.values())),
          "relis la grille de Breck et coll. (§207.6) : le score résume-t-il les quatre sommes par la plus grande ?": 4.0})

WB_ANSWER {"hash": "9db503750126ed5d1ed6504018ccc50d7becc6eccdee33f24cef1918e3c0ba8c", "id": "B7.Q7a", "kind": "str", "mistakes": {"2167b17c02d0db19852cb2efabe73524687aaf1a944ef592c2470658580971dc": "le schéma des données ne voit pas la boucle d'entraînement : relis « Quatre familles de tests »", "d8b183720a0aaab12cdfb03050b2ecf850ff407c6769a82d1611c48ed647e556": "la forme de la sortie dépend-elle de zero_grad ? Relis « Quatre familles de tests » (§207.6)"}}
📝 Ex B7.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "f5611ec22f0bdf675a92bed7404ecbf942a70ab6c4fa5f80d15beade923a049f", "id": "B7.Q7b", "kind": "str", "mistakes": {"183a1dd759cf3510a9e6e0818d5ef47b7c5f8f47f06fb6075bc17ce9d3e74382": "apprendre 16 images par cœur dit-il quelque chose du recouvrement des jeux ?", "685954d15fe389783fa98668d3f1071f4ab18b5fa7a87ef63df27662ef7b7a64": "l'entraînement fumée vérifie que tout va au bout, pas le contenu des données"}}
📝 Ex B7.Q7b : réponse enregistrée (str).
WB_ANSWER {"hash": "c551451

**B7.Q8 — Artefact de modèle : les poids seuls ne suffisent pas**

In [10]:
wb.record("B7.Q8a", "AB", mistakes={"il manque au moins une lettre : relis « L'artefact du modèle » (§207.7)": {'B', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis « L'artefact du modèle » (§207.7)": {'ABC', 'ABD', 'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)": {'AD', 'BCD', 'C', 'D', 'BD', 'CD', 'AC', 'ACD', 'BC'}})
wb.record("B7.Q8b", "B", mistakes={"load_state_dict vérifie-t-il les noms et les formes des tenseurs ? Relis §23.9.2 du ch. 23": "A", "PyTorch modifie-t-il les tenseurs qu'on lui donne ? Relis §23.9.2 du ch. 23": "C"})
wb.record("B7.Q8c", "C", mistakes={"relis le paragraphe qui suit le code de « L'artefact du modèle » : quels types prennent les métadonnées de safetensors ?": "A", "safetensors ignore-t-il ce qu'il ne sait pas écrire ? Relis « L'artefact du modèle » (§207.7)": "B"})
wb.record("B7.Q8d", True, mistakes={"relis « Les versions figées et l'empreinte des données » (§207.4) : que devient l'empreinte quand un seul pixel change ?": False})
wb.record("B7.Q8e", "C", mistakes={"la version change-t-elle le calcul ? Relis la traçabilité dans « L'artefact du modèle » (§207.7)": "A", "le format des pixels est une information à part de l'artefact : relis la liste de « L'artefact du modèle »": "B"})

WB_ANSWER {"hash": "ea4aebf76c73f6e3b80c5645edd388608f9b8c12ec1eb5110e4ae8ee23ca861e", "id": "B7.Q8a", "kind": "str", "mistakes": {"10bf94363465dde107e142525b74a4a8065fdd3dabb992c64dd09ee6ec1c481f": "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)", "21383226ff38b0574049a6842ce9287390682a1af27678504944dff477601b4c": "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)", "2a2406424f70218fc1ae2175dbf07fea8c99a181e9239171ca1165b247f2efcf": "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)", "2c6940f3f2a10a82157c7c84e37d89d5cea289af37fab1ac5cb2ee554e935f0a": "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)", "4e584ee318c6571d27cc42511d440c3f1333f6921a58dab6498e115241b37d0c": "il manque au moins une lettre, et une autre n'a pas sa place : relis « L'artefact du modèle » (§207.7)",

**B7.Q9 — Latence p50, p95 et débit : ce que ressent l'utilisateur**

In [11]:
wb.record("B7.Q9a", round(Q9A), mistakes={"relis la question : la moyenne pèse chaque requête, et 900 requêtes prennent 10 ms": 10,
          "c'est la moyenne des deux durées, sans tenir compte des effectifs": 105,
          "relis la question : on demande la moyenne": 200})
wb.record("B7.Q9b", round(Q9B), mistakes={"relis la question : on demande le p95, pas la moyenne": 29,
          "compte les requêtes de 10 ms : laissent-elles 95 % des mesures en dessous de cette valeur ?": 10,
          "c'est la moyenne des deux durées : le p95 se lit dans les mesures triées": 105})
wb.record("B7.Q9c", True, mistakes={"imagine 1 000 requêtes de 29 ms chacune : compare leur p95 à celui de l'énoncé, à moyenne égale": False})
wb.record("B7.Q9d", round(Q9D), mistakes={"convertis 150 ms en secondes avant de multiplier": 9000,
          "relis la loi de Little (§207.7) : L multiplie le débit par le temps de réponse": 400})
wb.record("B7.Q9e", round(Q9E), mistakes={"le débit divise la taille du batch par sa durée": 26})
wb.record("B7.Q9f", "A", mistakes={"une image d'un batch attend-elle les autres ? Relis la fin de « Latence, percentiles et débit » (§207.7)": "B", "relis la fin de « Latence, percentiles et débit » : que gagne-t-on à grouper, même sur CPU ?": "C"})

WB_ANSWER {"hash": "0dd71023e8dc038ad784ceaec27d2ad8cd6a0b61d1bd6a45fd08fd4c65593ddb", "id": "B7.Q9a", "kind": "int", "magnitude": 1, "mistakes": {"18a69b7b9d26f79e20cd2c24234aa89ca84f567ad47ba79333a0b9d86c8d1c79": "relis la question : on demande la moyenne", "6bb9163a756e598f8b911a9686716653dc6582c1a48223c3031b4ff1655902b2": "c'est la moyenne des deux durées, sans tenir compte des effectifs", "d84e1f0a8e335a451d675157bee3f0d6d9c2c6cae26bb2f456b5445c2ea9a49e": "relis la question : la moyenne pèse chaque requête, et 900 requêtes prennent 10 ms"}, "sign": 1}
📝 Ex B7.Q9a : réponse enregistrée (int).
WB_ANSWER {"hash": "1f73af8539eae8574cac043277750490ade34fdac30462c60526237df69736be", "id": "B7.Q9b", "kind": "int", "magnitude": 2, "mistakes": {"6c40e8935b935b41564f00cef9a5694e60dc0dccd2a7f6e4f64da9ba4b316f5f": "relis la question : on demande le p95, pas la moyenne", "93e568cd58458e94668ae7c60fe2e7d005380ec71e27c1a0c17721216ec4cf90": "compte les requêtes de 10 ms : laissent-elles 95 % des 

**B7.Q10 — torch.export, ONNX, TorchScript : lequel choisir aujourd'hui ?**

In [12]:
wb.record("B7.Q10a", "B", mistakes={"relis l'encadré 🕰️ de 207.8 : que dit PyTorch 2.11 de torch.jit.script ?": "A", "un objet sauvé par pickle se recharge-t-il sans le code de sa classe ?": "C"})
wb.record("B7.Q10b", "A", mistakes={"torch.compile accélère un modèle dans un programme Python : produit-il un fichier ?": "B", "un state_dict se recharge-t-il sans PyTorch ni le code du modèle ?": "C"})
wb.record("B7.Q10c", False, mistakes={"relis l'encadré 🕰️ de 207.8 : que produit torch.compile ?": True})
wb.record("B7.Q10d", "B", mistakes={"relis le paragraphe sur dynamic_shapes (§207.8) : que devient la taille de batch d'un export sans dynamic_shapes ?": "A", "un modèle exporté complète-t-il une entrée ? Relis le paragraphe sur les gardes (§207.8)": "C"})
wb.record("B7.Q10e", False, mistakes={"relis l'encadré 🕰️ de 207.8 : sur quoi s'appuie torch.onnx.export par défaut depuis la 2.9 ?": True})
wb.record("B7.Q10f", "A", mistakes={"un graphe figé peut-il contenir un if Python sur une valeur des données ? Relis le début de « torch.export et ONNX »": "B", "garder la seule branche de l'exemple, c'était le défaut de torch.jit.trace : relis le début de « torch.export et ONNX »": "C"})

WB_ANSWER {"hash": "99092009607c7cbf013311268473b981883bb16df141eb2fd6fff969c40580b3", "id": "B7.Q10a", "kind": "str", "mistakes": {"0c817e87325f61bb30963ef0ceb8f1b0280c7c8d5f66995ae5a60d8f19203b56": "relis l'encadré 🕰️ de 207.8 : que dit PyTorch 2.11 de torch.jit.script ?", "37353021c62937d9c498f88473575735beda079bcd3022ead92ce074cfb7bfb1": "un objet sauvé par pickle se recharge-t-il sans le code de sa classe ?"}}
📝 Ex B7.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "41941f2b6a55fb63562ff6a709ebd8ed0b142e86a33fb41dd6ba5a2d6d366cb6", "id": "B7.Q10b", "kind": "str", "mistakes": {"2627dfde651dfbc9df7e63092e4762b27e3ac7d275ec0cbb72d5f4007e5fda07": "torch.compile accélère un modèle dans un programme Python : produit-il un fichier ?", "5413ea0efefb3ac2529430db445753ffc7d3c7abfae4c37e269e2a2a68533aef": "un state_dict se recharge-t-il sans PyTorch ni le code du modèle ?"}}
📝 Ex B7.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "2da863ba30e214cc768bcdf241400f5b2fefb87c62f3d461070

**B7.Q11 — Écart entraînement-production et dérive des données : vrai ou faux**

In [13]:
wb.record("B7.Q11a", True, mistakes={"relis « Pourquoi un modèle se dégrade sans changer » (§207.10)": False})
wb.record("B7.Q11b", False, mistakes={"relis la définition du training-serving skew (§207.7) : d'où vient l'écart ?": True})
wb.record("B7.Q11c", True, mistakes={"relis la fin de « Le test de Kolmogorov-Smirnov » (§207.10) : que devient une p-valeur avec beaucoup de données ?": False})
wb.record("B7.Q11d", True, mistakes={"relis ce qu'on surveille sur les sorties (§207.10) : faut-il attendre les labels ?": False})
wb.record("B7.Q11e", "A", mistakes={"les deux chaînes de préparation ont-elles changé, ou les images reçues ? Relis 207.7 et 207.10": "B", "la relation entre une image et sa classe a-t-elle changé ? Relis « Pourquoi un modèle se dégrade sans changer » (§207.10)": "C"})
wb.record("B7.Q11f", False, mistakes={"relis la fin de « Les seuils » (§207.10) : quand fixe-t-on les bornes des intervalles ?": True})

WB_ANSWER {"hash": "34dbc1e843c450945954cb95ef24b1b43eab99929b2a8e52f011ce84c47dc6ea", "id": "B7.Q11a", "kind": "bool", "mistakes": {"9b84061b9ad9aac150bcd853c9b08bb83ab930e56aafe00d77ad0aac1e45c8c7": "relis « Pourquoi un modèle se dégrade sans changer » (§207.10)"}}
📝 Ex B7.Q11a : réponse enregistrée (bool).
WB_ANSWER {"hash": "ea5e31724ef8454d6df706f232b02ce28c63215667d16aea97cecf7281c7cb50", "id": "B7.Q11b", "kind": "bool", "mistakes": {"36fefe0014597072aa5ee6ad3d7a70a316ba673c2a42be37d61e01f5623e96a3": "relis la définition du training-serving skew (§207.7) : d'où vient l'écart ?"}}
📝 Ex B7.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "68314b862542832cd6911467ac382baff7afe8599f90966e0906f1881ce5be1f", "id": "B7.Q11c", "kind": "bool", "mistakes": {"fe6e326e241ad4bd4a78df77e38da5bf37025ac7105429c5a6948fd276274cd1": "relis la fin de « Le test de Kolmogorov-Smirnov » (§207.10) : que devient une p-valeur avec beaucoup de données ?"}}
📝 Ex B7.Q11c : réponse enregistrée (bool).
WB

**B7.R1 — Ce qu'une ligne de journal d'expériences doit contenir**

In [14]:
wb.record("B7.R1a", "ABCD", mistakes={"il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.4": {'A', 'ABD', 'AD', 'BC', 'BCD', 'D', 'BD', 'ABC', 'CD', 'AB', 'B', 'AC', 'ACD', 'C'}})
wb.record("B7.R1b", "A", mistakes={"relis la fin du paragraphe qui suit log_run (§207.5) : que fausse l'effacement d'un run raté ?": "B", "remplacer la ligne, c'est réécrire le passé : relis la docstring de log_run (§207.5)": "C"})
wb.record("B7.R1c", False, mistakes={"relis le paragraphe qui suit log_run (§207.5) : un champ ajouté gêne-t-il les anciennes lignes ?": True})
wb.record("B7.R1d", R1D, mistakes={"un journal JSONL s'écrit en ajoutant des lignes : relance-t-on un run en réécrivant la sienne ?": 7,
          "le journal garde-t-il les sept premiers runs ? Relis la docstring de log_run (§207.5)": 2})
wb.record("B7.R1e", "C", mistakes={"relis new_run_id (§207.5) : de quoi l'identifiant est-il fait ?": "A", "relis new_run_id (§207.5) : le commit y figure-t-il ?": "B"})

WB_ANSWER {"hash": "82402e6ba35ba00d6bbcf6dae129953e865f5049a222d2f836982761441c1815", "id": "B7.R1a", "kind": "str", "mistakes": {"04b6b649e63533014a31fc2c68cb38ad36d1d86aa1dc6350a44ca816a99418f0": "il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.4", "0a1e87f093f4917d576014627341f9f1855b8cf4d820b239df38223e2f601944": "il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.4", "256a8a0b2d41fed3d4e8752c0d178ecfb51e20f44bbc5a7280e0ec93ff941d4b": "il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.4", "25c969bedafe23a73677efcfa6de13ed1a3f7c79af15b259adef6bc3ec2a01de": "il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.4", "568ce87fd3e84626681f24bd146928c24e22c6c42804c7967e9a25be9a8efc83": "il manque au moins une lettre : relis le journal d'expériences du ch. 24 (🛠️ 24.15) et 207.4", "59e208d9104d3ffeed704f55290ef317a611d171259d1d73

**B7.R2 — Checkpoint complet, state_dict et weights_only**

In [15]:
wb.record("B7.R2a", "ABD", mistakes={"il manque au moins une lettre : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'AD', 'D', 'BD', 'AB', 'B', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)": {'BCD', 'C', 'ABC', 'CD', 'AC', 'ACD', 'BC'}})
wb.record("B7.R2b", "B", mistakes={"l'optimiseur sert-il à prédire ? Relis §23.9.2 du ch. 23 (« Les poids seuls »)": "A", "l'état de l'optimiseur contient-il les poids ? Relis §23.9.2 du ch. 23": "C"})
wb.record("B7.R2c", True, mistakes={"relis l'encadré 🕰️ du ch. 23 sur les formats de sauvegarde : que refuse weights_only=True ?": False})
wb.record("B7.R2d", False, mistakes={"relis §23.9.2 du ch. 23 : que faut-il faire avant load_state_dict ?": True})
wb.record("B7.R2e", "A", mistakes={"relis §23.9.1 du ch. 23 : que perd-on sans l'état de l'optimiseur ?": "B", "le load_state_dict du modèle se soucie-t-il de l'optimiseur ?": "C"})

WB_ANSWER {"hash": "a5f60fdeb65c38e980086d9bd119ed6fdd0e806566e67c3dd9604c42256b3a50", "id": "B7.R2a", "kind": "str", "mistakes": {"4a10d7a1a34860f1835d57638ae5fd4d11c8a3c10203a321412a5ea55dbd62e2": "il manque au moins une lettre, et une autre n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "565861efc990feef588b457f6ec779287bf09a5edad5582c8a5c5f612d0fa8e9": "il manque au moins une lettre, et une autre n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "5aebe8013527d03c7b0f03d3bb84c7869cf1e274b1e882216b911717e998be4f": "au moins une des lettres choisies n'a pas sa place : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "5b7c3c7ceaa4d7d093b9700526a870ec7fb98dd0c81dc97eccb287eac05e9e9c": "il manque au moins une lettre : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "5bd75c32ed760ae23b4e02b0687442c4f2e015a817bc826fd5bb4a674ef0152c": "il manque au moins une lettre : relis §23.9.1 du ch. 23 (« Tout dans un fichier »)", "6e4ae3dc7560313

**B7.R3 — Sauvegarder un Pipeline et figer les versions**

In [16]:
wb.record("B7.R3a", "A", mistakes={"relis la première règle de « sauvegarder un modèle et ses versions » (ch. 15)": "B", "des coefficients sans leur prétraitement servent-ils à prédire ? Relis la première règle du ch. 15": "C"})
wb.record("B7.R3b", False, mistakes={"relis la deuxième règle du ch. 15 : que garantit la documentation d'une version à l'autre ?": True})
wb.record("B7.R3c", "B", mistakes={"un FutureWarning annonce un changement d'API : relis la deuxième règle du ch. 15": "A", "relis la deuxième règle du ch. 15 : scikit-learn prévient-il, depuis la 1.3 ?": "C"})
wb.record("B7.R3d", True, mistakes={"relis la troisième règle du ch. 15 : que fait le chargement d'un pickle ?": False})
wb.record("B7.R3e", "ABC", mistakes={"il manque au moins une lettre : relis la deuxième règle du ch. 15 : que note-t-on à côté du fichier, et sous quelle forme ?": {'A', 'BC', 'AB', 'B', 'AC', 'C'}, "au moins une des lettres choisies n'a pas sa place : relis la deuxième règle du ch. 15 : que note-t-on à côté du fichier, et sous quelle forme ?": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis la deuxième règle du ch. 15 : que note-t-on à côté du fichier, et sous quelle forme ?": {'ABD', 'AD', 'BCD', 'D', 'BD', 'CD', 'ACD'}})
wb.record("B7.R3f", "C", mistakes={"skops.io vérifie les types avant de charger, mais en Python : relis la troisième règle du ch. 15": "A", "joblib demande Python et scikit-learn : relis la troisième règle du ch. 15": "B"})

WB_ANSWER {"hash": "e38281ba172b9457fed188dc5f94428d489b0e793d4ff31170d937b2a3f51161", "id": "B7.R3a", "kind": "str", "mistakes": {"0f9e6251328fd298269ea7ef86129e2822e7d8dec3285b3a401623dd2c5c73ac": "des coefficients sans leur prétraitement servent-ils à prédire ? Relis la première règle du ch. 15", "3e1bf7d29404a62d2031750d3ab9b59a1852071b7aef5ad67b3fb547e15390da": "relis la première règle de « sauvegarder un modèle et ses versions » (ch. 15)"}}
📝 Ex B7.R3a : réponse enregistrée (str).
WB_ANSWER {"hash": "528b4db8d3df07989768a2feaed69a9d1cee63a743bb5b0f94f8f3f52b23c1db", "id": "B7.R3b", "kind": "bool", "mistakes": {"8272eb99b515ec0ff2fa7895cdc2839991fe2c5dbb66888634d32bb0517751d9": "relis la deuxième règle du ch. 15 : que garantit la documentation d'une version à l'autre ?"}}
📝 Ex B7.R3b : réponse enregistrée (bool).
WB_ANSWER {"hash": "8c852ed41f363b3236da4f7bc8fb98dfda8e06c773a31f2c93378bcaa70c5fa8", "id": "B7.R3c", "kind": "str", "mistakes": {"82fdc42676b0289f2db40cb1318c20218d8e64

**Ex B7.1 — Taille d'un artefact : float32, bfloat16, int8, et la mémoire au chargement**

In [17]:
wb.record("B7.1a", F32_B71, decimals=1, mistakes={"un float32 occupe 4 octets : tu as compté 4 bits": N_B71 * 4 / 8 / MB,
          "relis la convention de l'énoncé : 1 Mo = 10⁶ octets": F32_MIB_B71,
          "un float32 occupe 4 octets par paramètre, pas 1": N_B71 / MB,
          "tu as compté en bits : un octet en fait 8": 32 * N_B71 / MB})
wb.record("B7.1b", F32_MIB_B71, decimals=1, mistakes={"relis la question : en Mio, avec 1 Mio = 2²⁰ octets": F32_B71,
          "1 Mio = 2²⁰ = 1 048 576 octets : relis la convention de l'énoncé": 4 * N_B71 / 1_024_000})
wb.record("B7.1c", BF16_B71, decimals=1, mistakes={"un bfloat16 occupe 2 octets par paramètre, pas 4": F32_B71,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": 2 * N_B71 / MIB,
          "le bfloat16 a 16 bits : combien d'octets ?": N_B71 / MB})
wb.record("B7.1d", INT8_B71, decimals=1, mistakes={"un int8 occupe 1 octet, pas 2": BF16_B71,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": N_B71 / MIB})
wb.record("B7.1e", DYNQ_B71, decimals=1, mistakes={"seules les matrices des couches denses passent en int8 : relis l'énoncé": INT8_B71,
          "les biais des couches denses restent en float32 : relis l'énoncé": (LIN_W_B71 + LIN_B_B71 + 4 * (N_B71 - LIN_W_B71 - LIN_B_B71)) / MB,
          "n'oublie pas les paramètres hors des matrices denses, en float32": LIN_W_B71 / MB,
          "relis l'unité : en Mo, 1 Mo = 10⁶ octets": DYNQ_B71 * MB / MIB})
wb.record("B7.1f", RATIO_B71, decimals=1, mistakes={"relis la question : e) ne passe pas tous les paramètres en int8": 4.0,
          "vérifie le sens du rapport : a) divisé par e)": 1 / RATIO_B71})
wb.record("B7.1g", PEAK_B71, decimals=1, mistakes={"relis l'énoncé : combien de copies des poids sont en mémoire au moment de load_state_dict ?": F32_MIB_B71,
          "relis l'unité : en Mio": 2 * F32_B71,
          "relis l'énoncé : deux copies des poids, pas trois": 3 * F32_MIB_B71})
wb.record("B7.1h", VGG_BYTES_B71, mistakes={"le state_dict contient aussi les buffers des batchnorms": 4 * VGG_PARAMS_B71,
          "relis l'énoncé : num_batches_tracked est un int64, de 8 octets": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71 + 6),
          "n'oublie pas les six compteurs num_batches_tracked": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71),
          "relis l'énoncé : chaque batchnorm a deux buffers d'une valeur par canal": 4 * (VGG_PARAMS_B71 + VGG_FLOAT_BUFFERS_B71 // 2) + 48})
wb.record("B7.1i", HEADER_B71, mistakes={"les buffers des batchnorms sont aussi dans le fichier : retire-les avec les paramètres": VGG_FILE_B71 - 4 * VGG_PARAMS_B71,
          "relis la question : on demande l'en-tête, ce qui reste une fois les tenseurs retirés": VGG_BYTES_B71})
wb.record("B7.1j", LIN_SHARE_B71, decimals=1, mistakes={"relis la question : en pourcentage": VGG_LINEAR_B71 / VGG_PARAMS_B71,
          "vérifie le sens du rapport : les paramètres de la couche dense divisés par tous les paramètres": 100 * VGG_PARAMS_B71 / VGG_LINEAR_B71})

WB_ANSWER {"decimals": 1, "hash": "bd55758152b071535a6add5e4da375c5e1cd966047acae51ceaaaf0ca54daafc", "hash_coarse": "9f5a187cf3a5cf66e9e1a6ecacb3674170119069f5c1bad99442d9e62dd227dc", "id": "B7.1a", "kind": "float", "magnitude": 2, "mistakes": {"74918b48994066c44b0c8bd261b94822cf64793db9c546910093ae7341fbca62": "un float32 occupe 4 octets par paramètre, pas 1", "7af881a605bcf66d5e451082e892c8ab3240d1152db4032f3c33a33456ce9ea7": "un float32 occupe 4 octets : tu as compté 4 bits", "8a98ca19b376d1f19d30bee68b1fee06d22ba8a5064a4013206ca049e1ee174a": "relis la convention de l'énoncé : 1 Mo = 10⁶ octets", "b4d9bdadaefda287bcf83f52ace89db71ebe3ddb45dbc15d0e62f94e4f5641e8": "tu as compté en bits : un octet en fait 8"}, "sign": 1}
📝 Ex B7.1a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "0d02495aa15139491441f0c41b1b820cef8cefb153ba7ec380c29605a1413f19", "hash_coarse": "d762b958c461860ffda094644e38f30da440f7dd863554548a6d7ff0aae65719", "id": "B7.1b", "kind": "f

**Ex B7.2 — Cinq graines, un intervalle : la dispersion d'une accuracy rapportée**

In [18]:
wb.record("B7.2a", MEAN_A_B72, decimals=1, mistakes={"divise la somme par le nombre de graines": float(A_B72.sum() / 4)})
wb.record("B7.2b", S_A_B72, decimals=3, mistakes={"l'écart-type corrigé divise par n − 1 : relis le rappel maths de 207.4": float(np.std(A_B72)),
          "c'est la variance : prends sa racine": float(np.var(A_B72, ddof=1)),
          "c'est la somme des carrés des écarts : divise-la, puis prends la racine": float(np.sum((A_B72 - A_B72.mean()) ** 2))})
wb.record("B7.2c", SE_A_B72, decimals=3, mistakes={"l'erreur type divise s par √n, avec n = 5 graines": S_A_B72 / 2,
          "l'erreur type divise s par √n, pas par n": S_A_B72 / 5,
          "pars de l'écart-type corrigé, divisé par n − 1 : relis b)": interval_b72(A_B72, ddof=0)[2]})
wb.record("B7.2d", HW_A_B72, decimals=2, mistakes={"avec cinq valeurs, on prend le quantile de Student : relis le rappel maths de 207.4": 1.96 * SE_A_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": 2.571 * SE_A_B72,
          "la demi-largeur multiplie t par l'erreur type, pas par l'écart-type": T4_B72 * S_A_B72,
          "pars de l'écart-type corrigé, divisé par n − 1": interval_b72(A_B72, ddof=0)[3]})
wb.record("B7.2e", LOW_A_B72, decimals=2, mistakes={"la borne basse retire la demi-largeur à la moyenne": MEAN_A_B72 + HW_A_B72,
          "avec cinq valeurs, on prend le quantile de Student : relis le rappel maths de 207.4": MEAN_A_B72 - 1.96 * SE_A_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": MEAN_A_B72 - 2.571 * SE_A_B72})
wb.record("B7.2f", OVERLAP_B72, mistakes={"calcule l'intervalle de B comme celui de A, puis compare leurs bornes": not OVERLAP_B72})
wb.record("B7.2g", MEAN_D_B72, decimals=1, mistakes={"vérifie le sens de la différence : B − A": -MEAN_D_B72})
wb.record("B7.2h", HW_D_B72, decimals=2, mistakes={"avec cinq différences, on prend le quantile de Student : relis le rappel maths de 207.4": 1.96 * SE_D_B72,
          "la demi-largeur multiplie t par l'erreur type des différences, pas par leur écart-type": T4_B72 * S_D_B72,
          "le quantile de Student se lit à n − 1 = 4 degrés de liberté : relis la table de 207.4": 2.571 * SE_D_B72})
wb.record("B7.2i", ZERO_IN_D_B72, mistakes={"calcule les deux bornes de l'intervalle de la différence avec g) et h)": not ZERO_IN_D_B72})

WB_ANSWER {"decimals": 1, "hash": "3cd85131e6de728cc20a2d23a392f66869d21f8d30b1bef06bbc8b2c0b0bc550", "hash_coarse": "4329d37d5acc95ccd6adc68516a9f83869448cd839b994ab6c6b35899094aa99", "id": "B7.2a", "kind": "float", "magnitude": 1, "mistakes": {"73d94fd51d99f72b51739c9a00163272113bf2031cd7b4118aa37de4c3c589b7": "divise la somme par le nombre de graines"}, "sign": 1}
📝 Ex B7.2a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "f83f3bd7e477f9fff21ab3b02e3e7126bef006e2ad0c3f202475954fb0e370c6", "hash_coarse": "8f172f65f040affa78c6be31c716299bcbdc6ec36ec237dd287298d15115a61d", "id": "B7.2b", "kind": "float", "magnitude": -1, "mistakes": {"21406df5f29f787e0b1cb6936218d944c270c9c612d6d71315d5f3cf3e475208": "c'est la variance : prends sa racine", "471a000c04b8ed0f9cb071f482288d20011be9cc84bfb9f91139cc4bbb52406b": "c'est la somme des carrés des écarts : divise-la, puis prends la racine", "9b0592b47e0528a46f0817749645b5784c16899caacb83328de9b1a4945ff42c": "l'écar

**Ex B7.3 — Latence et débit : p50 et p95 sur vingt mesures, batch 1 contre batch 32**

In [19]:
wb.record("B7.3a", MEAN_B73, decimals=2, mistakes={"relis la question : la moyenne des vingt mesures": P50_B73,
          "divise la somme par le nombre de mesures, 20": float(LAT_B73.sum() / 19)})
wb.record("B7.3b", P50_B73, decimals=2, mistakes={"c'est la plus petite valeur qui laisse la moitié des mesures en dessous (method=\"inverted_cdf\") : relis la convention de l'énoncé": 20.5,
          "relis la position : h = 0,5 × (n − 1), avec n = 20 mesures": 20.6,
          "relis la question : on demande le p50, la médiane": MEAN_B73})
wb.record("B7.3c", P95_B73, decimals=2, mistakes={"c'est la convention « la plus petite valeur qui laisse 95 % des mesures en dessous » (inverted_cdf) : relis celle de l'énoncé": 35.7,
          "c'est la moyenne des deux plus grandes mesures : relis la position h de l'énoncé": 41.8,
          "c'est la plus grande mesure : relis la position h de l'énoncé": 47.9,
          "c'est une autre convention (position q × (n + 1)) : relis celle de l'énoncé": float(np.percentile(LAT_B73, 95, method="weibull"))})
wb.record("B7.3d", ABOVE_B73, mistakes={"compte les mesures au-dessus de la moyenne, pas de la médiane": 10,
          "relis la question : au-dessus de la moyenne": 20 - ABOVE_B73})
wb.record("B7.3e", round(THR1_B73), mistakes={"relis la question : le débit se calcule ici avec le p50, pas la moyenne": round(1000 / MEAN_B73),
          "relis la question : le débit se calcule ici avec le p50, pas le p95": round(1000 / P95_B73)},
          fractional="le débit se donne ici en images par seconde, arrondi à l'entier")
wb.record("B7.3f", round(THR32_B73), mistakes={"un batch de 32 images : le débit divise sa taille par sa durée": round(1000 / P50_32_B73),
          "relis la question : le débit se calcule avec le p50 du batch, pas son p95": round(32_000 / P95_32_B73)},
          fractional="le débit se donne ici en images par seconde, arrondi à l'entier")
wb.record("B7.3g", RATIO_B73, decimals=1, mistakes={"vérifie le sens du rapport : f) divisé par e)": 1 / RATIO_B73,
          "tu as divisé les débits arrondis à l'entier : garde les valeurs non arrondies de e) et f)": round(THR32_B73) / round(THR1_B73),
          "le débit ne croît pas comme la taille du batch : calcule-le avec les mesures": 32.0})
wb.record("B7.3h", PER_IMAGE_B73, decimals=2, mistakes={"relis la question : le p50 du batch divisé par 32": P50_32_B73,
          "relis la question : on part du p50 du batch, pas de son p95": P95_32_B73 / 32})
wb.record("B7.3i", SLO_B73, mistakes={"chaque image attend la fin de son batch : compare le p95 d'un batch de 32 à l'objectif": not SLO_B73})
wb.record("B7.3j", PAGE_B73, decimals=2, mistakes={"relis la question : au moins une des huit dépasse le p95": 0.95 ** 8,
          "les probabilités ne s'additionnent pas ici : passe par l'événement contraire": 8 * 0.05,
          "relis la question : huit prédictions, pas une seule": 0.05})

WB_ANSWER {"decimals": 2, "hash": "6bac5d3036ddb4e895ab6befe2aaaba3b69d0e7edfe3411dc5cc7390de21f955", "hash_coarse": "a3a9a9b797358ed5352d72ff2c01443af4db05435c5a0404d408c47a379ca9b5", "id": "B7.3a", "kind": "float", "magnitude": 1, "mistakes": {"5bfdc5949835df66c4b7752c8f528390ea41e0f79330ee3672979ea9bbdd2b34": "divise la somme par le nombre de mesures, 20", "7e0b6bb775363e869dde932b9d9786a67a2f192781c310c842ed56f5834a96a7": "relis la question : la moyenne des vingt mesures"}, "sign": 1}
📝 Ex B7.3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "1237d60b7d935b8b084e1b3c8abedcd8bf766a29ec03345b5ed9c5ae61a02965", "hash_coarse": "1a02c4c153eda98ea4f2e225bcba446d3c37eff723b9fa25de4dc2aaec9cc5d2", "id": "B7.3b", "kind": "float", "magnitude": 1, "mistakes": {"1912ca7c11c51714030ef0dceee9b32b3cff30fa96a5d676c0b37441aa9e2c57": "relis la question : on demande le p50, la médiane", "e49f47d7c87572c19a2a3a25e900351aed770ed9ad37693e38b28638b98d5ba5": "relis la posi

**Ex B7.4 — Indice de stabilité de population (PSI) à la main sur quatre intervalles**

In [20]:
wb.record("B7.4a", C1_W1_B74, decimals=4, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[1], log=np.log10)[0]),
          "vérifie le signe : p − q et ln(p/q) ont toujours le même signe": -C1_W1_B74,
          "relis la formule : la contribution multiplie ln(p/q) par p − q": float(np.log(P_B74[1][0] / 0.25))})
wb.record("B7.4b", PSI_W1_B74, decimals=3, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[1], log=np.log10).sum()),
          "il manque une partie du PSI : relis le lien avec la divergence KL (§207.10)": float(np.sum(P_B74[1] * np.log(P_B74[1] / Q_B74)))})
wb.record("B7.4c", PSI_W2_B74, decimals=3, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[2], log=np.log10).sum()),
          "il manque une partie du PSI : relis le lien avec la divergence KL (§207.10)": KL_PQ_W2_B74,
          "relis la question : on demande le PSI, la somme des quatre contributions (p − q) ln(p/q)": KL_QP_W2_B74})
wb.record("B7.4d", "D", mistakes={"calcule la contribution de I₁ en semaine 2 et compare-la aux trois autres": "A", "calcule la contribution de I₂ en semaine 2 et compare-la aux trois autres": "B", "calcule la contribution de I₃ en semaine 2 et compare-la aux trois autres": "C"})
wb.record("B7.4e", KL_QP_W2_B74, decimals=3, mistakes={"relis la question : q en premier, Σ q ln(q/p)": KL_PQ_W2_B74,
          "relis la question : on demande une seule des deux divergences": PSI_W2_B74,
          "la divergence KL utilise le logarithme naturel ln": float(np.sum(Q_B74 * np.log10(Q_B74 / P_B74[2])))})
wb.record("B7.4f", "B", mistakes={"compare le PSI de la semaine 2 au second seuil, 0,25": "A", "compare le PSI de la semaine 1 au premier seuil, 0,1": "C"})
wb.record("B7.4g", True, mistakes={"calcule ln(0 / 0,25) : que vaut-il pour NumPy, et que devient son produit par (0 − 0,25) ?": False})
wb.record("B7.4h", PSI_W3_B74, decimals=2, mistakes={"le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[3], eps=1e-4, log=np.log10).sum()),
          "relis l'énoncé : ε vaut 10⁻⁴ ici": PSI_W3_EPS3_B74})
wb.record("B7.4i", PSI_W3_EPS3_B74, decimals=3, mistakes={"relis la question : ε = 10⁻³": PSI_W3_B74,
          "relis l'énoncé : ε s'ajoute à toutes les parts, celles de la production et celles de la référence": PSI_W3_EPS3_PART_B74,
          "le PSI utilise le logarithme naturel ln": float(psi_terms_b74(P_B74[3], eps=1e-3, log=np.log10).sum())})

WB_ANSWER {"decimals": 4, "hash": "0dd5de8ef69669b49252267202a1a40ebf7581e56c5f068ee7135b256550b5ca", "hash_coarse": "0190b71b33be8a0faaab4c265b1b0a10d2fabf066cd0871c90cd21c658eeb8e5", "id": "B7.4a", "kind": "float", "magnitude": -2, "mistakes": {"21bc36827ca4e78ad6601391ab91493d19e5f42b6e97756bd36979a54c8902e3": "relis la formule : la contribution multiplie ln(p/q) par p − q", "5e07c1695b1bc9a5d4d67fe7555b8b028314a9449dc029df127e8be02e2be2a2": "le PSI utilise le logarithme naturel ln", "fc2a3f1845a05d2d16e366b5c36a1b074e40e8ec6a0ea3b82b757a4b66ef2b4a": "vérifie le signe : p − q et ln(p/q) ont toujours le même signe"}, "sign": 1}
📝 Ex B7.4a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "9803e5e4dcc7623a539cfdbff4dbbdc3a6926acd2db40bd90f2770141f93ae91", "hash_coarse": "a8a4ab28c200ce8b5fc1e7251b205cbbe7f5f444671fb5630aed885fae97e118", "id": "B7.4b", "kind": "float", "magnitude": -2, "mistakes": {"2b1f37643b702710ef8644a654253dcce6a91ef32056e16db350d66b4

## Partie A · Le projet : du notebook au paquet, ligne de commande, configuration, reproductibilité, suivi et tests

Au MP5, `cifarkit` rangeait déjà ton code dans un dossier que le notebook importait. Ici, ce code devient un **projet** : un paquet installable (B7.9), piloté par une ligne de commande (B7.10) et une configuration validée (B7.11), dont tu mesures la reproductibilité (B7.12, B7.13), qui journalise ses runs (B7.14) et que des tests protègent (B7.15). Le modèle est volontairement petit (le CNN de 207.6, 2 000 images, une epoch : quelques secondes) : on juge ici la mécanique du projet, pas l'accuracy.

La cellule suivante crée ton dossier de projet, `PROJECT` (`mon_travail/b7_production/cifar-prod/` ; le notebook de solutions travaille dans un dossier temporaire), y écrit `data/cifar10_2000.npz` (200 images par classe pour l'entraînement, 50 pour la validation, un sous-ensemble figé de CIFAR-10, `uint8` `(N, 32, 32, 3)`, avec les tableaux `x_train`, `y_train`, `x_val`, `y_val`) et charge `wbp` (`wb.prod`, l'infrastructure du chapitre : écrire un fichier, lancer une commande, installer dans un environnement neuf). Toutes les commandes partent du dossier du projet, avec des chemins absolus.

In [21]:
# Tools of part A: the folder of your project, the data of the chapter, the commands; nothing here is your package
import contextlib
import io
import json
import os
import re
import sys
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
import torch
from wb import prod as wbp
from wb.nbtools import StopCheck, checked, fr, refuses, run_pytest_group, same, verdict, written

PROJECT = wbp.project_dir("solution")             # the folder of your project
DATA_FILE = wbp.write_subset(PROJECT / "data")  # the 2 000 + 500 images of the chapter (.npz, 7.7 MB, not for git)
SUBSET = wbp.cifar_subset()                     # the same arrays, for the checks
CLASSES = wbp.CLASSES
GITIGNORE = "data/\nruns/\n__pycache__/\n*.egg-info/\n.pytest_cache/\n"
DEFAULTS = {"seed": 0, "epochs": 1, "lr": 0.05, "batch_size": 128, "n_train": 2000, "arch": "cnn", "widths": [16, 32],
            "data": "data/cifar10_2000.npz", "deterministic": True, "num_workers": 0}   # TrainConfig().to_dict()


def scratch(prefix):
    """A new temporary folder (outside the project): the checks write their test files there."""
    return Path(tempfile.mkdtemp(prefix=f"wb_b7_{prefix}_"))


def run_line(ex_id, result, ok, success, failure):
    """One verdict about a command of your package: what was run, then the end of stderr when it went wrong (⏳ when
    the command stopped on a NotImplementedError of your skeleton)."""
    shown = result.command().replace(f"python -m {wbp.PACKAGE}", "cifarprod")
    pending = re.findall(r"NotImplementedError: (.+)", result.stderr)
    if not ok and pending:
        print(f"⏳ Ex {ex_id} : {shown} → pas encore fait ({pending[-1].strip()}).")
        return False
    if result.timed_out:
        ok, failure = False, f"délai dépassé ({result.seconds:.0f} s) : {failure}"
    seen = (f"stderr : {result.tail(2)[:400]}" if result.stderr.strip() else
            f"rien sur stderr ; stdout : {result.stdout.strip()[-300:]!r}" if result.stdout.strip() else "rien sur stderr")
    verdict(ex_id, ok, f"{shown} → {success}", f"{shown} → {failure} (code {result.returncode} ; {seen})")
    return ok


print(wbp.project_line(PROJECT, "solution"))
print(f"ℹ️ données : data/{DATA_FILE.name} ({len(SUBSET['x_train'])} images d'entraînement, {len(SUBSET['x_val'])} de "
      f"validation, 200 et 50 par classe ; empreinte {wbp.SUBSET_FINGERPRINT[:12]}…)")

ℹ️ le projet : un dossier temporaire (le notebook de solutions n'écrit jamais dans le dépôt).
ℹ️ données : data/cifar10_2000.npz (2000 images d'entraînement, 500 de validation, 200 et 50 par classe ; empreinte 286106e2dece…)


### Ex B7.9 — Du notebook au paquet : data.py, model.py, train.py, pyproject.toml et installation éditable 📦 ★★★ ⏱️ 60 min
**Objectif :** transformer le code d'un notebook en paquet Python installable, avec un layout `src/`.  
**Prérequis :** MP5 (`cifarkit`) · ch. 23 · fiche §207.1 (« Une structure de dépôt », « pyproject.toml et installation éditable »), §207.4 (`seed_everything`, `array_fingerprint`), §207.6 · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat.** Ton projet doit avoir cette forme (le `.gitignore` et le fichier de données sont fournis) :

```text
cifar-prod/
├── pyproject.toml        .gitignore        data/cifar10_2000.npz
└── src/cifarprod/
    ├── __init__.py       (__version__ = "0.1.0", fourni)
    ├── data.py           model.py           train.py
```

- `pyproject.toml` : le backend de construction de la fiche (`setuptools`, déjà installé : un autre, comme `hatchling`, n'est pas dans l'environnement du workbook) ; le paquet `cifarprod`, version `0.1.0`, Python ≥ 3.11 ; les dépendances avec des bornes basses (`numpy`, `torch`, `pyyaml`, `safetensors`) ; deux extras, `dev` (`pytest`) et `serve` (`fastapi`, `uvicorn`, `pydantic`) ; le point d'entrée `cifarprod = "cifarprod.cli:main"` (le fichier `cli.py` viendra en B7.10) ; et, pour B7.15, une table `[tool.pytest.ini_options]` avec `testpaths = ["tests"]` et le marqueur `slow` déclaré dans `markers`.
- `data.py` : `CLASSES` (les 10 noms, dans l'ordre des sorties) ; `load_npz(path, part="train")` rend `(X, y)`, `X` `uint8` `(N, 32, 32, 3)` et `y` `int64` ; elle **vérifie le schéma** : `ValueError` pour des images qui ne sont pas `uint8` `(N, 32, 32, 3)`, des labels qui ne sont pas $N$ entiers de 0 à 9, ou une autre partie que `"train"` et `"val"` ; `FileNotFoundError` pour un fichier absent ; `fingerprint(*arrays)`, l'empreinte de 207.4 (le type, la forme, puis les octets de chaque tableau) ; `channel_stats(images)` rend `(mean, std)`, trois floats Python chacun, la moyenne et l'écart-type (divisé par $n$) de chaque canal dans $[0, 1]$ ; `preprocess(images, mean, std)` rend un tenseur `float32` `(N, 3, 32, 32)`, chaque canal valant $(\text{pixel}/255 - \text{mean}_c)/\text{std}_c$, accepte aussi **une** image `(32, 32, 3)` (rendue `(1, 3, 32, 32)`) et refuse un tableau qui n'est pas `uint8` (`ValueError`) ; `make_loader(X, y, mean, std, batch_size=128, shuffle=False, seed=0, num_workers=0)` rend un `DataLoader` de mini-batches `(preprocess(X), y)`, `float32` et `int64`, dont l'ordre mélangé vient d'un `torch.Generator().manual_seed(seed)` **à lui** (il ne dépend pas du générateur global) ; `synthetic(n, seed=0)` rend $n$ images `uint8` tirées par `np.random.default_rng(seed).integers(0, 256, (n, 32, 32, 3), dtype=np.uint8)` et les labels `np.arange(n) % 10` (pour les tests et l'intégration continue, qui n'ont pas CIFAR-10).
- `model.py` : `build_model(arch="cnn", widths=(12, 24), n_classes=10)` rend un `nn.Sequential` des couches de la docstring, **dans cet ordre** (`"cnn"`, le CNN de la fiche et de `cifarkit.small_cnn` ; `"vgg"`, celui de `cifarkit.vgg` avec batchnorm et max pooling) : les noms de ses poids (`0.weight`, `1.weight`…) sont alors ceux de `cifarkit`, et le modèle livré du MP5 s'y recharge ; `ValueError` pour une autre architecture ou des largeurs vides ; `n_parameters(model)`.
- `train.py` : `seed_everything(seed, deterministic=True)` (207.4) ; `evaluate(model, loader)` rend `(loss moyenne, accuracy)` en mode `eval()`, sans graphe, puis rend au modèle son mode d'avant ; `train(X, y, X_val, y_val, *, arch="cnn", widths=(16, 32), epochs=1, lr=0.05, batch_size=128, seed=0, deterministic=True, num_workers=0)` appelle d'abord `seed_everything(seed, deterministic)`, apprend la normalisation sur `X` **seulement**, entraîne avec SGD (`momentum=0.9`) et la cross-entropy sur les mini-batches de `make_loader(..., shuffle=True, seed=seed)`, et rend `{"model": le modèle en mode eval(), "mean", "std", "history": {"train_loss", "val_loss", "val_acc"} (un float par epoch), "seconds"}`. **Aucun `print`** : une ligne INFO par epoch et une ligne DEBUG par mini-batch avec `log = logging.getLogger(__name__)` (207.2), que B7.10 affichera.

La cellule de vérification écrit tes fichiers, importe ton paquet **neuf** depuis `src/` et compare chaque fonction : `load_npz` aux tableaux du fichier et quatre fichiers mal formés, `fingerprint` à `hashlib`, `channel_stats` et `preprocess` à NumPy, `make_loader` avec et sans mélange (le générateur global changé entre deux appels), `build_model` à l'architecture du contrat (mêmes tenseurs et mêmes formes, quatre réglages), puis lance `train` deux fois avec la même graine (mêmes poids) et une fois avec une autre, et l'entraînement fumée des réglages par défaut sur les 2 000 images (moins d'une minute, une accuracy de validation au-dessus du hasard). Enfin, elle fait `pip install -e .` de ton projet dans un **environnement virtuel neuf** (qui voit les bibliothèques du notebook, mais pas ton dossier `src/`) et vérifie, depuis un autre dossier, que `import cifarprod` lit ton dossier `src/` (une installation éditable, pas une copie), la version et le point d'entrée. Ton environnement à toi n'est jamais modifié : pour avoir la commande `cifarprod` dans un terminal, lance toi-même `pip install -e .` dans le dossier du projet.

Dans tes notes : pourquoi le layout `src/` empêche-t-il `import cifarprod` depuis la racine du projet tant que le paquet n'est pas installé, et qu'est-ce que cela protège ? Pourquoi `train` apprend-il la normalisation sur `X` seulement, et que deviendrait l'évaluation si on la calculait sur l'entraînement et la validation ensemble (ch. 12) ? Combien de secondes prend l'entraînement fumée, et que juge-t-il (et que ne juge-t-il pas) ?

In [22]:
# Ex B7.9: the files of the package, as strings (the reference package of the chapter, solutions/b7_cifarprod)

# pyproject.toml
PYPROJECT_TOML = r'''[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "cifarprod"
version = "0.1.0"
description = "A CIFAR-10 classifier, from the notebook to production (bonus chapter B7 of the workbook)."
requires-python = ">=3.11"
dependencies = ["numpy>=2.0", "torch>=2.5", "pyyaml>=6.0", "safetensors>=0.4"]   # loose bounds: what the code needs

[project.optional-dependencies]
serve = ["fastapi>=0.110", "uvicorn>=0.30", "pydantic>=2.6"]
dev = ["pytest>=8.0"]

[project.scripts]
cifarprod = "cifarprod.cli:main"

[tool.pytest.ini_options]
testpaths = ["tests"]
markers = ["slow: a long test (an end-to-end training); skipped with -m 'not slow'"]
'''

# src/cifarprod/__init__.py
INIT_PY = r'''"""cifarprod: a CIFAR-10 classifier as an installable package (data, model, training, configuration, command line,
run journal), from the notebook of the MP5 to production (bonus chapter B7)."""

__version__ = "0.1.0"
'''

# src/cifarprod/data.py
DATA_PY = r'''"""Data of cifarprod: the CIFAR-10 subset file, its schema, its fingerprint, the normalization and the loaders.

    X, y = load_npz("data/cifar10_2000.npz", "train")     # uint8 (N, 32, 32, 3), int64 labels 0-9
    mean, std = channel_stats(X)                           # learned on the training images only
    x = preprocess(X[:8], mean, std)                       # float32 tensor (8, 3, 32, 32)

The images are uint8 RGB arrays (N, 32, 32, 3), as CIFAR-10 is stored; the model receives float32 tensors (N, 3, 32,
32), each channel scaled to [0, 1] then normalized with the mean and the standard deviation of the training images.
One function, preprocess, does it for the training AND for the service.
"""

from __future__ import annotations

import hashlib
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
PARTS = ("train", "val")


def check_images(X: np.ndarray, y: np.ndarray) -> None:
    """The schema of the data: X uint8 of shape (N, 32, 32, 3), y integers of shape (N,) in 0..9. ValueError if not."""
    if not isinstance(X, np.ndarray) or X.dtype != np.uint8:
        raise ValueError(f"expected uint8 images, got {getattr(X, 'dtype', type(X).__name__)}")
    if X.ndim != 4 or X.shape[1:] != (32, 32, 3):
        raise ValueError(f"expected images of shape (N, 32, 32, 3), got {X.shape}")
    y = np.asarray(y)
    if y.ndim != 1 or len(y) != len(X):
        raise ValueError(f"expected {len(X)} labels of shape ({len(X)},), got shape {y.shape}")
    if not np.issubdtype(y.dtype, np.integer):
        raise ValueError(f"expected integer labels, got {y.dtype}")
    if len(y) and (y.min() < 0 or y.max() >= len(CLASSES)):
        raise ValueError(f"expected labels in 0..{len(CLASSES) - 1}, got {int(y.min())}..{int(y.max())}")


def load_npz(path: str | Path, part: str = "train") -> tuple[np.ndarray, np.ndarray]:
    """(X, y) of one part ("train" or "val") of a .npz file with the arrays x_train, y_train, x_val, y_val: X uint8
    (N, 32, 32, 3), y int64 (N,). The schema is checked (check_images): ValueError for a wrong dtype, shape or label,
    or for another part; FileNotFoundError if the file does not exist."""
    if part not in PARTS:
        raise ValueError(f"part must be one of {PARTS}, got {part!r}")
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"no data file {path}")
    with np.load(path) as archive:
        if f"x_{part}" not in archive or f"y_{part}" not in archive:
            raise ValueError(f"{path} has no arrays x_{part} and y_{part}")
        X, y = archive[f"x_{part}"], archive[f"y_{part}"]
    check_images(X, y)
    return np.ascontiguousarray(X), y.astype(np.int64)


def fingerprint(*arrays: np.ndarray) -> str:
    """SHA-256 of the dtype, the shape and the bytes of each array (in this order): another value if one pixel, one
    label or the dtype changes."""
    h = hashlib.sha256()
    for a in arrays:
        a = np.ascontiguousarray(a)
        h.update(f"{a.dtype}{a.shape}".encode())
        h.update(a.tobytes())
    return h.hexdigest()


def channel_stats(images: np.ndarray) -> tuple[tuple[float, float, float], tuple[float, float, float]]:
    """(mean, std): three Python floats each, the mean and the standard deviation (divided by n) of the pixels of each
    channel on the scale [0, 1], over all the uint8 images (N, H, W, 3) given (the training images only)."""
    x = np.asarray(images, dtype=np.float64) / 255.0
    if x.ndim != 4 or x.shape[-1] != 3:
        raise ValueError(f"expected images of shape (N, H, W, 3), got {x.shape}")
    mean, std = x.mean(axis=(0, 1, 2)), x.std(axis=(0, 1, 2))
    return tuple(float(m) for m in mean), tuple(float(s) for s in std)


def preprocess(images: np.ndarray, mean, std) -> torch.Tensor:
    """The input of the model: uint8 RGB images (N, 32, 32, 3), or one image (32, 32, 3), as a float32 tensor (N, 3,
    32, 32) (N = 1 for one image), each channel c equal to (pixel / 255 - mean[c]) / std[c]. ValueError for an array
    that is not uint8 (a float image would not be scaled the same way) or not 32 x 32 x 3."""
    images = np.asarray(images)
    if images.dtype != np.uint8:
        raise ValueError(f"preprocess expects uint8 images (0..255), got {images.dtype}")
    if images.ndim == 3:
        images = images[None]
    if images.ndim != 4 or images.shape[1:] != (32, 32, 3):
        raise ValueError(f"expected images of shape (N, 32, 32, 3) or (32, 32, 3), got {images.shape}")
    x = torch.from_numpy(np.ascontiguousarray(images)).permute(0, 3, 1, 2).to(torch.float32) / 255.0
    m = torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)
    s = torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1)
    return ((x - m) / s).contiguous()


def make_loader(X: np.ndarray, y: np.ndarray, mean, std, batch_size: int = 128, shuffle: bool = False,
                seed: int = 0, num_workers: int = 0) -> DataLoader:
    """A DataLoader of (preprocess(X), y) mini-batches: float32 (B, 3, 32, 32) and int64 (B,). When shuffle, the order
    comes from its own torch.Generator().manual_seed(seed), so that it depends on nothing drawn before."""
    dataset = TensorDataset(preprocess(X, mean, std), torch.as_tensor(np.asarray(y), dtype=torch.int64))
    generator = torch.Generator().manual_seed(int(seed)) if shuffle else None
    return DataLoader(dataset, batch_size=int(batch_size), shuffle=shuffle, generator=generator,
                      num_workers=int(num_workers))


def synthetic(n: int, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """(X, y): n random uint8 images (n, 32, 32, 3) drawn by np.random.default_rng(seed), and the labels i % 10, for
    the tests and the continuous integration, which have no CIFAR-10."""
    rng = np.random.default_rng(seed)
    X = rng.integers(0, 256, size=(n, 32, 32, 3), dtype=np.uint8)
    return X, np.arange(n, dtype=np.int64) % len(CLASSES)
'''

# src/cifarprod/model.py
MODEL_PY = r'''"""The models of cifarprod: the CNNs of the MP5 (cifarkit), rebuilt from a few settings, so that an artifact can
rebuild them.

    model = build_model("vgg", widths=(16, 32))       # float32 (N, 3, 32, 32) -> logits (N, 10)
    model = build_model(widths=(12, 24))              # the defaults: the CNN of the fiche (18 322 parameters)

- "cnn": for each width w, Conv2d(c, w, 3, padding=1), ReLU, MaxPool2d(2); then Flatten and Linear (no global pooling);
- "vgg": for each width w, two Conv2d(c, w, 3, padding=1, bias=False), each followed by BatchNorm2d(w) and ReLU; a
  MaxPool2d(2) after every stage but the last; then AdaptiveAvgPool2d(1), Flatten, Linear(widths[-1], n_classes).
"""

from __future__ import annotations

import torch.nn as nn

ARCHS = ("cnn", "vgg")


def build_model(arch: str = "cnn", widths=(12, 24), n_classes: int = 10) -> nn.Sequential:
    """The network ``arch`` ("cnn" or "vgg") with one stage per width; ValueError for another arch or empty widths."""
    widths = tuple(int(w) for w in widths)
    if arch not in ARCHS:
        raise ValueError(f"unknown arch {arch!r}: expected one of {ARCHS}")
    if not widths or min(widths) < 1:
        raise ValueError(f"widths must be positive whole numbers, got {widths}")
    layers, c = [], 3
    if arch == "cnn":
        side = 32
        for w in widths:
            layers += [nn.Conv2d(c, w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)]
            c, side = w, side // 2
        return nn.Sequential(*layers, nn.Flatten(), nn.Linear(c * side * side, n_classes))
    for i, w in enumerate(widths):
        for _ in range(2):
            layers += [nn.Conv2d(c, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU()]
            c = w
        if i < len(widths) - 1:
            layers.append(nn.MaxPool2d(2))
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, n_classes)]
    return nn.Sequential(*layers)


def n_parameters(model: nn.Module) -> int:
    """The number of trainable parameters of a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
'''

# src/cifarprod/train.py
TRAIN_PY = r'''"""The training of cifarprod: the seeds, one evaluation, and train(), which returns the model with its normalization.

    result = train(X, y, X_val, y_val, arch="cnn", widths=(16, 32), epochs=1, lr=0.05, batch_size=128, seed=0)
    result["model"], result["mean"], result["std"], result["history"]["val_acc"]

SGD with momentum 0.9, the cross-entropy, mini-batches in the order of the generator of seed_everything; one INFO line
per epoch, one DEBUG line per mini-batch (logger "cifarprod.train"). Nothing is printed.
"""

from __future__ import annotations

import logging
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .data import channel_stats, make_loader
from .model import build_model

log = logging.getLogger(__name__)


def seed_everything(seed: int, deterministic: bool = True) -> torch.Generator:
    """Seed Python, NumPy and PyTorch, switch the deterministic algorithms on or off, and return a generator seeded
    with ``seed`` for the DataLoader."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(bool(deterministic))
    return torch.Generator().manual_seed(seed)


def evaluate(model: nn.Module, loader) -> tuple[float, float]:
    """(mean cross-entropy, accuracy) of the model on all the examples of the loader, in evaluation mode and without a
    graph; the mode of the model is restored afterwards."""
    was_training = model.training
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.inference_mode():
        for x, t in loader:
            out = model(x)
            total_loss += F.cross_entropy(out, t, reduction="sum").item()
            correct += int((out.argmax(dim=1) == t).sum())
            n += len(t)
    model.train(was_training)
    return total_loss / max(n, 1), correct / max(n, 1)


def train(X: np.ndarray, y: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, *, arch: str = "cnn",
          widths=(16, 32), epochs: int = 1, lr: float = 0.05, batch_size: int = 128, seed: int = 0,
          deterministic: bool = True, num_workers: int = 0) -> dict:
    """Train ``build_model(arch, widths)`` on the uint8 images X (N, 32, 32, 3) and their labels y.

    The normalization (mean, std) is learned on X only (channel_stats). Returns {"model": the trained model in
    evaluation mode, "mean", "std", "history": {"train_loss", "val_loss", "val_acc"} (one value per epoch),
    "seconds"}."""
    start = time.perf_counter()
    seed_everything(seed, deterministic)               # the initialization of the weights
    mean, std = channel_stats(X)
    model = build_model(arch, widths)
    # the order of the mini-batches: a generator of its own, seeded with the same seed
    loader = make_loader(X, y, mean, std, batch_size, shuffle=True, seed=seed, num_workers=num_workers)
    val_loader = make_loader(X_val, y_val, mean, std, batch_size=500)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    log.info("training %s %s on %d images, %d epoch(s)", arch, list(widths), len(X), epochs)
    for epoch in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for step, (x, t) in enumerate(loader, start=1):
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), t)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(t)
            n += len(t)
            log.debug("epoch %d batch %d/%d loss %.4f", epoch, step, len(loader), loss.item())
        val_loss, val_acc = evaluate(model, val_loader)
        history["train_loss"].append(total / n)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        log.info("epoch %d/%d train_loss %.4f val_loss %.4f val_acc %.4f", epoch, epochs, total / n, val_loss, val_acc)
    model.eval()
    return {"model": model, "mean": mean, "std": std, "history": history, "seconds": time.perf_counter() - start}
'''


FILES_9 = {"pyproject.toml": PYPROJECT_TOML, ".gitignore": GITIGNORE, "src/cifarprod/__init__.py": INIT_PY,
           "src/cifarprod/data.py": DATA_PY, "src/cifarprod/model.py": MODEL_PY, "src/cifarprod/train.py": TRAIN_PY}
ok_9 = {}
with checked("B7.9"):
    wbp.write_files(PROJECT, FILES_9)
    missing_9 = [rel for rel, text in FILES_9.items() if text is ...]
    print("ℹ️ fichiers écrits dans le projet : " + ", ".join(r for r, t in FILES_9.items() if t is not ...)
          + (f" ; pas encore écrits : {', '.join(missing_9)}" if missing_9 else ""))
    pkg_9 = wbp.import_package(PROJECT)
    ok_9["import"] = getattr(pkg_9, "__version__", None) == "0.1.0"
    verdict("B7.9", ok_9["import"], "import cifarprod depuis src/ : version 0.1.0.",
            f"cifarprod.__version__ vaut {getattr(pkg_9, '__version__', None)!r}, attendu '0.1.0'.")

with checked("B7.9"):                                    # data.py
    data_9 = wbp.import_module(PROJECT, "data")
    X_9, y_9 = data_9.load_npz(DATA_FILE, "train")
    Xv_9, yv_9 = data_9.load_npz(DATA_FILE, "val")
    got_9 = [(np.asarray(a).dtype, np.asarray(a).shape) for a in (X_9, y_9, Xv_9, yv_9)]
    ok_9["load"] = (isinstance(X_9, np.ndarray) and X_9.dtype == np.uint8 and np.asarray(y_9).dtype == np.int64
                    and np.array_equal(X_9, SUBSET["x_train"]) and np.array_equal(y_9, SUBSET["y_train"])
                    and np.array_equal(Xv_9, SUBSET["x_val"]) and np.array_equal(yv_9, SUBSET["y_val"]))
    verdict("B7.9", ok_9["load"], "load_npz : les 2 000 images d'entraînement et les 500 de validation, uint8 et int64.",
            f"load_npz ne rend pas les tableaux du fichier en uint8 et int64 : reçu (dtype, forme) {got_9}.")
    bad_9 = scratch("bad")
    zeros_9 = np.zeros((4, 32, 32, 3), np.uint8)
    for name_9, (xb_9, yb_9) in {"des images float32": (zeros_9.astype(np.float32), np.arange(4)),
                                 "des images 32 × 32 × 1": (zeros_9[..., :1], np.arange(4)),
                                 "un label 10": (zeros_9, np.array([0, 1, 2, 10])),
                                 "3 labels pour 4 images": (zeros_9, np.arange(3))}.items():
        np.savez(bad_9 / "bad.npz", x_train=xb_9, y_train=yb_9, x_val=xb_9, y_val=yb_9)
        ok_9[name_9] = refuses("B7.9", f"load_npz d'un fichier avec {name_9}",
                               lambda: data_9.load_npz(bad_9 / "bad.npz", "train"))
    ok_9["part"] = refuses("B7.9", 'load_npz(..., "test")', lambda: data_9.load_npz(DATA_FILE, "test"))
    ok_9["absent"] = refuses("B7.9", "load_npz d'un fichier absent", lambda: data_9.load_npz(bad_9 / "absent.npz"),
                             error=FileNotFoundError)
    fp_9 = data_9.fingerprint(SUBSET["x_train"][:5], SUBSET["y_train"][:5])
    ok_9["fingerprint"] = fp_9 == wbp.array_fingerprint(SUBSET["x_train"][:5], SUBSET["y_train"][:5])
    verdict("B7.9", ok_9["fingerprint"], f"fingerprint de 5 images et de leurs labels : {str(fp_9)[:12]}…, comme "
            "hashlib sur le type, la forme et les octets.", f"fingerprint de 5 images et de leurs labels : "
            f"{str(fp_9)[:16]!r}…, attendu un autre SHA-256 (le type, la forme, puis les octets de chaque tableau).")
    mean_9, std_9 = data_9.channel_stats(SUBSET["x_train"])
    want_9 = (SUBSET["x_train"] / 255.0).mean(axis=(0, 1, 2)), (SUBSET["x_train"] / 255.0).std(axis=(0, 1, 2))
    ok_9["stats"] = (len(mean_9) == len(std_9) == 3 and all(type(v) is float for v in [*mean_9, *std_9])
                     and np.allclose(mean_9, want_9[0], atol=1e-10) and np.allclose(std_9, want_9[1], atol=1e-10))
    verdict("B7.9", ok_9["stats"], f"channel_stats : moyennes {np.round(mean_9, 4).tolist()}, écarts-types "
            f"{np.round(std_9, 4).tolist()} (trois floats Python chacun).",
            f"channel_stats rend {mean_9!r}, {std_9!r} : attendu trois floats Python chacun, moyenne et écart-type "
            "(divisé par n) des pixels de chaque canal dans [0, 1].")
    four_9 = SUBSET["x_val"][:4]
    want_x_9 = ((four_9 / 255.0 - np.array(want_9[0])) / np.array(want_9[1])).transpose(0, 3, 1, 2)
    x_9 = data_9.preprocess(four_9.copy(), tuple(want_9[0]), tuple(want_9[1]))
    ok_9["preprocess"] = (isinstance(x_9, torch.Tensor) and x_9.dtype == torch.float32
                          and same("B7.9", "preprocess de 4 images de validation", x_9, want_x_9, rtol=0, atol=1e-5))
    if not (isinstance(x_9, torch.Tensor) and x_9.dtype == torch.float32):
        verdict("B7.9", False, "", f"preprocess doit rendre un tenseur float32, reçu {type(x_9).__name__} "
                f"{getattr(x_9, 'dtype', '')}.")
    one_9 = data_9.preprocess(four_9[0], tuple(want_9[0]), tuple(want_9[1]))
    ok_9["one"] = tuple(getattr(one_9, "shape", ())) == (1, 3, 32, 32)
    verdict("B7.9", ok_9["one"], "preprocess d'une seule image (32, 32, 3) : (1, 3, 32, 32).",
            f"preprocess d'une seule image (32, 32, 3) : attendu (1, 3, 32, 32), reçu {tuple(getattr(one_9, 'shape', ()))}.")
    ok_9["float"] = refuses("B7.9", "preprocess d'images en float", lambda: data_9.preprocess(four_9 / 255.0, *want_9))

    def order_9(seed, global_seed):
        torch.manual_seed(global_seed)                  # the order must not depend on the global generator
        loader = data_9.make_loader(SUBSET["x_val"][:64], np.arange(64), *want_9, batch_size=16, shuffle=True, seed=seed)
        return np.concatenate([np.asarray(t) for _, t in loader])

    with wbp.torch_state_kept():
        plain_9 = data_9.make_loader(SUBSET["x_val"][:64], SUBSET["y_val"][:64], *want_9, batch_size=16)
        xb_9, tb_9 = next(iter(plain_9))
        a_9, b_9, c_9 = order_9(0, 1), order_9(0, 2), order_9(1, 1)
    ok_9["loader"] = (tuple(xb_9.shape) == (16, 3, 32, 32) and xb_9.dtype == torch.float32 and tb_9.dtype == torch.int64
                      and np.allclose(xb_9.numpy(), data_9.preprocess(SUBSET["x_val"][:16], *want_9).numpy())
                      and np.array_equal(tb_9.numpy(), SUBSET["y_val"][:16]))
    verdict("B7.9", ok_9["loader"], "make_loader sans mélange : les 16 premières images prétraitées et leurs labels, "
            "float32 et int64.", f"make_loader sans mélange : attendu le batch (16, 3, 32, 32) float32 des 16 premières "
            f"images prétraitées et leurs labels int64, reçu {tuple(xb_9.shape)} {xb_9.dtype}, {tb_9.dtype}.")
    ok_9["order"] = sorted(a_9) == list(range(64)) and np.array_equal(a_9, b_9) and not np.array_equal(a_9, c_9)
    verdict("B7.9", ok_9["order"], "make_loader mélangé : le même ordre pour la même graine, quel que soit le "
            "générateur global ; un autre pour une autre graine.",
            "make_loader mélangé : attendu le même ordre des 64 images pour seed=0 après torch.manual_seed(1) ou (2), "
            "et un autre pour seed=1 ; " + ("l'ordre suit le générateur global." if not np.array_equal(a_9, b_9)
                                           else "seed ne change pas l'ordre." if np.array_equal(a_9, c_9)
                                           else "des images manquent ou sont répétées."))
    Xs_9, ys_9 = data_9.synthetic(12, seed=3)
    ok_9["synthetic"] = (np.asarray(Xs_9).dtype == np.uint8 and np.asarray(Xs_9).shape == (12, 32, 32, 3)
                         and np.array_equal(Xs_9, np.random.default_rng(3).integers(0, 256, (12, 32, 32, 3), dtype=np.uint8))
                         and np.array_equal(ys_9, np.arange(12) % 10))
    verdict("B7.9", ok_9["synthetic"], "synthetic(12, seed=3) : 12 images uint8 de default_rng(3) et les labels i % 10.",
            "synthetic(12, seed=3) : attendu np.random.default_rng(3).integers(0, 256, (12, 32, 32, 3), "
            "dtype=np.uint8) et les labels np.arange(12) % 10.")

with checked("B7.9"):                                    # model.py
    model_9 = wbp.import_module(PROJECT, "model")
    for arch_9, widths_9 in (("cnn", (12, 24)), ("cnn", (8, 16, 32)), ("vgg", (16, 32, 64)), ("vgg", (8,))):
        mine_9 = model_9.build_model(arch_9, widths_9)
        want_m_9 = wbp.build_reference(arch_9, widths_9)
        shapes_9 = {k: tuple(v.shape) for k, v in mine_9.state_dict().items()}
        ref_shapes_9 = {k: tuple(v.shape) for k, v in want_m_9.state_dict().items()}
        out_9 = mine_9.eval()(torch.zeros(5, 3, 32, 32))
        ok_9[arch_9 + str(widths_9)] = shapes_9 == ref_shapes_9 and tuple(out_9.shape) == (5, 10)
        extra_9 = sorted(set(shapes_9) - set(ref_shapes_9))
        lacking_9 = sorted(set(ref_shapes_9) - set(shapes_9))
        other_9 = sorted(k for k in set(shapes_9) & set(ref_shapes_9) if shapes_9[k] != ref_shapes_9[k])
        diff_9 = "; ".join(part for part in [f"en trop : {extra_9[:4]}" if extra_9 else "",
                                             f"manquants : {lacking_9[:4]}" if lacking_9 else "",
                                             f"d'autres formes : {other_9[:4]}" if other_9 else ""] if part)
        verdict("B7.9", ok_9[arch_9 + str(widths_9)],
                f"build_model({arch_9!r}, {widths_9}) : {model_9.n_parameters(mine_9):_} paramètres, les mêmes couches "
                "que l'architecture du contrat, sortie (5, 10).".replace("_", " "),
                f"build_model({arch_9!r}, {widths_9}) : les noms et les formes des tenseurs du state_dict diffèrent de "
                f"ceux de l'architecture du contrat ({diff_9 or 'mêmes tenseurs'}) ; sortie {tuple(out_9.shape)}, "
                "attendu (5, 10).")
    n_9 = model_9.n_parameters(wbp.build_reference("cnn", (12, 24)))
    ok_9["n_parameters"] = n_9 == sum(p.numel() for p in wbp.build_reference("cnn", (12, 24)).parameters())
    verdict("B7.9", ok_9["n_parameters"], "n_parameters compte les paramètres entraînables.",
            f"n_parameters du CNN (12, 24) de la fiche : reçu {n_9}, attendu le nombre de valeurs de ses paramètres.")
    ok_9["default"] = ({k: tuple(v.shape) for k, v in model_9.build_model().state_dict().items()}
                       == {k: tuple(v.shape) for k, v in wbp.build_reference("cnn", (12, 24)).state_dict().items()})
    verdict("B7.9", ok_9["default"], "build_model() : le CNN (12, 24) de la fiche.",
            "build_model() sans argument doit construire le CNN (12, 24) de la fiche (arch=\"cnn\", widths=(12, 24)).")
    ok_9["arch"] = refuses("B7.9", 'build_model("resnet")', lambda: model_9.build_model("resnet"))

TRAINED_9 = None
with checked("B7.9"):                                    # train.py, in this notebook
    train_9 = wbp.import_module(PROJECT, "train")
    small_9 = dict(arch="cnn", widths=(8, 16), epochs=2, lr=0.05, batch_size=64, seed=0)
    printed_9 = io.StringIO()
    with wbp.torch_state_kept(), contextlib.redirect_stdout(printed_9):
        r1_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **small_9)
        r2_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **{**small_9, "seed": 1})
        r3_9 = train_9.train(SUBSET["x_train"][:256], SUBSET["y_train"][:256], SUBSET["x_val"][:100],
                             SUBSET["y_val"][:100], **small_9)
    keys_9 = {"model", "mean", "std", "history", "seconds"}
    if not (isinstance(r1_9, dict) and keys_9 <= set(r1_9)):
        verdict("B7.9", False, "", f"train doit rendre un dictionnaire avec les clés {sorted(keys_9)}, reçu "
                f"{type(r1_9).__name__} {sorted(r1_9) if isinstance(r1_9, dict) else ''}.")
        raise StopCheck
    h_9 = r1_9["history"]
    ok_9["history"] = (isinstance(h_9, dict) and all(len(h_9.get(k, [])) == 2 for k in ("train_loss", "val_loss", "val_acc"))
                       and all(type(v) is float for k in ("train_loss", "val_loss", "val_acc") for v in h_9[k])
                       and all(0 <= v <= 1 for v in h_9["val_acc"]))
    verdict("B7.9", ok_9["history"], "train sur 256 images, 2 epochs : un historique de deux floats par courbe.",
            f"train sur 256 images, 2 epochs : attendu history = {{train_loss, val_loss, val_acc}}, deux floats "
            f"chacune (val_acc dans [0, 1]), reçu {h_9!r:.200}.")
    stats_256_9 = (SUBSET["x_train"][:256] / 255.0).mean(axis=(0, 1, 2))
    ok_9["mean"] = np.allclose(r1_9["mean"], stats_256_9, atol=1e-10)
    verdict("B7.9", ok_9["mean"], "la normalisation est apprise sur les images d'entraînement reçues.",
            "result['mean'] doit être channel_stats des images d'entraînement reçues par train (ni la validation, ni "
            "les statistiques d'ImageNet).")
    ok_9["eval"] = not r1_9["model"].training
    verdict("B7.9", ok_9["eval"], "le modèle rendu est en mode eval().", "le modèle rendu est encore en mode "
            "entraînement : relis la docstring de train.")
    ok_9["seed"] = wbp.same_weights(r1_9["model"].state_dict(), r3_9["model"].state_dict()) and \
        not wbp.same_weights(r1_9["model"].state_dict(), r2_9["model"].state_dict())
    verdict("B7.9", ok_9["seed"], "deux appels avec seed=0 donnent les mêmes poids, seed=1 d'autres.",
            "deux appels de train avec seed=0 doivent donner exactement les mêmes poids, et seed=1 d'autres poids "
            "(seed_everything au début de train, le générateur de make_loader).")
    ok_9["print"] = printed_9.getvalue() == ""
    verdict("B7.9", ok_9["print"], "train n'écrit rien avec print.",
            f"train écrit avec print ({printed_9.getvalue().strip()[:80]!r}…) : passe par log.info et log.debug.")
    with wbp.torch_state_kept():
        TRAINED_9 = train_9.train(SUBSET["x_train"], SUBSET["y_train"], SUBSET["x_val"], SUBSET["y_val"])
    acc_9, s_9 = TRAINED_9["history"]["val_acc"][-1], TRAINED_9["seconds"]
    ok_9["smoke"] = s_9 < 60 and acc_9 > 0.15
    verdict("B7.9", ok_9["smoke"], f"entraînement fumée (réglages par défaut, 2 000 images, 1 epoch) : val_acc "
            f"{fr(acc_9, 3)} en {fr(s_9, 1)} s.", f"entraînement fumée (réglages par défaut, 2 000 images, 1 epoch) : "
            f"val_acc {fr(acc_9, 3)} en {fr(s_9, 1)} s ; attendu plus de 0,15 (le hasard : 0,10) en moins de 60 s.")

with checked("B7.9"):                                    # pip install -e, in a new environment
    if PYPROJECT_TOML is ...:
        print("⏳ Ex B7.9 : l'installation attend ton pyproject.toml.")
        raise StopCheck
    install_9 = wbp.install_editable(PROJECT)
    try:
        if install_9.ok is None:
            print("⏳ Ex B7.9 : ni pip ni uv dans cet environnement : l'installation éditable n'est pas vérifiée ici.")
            raise StopCheck
        ok_9["install"] = bool(install_9.ok)
        verdict("B7.9", install_9.ok, f"pip install -e . dans un environnement neuf ({install_9.method}, "
                f"{fr(install_9.seconds, 1)} s).", "pip install -e . échoue : " + install_9.log[-600:])
        if not install_9.ok:
            raise StopCheck
        probe_9 = ("import cifarprod, importlib.metadata as m, json; print(json.dumps([cifarprod.__file__, "
                   "m.version('cifarprod'), [e.value for e in m.entry_points(group='console_scripts') if e.name == "
                   "'cifarprod']]))")
        seen_9 = install_9.run(["python", "-c", probe_9], scratch("elsewhere"))
        where_9, version_9, entry_9 = json.loads(seen_9.stdout) if seen_9.ok else (None, None, None)
        ok_9["editable"] = seen_9.ok and Path(where_9).resolve().is_relative_to(PROJECT / "src")
        verdict("B7.9", ok_9["editable"], "depuis un autre dossier, import cifarprod lit ton dossier src/ (une "
                "installation éditable, pas une copie).", f"depuis un autre dossier, import cifarprod : {where_9 or seen_9.tail(1)}.")
        ok_9["metadata"] = version_9 == "0.1.0" and entry_9 == ["cifarprod.cli:main"]
        verdict("B7.9", ok_9["metadata"], "métadonnées installées : version 0.1.0, commande cifarprod → cifarprod.cli:main.",
                f"métadonnées installées : version {version_9!r} (attendu '0.1.0'), point d'entrée {entry_9!r} "
                "(attendu ['cifarprod.cli:main']).")
    finally:
        install_9.cleanup()
if ok_9 and all(ok_9.values()) and {"smoke", "editable", "metadata", "synthetic", "default"} <= set(ok_9):
    print("✅ Ex B7.9 : ton paquet s'installe, s'importe et s'entraîne : le notebook appelle, le paquet fait.")

ℹ️ fichiers écrits dans le projet : pyproject.toml, .gitignore, src/cifarprod/__init__.py, src/cifarprod/data.py, src/cifarprod/model.py, src/cifarprod/train.py


✅ Ex B7.9 : import cifarprod depuis src/ : version 0.1.0.
✅ Ex B7.9 : load_npz : les 2 000 images d'entraînement et les 500 de validation, uint8 et int64.
✅ Ex B7.9 : load_npz d'un fichier avec des images float32 : ValueError levée (expected uint8 images, got float32).
✅ Ex B7.9 : load_npz d'un fichier avec des images 32 × 32 × 1 : ValueError levée (expected images of shape (N, 32, 32, 3), got (4, 32, 32, 1)).
✅ Ex B7.9 : load_npz d'un fichier avec un label 10 : ValueError levée (expected labels in 0..9, got 0..10).
✅ Ex B7.9 : load_npz d'un fichier avec 3 labels pour 4 images : ValueError levée (expected 4 labels of shape (4,), got shape (3,)).
✅ Ex B7.9 : load_npz(..., "test") : ValueError levée (part must be one of ('train', 'val'), got 'test').
✅ Ex B7.9 : load_npz d'un fichier absent : FileNotFoundError levée (no data file /tmp/wb_b7_bad_l8v_0085/absent.npz).


✅ Ex B7.9 : fingerprint de 5 images et de leurs labels : 6bd5e8a92dfb…, comme hashlib sur le type, la forme et les octets.


✅ Ex B7.9 : channel_stats : moyennes [0.4924, 0.4803, 0.4418], écarts-types [0.2468, 0.2433, 0.261] (trois floats Python chacun).
✅ Ex B7.9 : preprocess de 4 images de validation : un tableau (4, 3, 32, 32), comme attendu.
✅ Ex B7.9 : preprocess d'une seule image (32, 32, 3) : (1, 3, 32, 32).
✅ Ex B7.9 : preprocess d'images en float : ValueError levée (preprocess expects uint8 images (0..255), got float64).
✅ Ex B7.9 : make_loader sans mélange : les 16 premières images prétraitées et leurs labels, float32 et int64.
✅ Ex B7.9 : make_loader mélangé : le même ordre pour la même graine, quel que soit le générateur global ; un autre pour une autre graine.
✅ Ex B7.9 : synthetic(12, seed=3) : 12 images uint8 de default_rng(3) et les labels i % 10.
✅ Ex B7.9 : build model('cnn', (12, 24)) : 18 322 paramètres, les mêmes couches que l'architecture du contrat, sortie (5, 10).
✅ Ex B7.9 : build model('cnn', (8, 16, 32)) : 11 162 paramètres, les mêmes couches que l'architecture du contrat, sortie (

✅ Ex B7.9 : train sur 256 images, 2 epochs : un historique de deux floats par courbe.
✅ Ex B7.9 : la normalisation est apprise sur les images d'entraînement reçues.
✅ Ex B7.9 : le modèle rendu est en mode eval().
✅ Ex B7.9 : deux appels avec seed=0 donnent les mêmes poids, seed=1 d'autres.
✅ Ex B7.9 : train n'écrit rien avec print.


✅ Ex B7.9 : entraînement fumée (réglages par défaut, 2 000 images, 1 epoch) : val_acc 0,302 en 0,8 s.


✅ Ex B7.9 : pip install -e . dans un environnement neuf (uv, 0,8 s).
✅ Ex B7.9 : depuis un autre dossier, import cifarprod lit ton dossier src/ (une installation éditable, pas une copie).
✅ Ex B7.9 : métadonnées installées : version 0.1.0, commande cifarprod → cifarprod.cli:main.
✅ Ex B7.9 : ton paquet s'installe, s'importe et s'entraîne : le notebook appelle, le paquet fait.


### Ex B7.10 — Une interface en ligne de commande : argparse, logging et codes de sortie 📦 ★★ ⏱️ 30 min
**Objectif :** écrire la ligne de commande d'un entraînement : options, logging et codes de sortie qu'un autre programme sait lire.  
**Prérequis :** Ex B7.9 · fiche §207.2 (« argparse et le point d'entrée », « Le logging plutôt que print », « Les codes de sortie ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Le contrat.** Deux nouveaux fichiers ; la configuration, `config.py`, est fournie dans sa version de départ par la cellule suivante (une dataclass `TrainConfig` avec les réglages et leurs valeurs par défaut — `seed`, `epochs`, `lr`, `batch_size`, `n_train`, `arch`, `widths`, `data`, `deterministic`, `num_workers` —, et `load_config(path, overrides)`, sans aucune vérification des types ni des bornes : c'est B7.11). Elle n'est écrite que si `config.py` n'existe pas encore.

- `__main__.py` : `python -m cifarprod ...` appelle `main()` et transmet son code de sortie (207.2).
- `cli.py` : `build_parser()` rend un `ArgumentParser` de `prog="cifarprod"`, avec `--version` (« cifarprod 0.1.0 ») et la sous-commande **obligatoire** `train`, de quatre options : `--config FILE` (défaut `None` : les valeurs par défaut de `TrainConfig`), `--set KEY=VALUE` (répétable, défaut `[]`), `--out DIR` (défaut `runs/latest`) et `-v`/`--verbose`.
- `main(argv=None)` lit `argv` (`None` : `sys.argv`), configure le logging **une fois** (`logging.basicConfig`, niveau INFO, DEBUG avec `-v`, un format qui montre le niveau et le nom du logger : `%(levelname)s %(name)s: %(message)s`, sur stderr), puis : `load_config(args.config, args.set)` ; une `ValueError` ou une `OSError` → une ligne ERROR et le code **2** ; les parties `"train"` et `"val"` de `config.data` (un chemin relatif au dossier courant), dont il garde les `n_train` premières images d'entraînement ; des données illisibles ou trop peu d'images → une ligne ERROR et le code **1** ; `train(...)` avec les réglages de la configuration ; les poids dans `<out>/model.safetensors` (`safetensors.torch.save_file` du `state_dict`) ; une ligne INFO ; et sur stdout **une seule** ligne, le résumé JSON `{"status": "ok", "val_acc": <float>, "weights": <chemin absolu du fichier de poids>}` ; le code **0**. Toute autre exception pendant l'entraînement : `log.exception(...)` et le code 1. `main` **rend** son code (il n'appelle pas `sys.exit`, sauf argparse devant une mauvaise option, qui sort avec 2).

La cellule de vérification lance ta commande dans un autre processus (`python -m cifarprod`, depuis le dossier du projet, ton `src/` dans le `PYTHONPATH`) : `--version`, `--help`, sans sous-commande, avec une option inconnue, `--set lr` (sans `=`), des données absentes, un entraînement sur 200 images (son JSON, son fichier de poids, ses lignes INFO et l'absence de DEBUG), le même avec `-v` ; elle lit chaque fois le code de sortie, stdout et stderr. Puis elle appelle `main(["train", "--set", "lr"])` dans le notebook, qui doit rendre l'entier 2.

Dans tes notes : pourquoi le résumé va-t-il sur stdout et tout le reste sur stderr (pense à un script qui lit ce JSON) ? Pourquoi `main(argv)` reçoit-il ses arguments au lieu de lire `sys.argv`, et pourquoi rend-il son code au lieu d'appeler `sys.exit` ? Une intégration continue lance ta commande : que se passe-t-il si un entraînement qui échoue affiche son erreur mais rend 0 ?

In [23]:
# the starting version of src/cifarprod/config.py, written by the check cell only if config.py does not exist yet (Ex B7.11 replaces it)
CONFIG_V0 = r'''"""The configuration of cifarprod, STARTING VERSION (given in Ex B7.10, replaced in Ex B7.11): the settings and their
defaults, read from a YAML file and the overrides "KEY=VALUE", without any check of the types or of the ranges."""

from __future__ import annotations

from dataclasses import asdict, dataclass, fields
from pathlib import Path

import yaml


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    epochs: int = 1
    lr: float = 0.05
    batch_size: int = 128
    n_train: int = 2000
    arch: str = "cnn"
    widths: tuple = (16, 32)
    data: str = "data/cifar10_2000.npz"
    deterministic: bool = True
    num_workers: int = 0

    def to_dict(self) -> dict:
        out = asdict(self)
        out["widths"] = list(self.widths)
        return out


def load_config(path=None, overrides=()):
    raw = dict(yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}) if path is not None else {}
    for item in overrides:
        key, sep, value = str(item).partition("=")
        if not sep:
            raise ValueError(f"--set expects KEY=VALUE, got {item!r}")
        raw[key.strip()] = yaml.safe_load(value)
    unknown = set(raw) - {f.name for f in fields(TrainConfig)}
    if unknown:
        raise ValueError(f"unknown settings: {sorted(unknown)}")
    return TrainConfig(**raw)
'''

In [24]:
# Ex B7.10: the command line of the reference package. Its run journal (the import of .tracking, --runs, the record
# of each run) belongs to Ex B7.14: the solutions notebook writes the final cli.py at once, and tracking.py with it.

# src/cifarprod/__main__.py
MAIN_PY = r'''"""python -m cifarprod ...: the same command line as the `cifarprod` entry point."""

from .cli import main

raise SystemExit(main())
'''

# src/cifarprod/cli.py
CLI_PY = r'''"""The command line of cifarprod.

    cifarprod train --config configs/fast.yaml --set lr=0.01 -v      # or: python -m cifarprod train ...
    cifarprod --version

Exit codes: 0 success; 2 bad usage (argparse) or invalid configuration; 1 the data cannot be read or the training
failed. Messages go through logging (stderr); the only line written on stdout is the JSON summary of the run.
"""

from __future__ import annotations

import argparse
import json
import logging
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

from . import __version__
from .config import load_config
from .tracking import artifact, environment, log_run, new_run_id

log = logging.getLogger(__name__)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="cifarprod", description="Train a CIFAR-10 classifier.")
    parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")
    sub = parser.add_subparsers(dest="command", required=True)
    train = sub.add_parser("train", help="train a model, save its weights and log the run")
    train.add_argument("--config", default=None, help="YAML file of settings (default: the defaults of TrainConfig)")
    train.add_argument("--set", action="append", default=[], metavar="KEY=VALUE", help="override a setting")
    train.add_argument("--out", default=None, help="folder of the weights (default: runs/<run id>)")
    train.add_argument("--runs", default="runs/runs.jsonl", help="the run journal (default: %(default)s)")
    train.add_argument("-v", "--verbose", action="store_true", help="also show the DEBUG messages")
    return parser


def run_train(args: argparse.Namespace) -> int:
    try:
        config = load_config(args.config, args.set)
    except (OSError, ValueError) as err:
        log.error("invalid configuration: %s", err)
        return 2
    run_id = new_run_id()
    out = Path(args.out) if args.out else Path("runs") / run_id
    record = {"run_id": run_id, "started": datetime.now(timezone.utc).isoformat(timespec="seconds"),
              "config": config.to_dict(), "environment": environment(), "status": "failed"}
    start = time.perf_counter()
    try:
        from safetensors.torch import save_file

        from .data import fingerprint, load_npz
        from .train import train

        try:
            X, y = load_npz(config.data, "train")
            X_val, y_val = load_npz(config.data, "val")
            if config.n_train > len(X):
                raise ValueError(f"n_train = {config.n_train}, but {config.data} has {len(X)} training images")
        except (OSError, ValueError) as err:
            log.error("cannot load the data: %s", err)
            record["error"] = f"{type(err).__name__}: {err}"
            return 1
        X, y = X[:config.n_train], y[:config.n_train]
        log.info("loaded %d training and %d validation images from %s", len(X), len(X_val), config.data)
        record["data"] = {"path": config.data, "n_train": len(X), "sha256": fingerprint(X, y)}
        result = train(X, y, X_val, y_val, arch=config.arch, widths=config.widths, epochs=config.epochs,
                       lr=config.lr, batch_size=config.batch_size, seed=config.seed,
                       deterministic=config.deterministic, num_workers=config.num_workers)
        out.mkdir(parents=True, exist_ok=True)
        weights = out / "model.safetensors"
        save_file(result["model"].state_dict(), str(weights))
        history = result["history"]
        record.update(status="ok", metrics={**history, "val_acc_final": history["val_acc"][-1]},
                      normalization={"mean": list(result["mean"]), "std": list(result["std"])},
                      artifacts=[artifact(weights)])
        log.info("run %s: val_acc %.4f, weights in %s", run_id, history["val_acc"][-1], weights)
        print(json.dumps({"run_id": run_id, "status": "ok", "val_acc": history["val_acc"][-1],
                          "weights": str(weights.resolve())}))
        return 0
    except Exception as err:                      # noqa: BLE001 - logged with its traceback, exit code 1
        log.exception("training failed")
        record["error"] = f"{type(err).__name__}: {err}"
        return 1
    finally:
        record["seconds"] = round(time.perf_counter() - start, 3)
        log_run(args.runs, record)                # every run is logged, the failed ones too


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)        # a bad option: a message and exit code 2
    logging.basicConfig(level=logging.DEBUG if args.verbose else logging.INFO, stream=sys.stderr,
                        format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    return run_train(args)


if __name__ == "__main__":
    sys.exit(main())
'''

# src/cifarprod/tracking.py (Ex B7.14)
TRACKING_PY = r'''"""The run journal of cifarprod: one JSON line per run, appended to a JSONL file, never rewritten.

    run_id = new_run_id()                              # "20261104-161731-e06339"
    log_run(Path("runs/runs.jsonl"), {"run_id": run_id, "status": "ok", ...})
    runs = read_runs(Path("runs/runs.jsonl"))          # a list of dictionaries, oldest first

A run records what defines it (configuration, environment, data fingerprint), what it measures (metrics per epoch,
duration) and what it produces (its artifacts, with their path, size and fingerprint).
"""

from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import uuid
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path


def new_run_id() -> str:
    """A unique run identifier: the UTC date and time, then six random hexadecimal characters."""
    return f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:6]}"


def file_sha256(path: str | Path) -> str:
    """The SHA-256 of the bytes of a file (64 hexadecimal characters)."""
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def artifact(path: str | Path) -> dict:
    """{"path", "bytes", "sha256"} of a file written by the run."""
    path = Path(path)
    return {"path": str(path), "bytes": path.stat().st_size, "sha256": file_sha256(path)}


def _git(*args: str, cwd=None) -> str | None:
    try:
        done = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, timeout=10)
    except (OSError, subprocess.SubprocessError):
        return None
    return done.stdout.strip() if done.returncode == 0 else None


def environment(cwd=None) -> dict:
    """What runs the code: the versions of Python, NumPy and PyTorch, the platform, the git commit (None outside a
    repository) and whether the working tree has uncommitted changes."""
    versions = {}
    for name in ("numpy", "torch"):
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    status = _git("status", "--porcelain", cwd=cwd)
    return {"python": platform.python_version(), **versions, "platform": platform.platform(),
            "commit": _git("rev-parse", "--short", "HEAD", cwd=cwd), "dirty": bool(status) if status is not None else None}


def log_run(path: str | Path, record: dict) -> None:
    """Append one JSON line (keys sorted) to the journal ``path``; its folder is created if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, sort_keys=True) + "\n")


def read_runs(path: str | Path) -> list[dict]:
    """The runs of the journal, oldest first ([] if the file does not exist)."""
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
'''


ok_10 = {}
with checked("B7.10"):
    wbp.write_file(PROJECT, "src/cifarprod/config.py", CONFIG_V0, if_absent=True)
    files_10 = {"src/cifarprod/__main__.py": MAIN_PY, "src/cifarprod/cli.py": CLI_PY}
    if "TRACKING_PY" in globals() and TRACKING_PY is not ...:
        files_10["src/cifarprod/tracking.py"] = TRACKING_PY          # Ex B7.14, if it is already done
    wbp.write_files(PROJECT, files_10)
    cli_10 = wbp.import_module(PROJECT, "cli")
    cli_10.build_parser()                                 # ⏳ while it raises NotImplementedError
    out_10 = scratch("out")
    r_10 = wbp.cli(PROJECT, "--version")
    ok_10["version"] = run_line("B7.10", r_10, r_10.ok and "0.1.0" in r_10.stdout, "code 0, « cifarprod 0.1.0 ».",
                                "attendu le code 0 et la version 0.1.0 sur stdout")
    r_10 = wbp.cli(PROJECT, "--help")
    ok_10["help"] = run_line("B7.10", r_10, r_10.ok and "train" in r_10.stdout, "code 0, l'aide cite train.",
                             "attendu le code 0 et une aide qui cite la sous-commande train")
    r_10 = wbp.cli(PROJECT)
    ok_10["no command"] = run_line("B7.10", r_10, r_10.returncode == 2 and "usage" in r_10.stderr.lower(),
                                   "code 2 et l'usage sur stderr (la sous-commande est obligatoire).",
                                   "attendu le code 2 d'argparse : la sous-commande train est obligatoire")
    r_10 = wbp.cli(PROJECT, "train", "--epochs", "3")
    ok_10["unknown option"] = run_line("B7.10", r_10, r_10.returncode == 2, "code 2 (option inconnue).",
                                       "attendu le code 2 d'argparse pour une option inconnue")
    r_10 = wbp.cli(PROJECT, "train", "--set", "lr", "--out", out_10 / "a")
    ok_10["bad set"] = run_line("B7.10", r_10, r_10.returncode == 2 and r_10.stderr_lines("ERROR") and not r_10.stdout.strip(),
                                "code 2, une ligne ERROR, rien sur stdout.",
                                "attendu le code 2 (configuration invalide), une ligne ERROR sur stderr et rien sur stdout")
    r_10 = wbp.cli(PROJECT, "train", "--set", "data=data/absent.npz", "--out", out_10 / "b")
    ok_10["no data"] = run_line("B7.10", r_10, r_10.returncode == 1 and r_10.stderr_lines("ERROR") and not r_10.stdout.strip(),
                                "code 1, une ligne ERROR, rien sur stdout.",
                                "attendu le code 1 (données illisibles), une ligne ERROR sur stderr et rien sur stdout")
    r_10 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_10 / "c")
    lines_10 = [line for line in r_10.stdout.splitlines() if line.strip()]
    summary_10 = r_10.last_json()
    ok_json_10 = (len(lines_10) == 1 and isinstance(summary_10, dict) and summary_10.get("status") == "ok"
                  and isinstance(summary_10.get("val_acc"), float) and 0 <= summary_10["val_acc"] <= 1)
    ok_10["train"] = run_line("B7.10", r_10, r_10.ok and ok_json_10,
                              f"code 0, une ligne JSON sur stdout (val_acc {fr(summary_10['val_acc'], 3) if ok_json_10 else '?'}).",
                              "attendu le code 0 et UNE ligne JSON sur stdout, {\"status\": \"ok\", \"val_acc\": <float>, "
                              f"\"weights\": ...}} ; reçu {r_10.stdout.strip()[:200]!r}")
    if ok_10["train"]:
        weights_10 = Path(summary_10.get("weights", ""))
        from safetensors.torch import load_file
        ok_10["weights"] = (weights_10.is_absolute() and weights_10 == (out_10 / "c" / "model.safetensors").resolve()
                            and weights_10.is_file() and set(load_file(str(weights_10)))
                            == set(wbp.build_reference("cnn", (16, 32)).state_dict()))
        verdict("B7.10", ok_10["weights"], "les poids sont dans <out>/model.safetensors (chemin absolu dans le JSON), "
                "ceux du CNN (16, 32).", f"attendu \"weights\" = le chemin absolu de <out>/model.safetensors, un "
                f"fichier safetensors du CNN (16, 32) ; reçu {str(weights_10)!r}.")
        infos_10 = [line for line in r_10.stderr_lines("INFO") if "cifarprod.train" in line]
        ok_10["logging"] = bool(infos_10) and not r_10.stderr_lines("DEBUG")
        verdict("B7.10", ok_10["logging"], f"stderr : {len(r_10.stderr_lines('INFO'))} lignes INFO, dont "
                f"« {infos_10[-1].split(': ', 1)[-1][:60] if infos_10 else ''} » de cifarprod.train ; aucune DEBUG.",
                "attendu sur stderr des lignes INFO (au moins une du logger cifarprod.train, avec le niveau et le nom "
                f"du logger) et aucune ligne DEBUG sans -v ; reçu {r_10.tail(3)[:300]!r}")
        r_10 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_10 / "d", "-v")
        ok_10["verbose"] = run_line("B7.10", r_10, r_10.ok and any("cifarprod" in line for line in r_10.stderr_lines("DEBUG")),
                                    "avec -v, les lignes DEBUG des mini-batches.",
                                    "attendu, avec -v, des lignes DEBUG du paquet sur stderr")
    with wbp.isolated_logging():                         # main() configures the logging of THIS process
        try:
            code_10 = cli_10.main(["train", "--set", "lr"])
        except SystemExit as exit_10:                     # sys.exit inside main: the notebook would stop there
            code_10 = f"SystemExit({exit_10.code!r})"
    ok_10["main"] = type(code_10) is int and code_10 == 2
    verdict("B7.10", ok_10["main"], 'main(["train", "--set", "lr"]) rend l\'entier 2 dans le notebook.',
            f'main(["train", "--set", "lr"]) doit rendre l\'entier 2 (sans sys.exit) ; reçu {code_10!r}.')
if ok_10 and all(ok_10.values()):
    print("✅ Ex B7.10 : ta ligne de commande parle avec des codes de sortie et du logging.")

✅ Ex B7.10 : cifarprod --version → code 0, « cifarprod 0.1.0 ».
✅ Ex B7.10 : cifarprod --help → code 0, l'aide cite train.


✅ Ex B7.10 : cifarprod → code 2 et l'usage sur stderr (la sous-commande est obligatoire).
✅ Ex B7.10 : cifarprod train --epochs 3 → code 2 (option inconnue).


✅ Ex B7.10 : cifarprod train --set lr --out a → code 2, une ligne ERROR, rien sur stdout.


✅ Ex B7.10 : cifarprod train --set data=data/absent.npz --out b → code 1, une ligne ERROR, rien sur stdout.


✅ Ex B7.10 : cifarprod train --set n_train=200 --out c → code 0, une ligne JSON sur stdout (val_acc 0,136).
✅ Ex B7.10 : les poids sont dans <out>/model.safetensors (chemin absolu dans le JSON), ceux du CNN (16, 32).
✅ Ex B7.10 : stderr : 4 lignes INFO, dont « epoch 1/1 train_loss 2.2952 val_loss 2.2808 val_acc 0.1360 » de cifarprod.train ; aucune DEBUG.


✅ Ex B7.10 : cifarprod train --set n_train=200 --out d -v → avec -v, les lignes DEBUG des mini-batches.
✅ Ex B7.10 : main(["train", "--set", "lr"]) rend l'entier 2 dans le notebook.
✅ Ex B7.10 : ta ligne de commande parle avec des codes de sortie et du logging.


### Ex B7.11 — Configuration : dataclass + YAML, surcharges en ligne de commande, validation 📦 ★★ ⏱️ 30 min
**Objectif :** remplacer les réglages en dur par une configuration typée, validée au chargement et surchargeable.  
**Prérequis :** Ex B7.10 · fiche §207.3 (le code de `TrainConfig` et de `load_config`, « Rappel outil — YAML avec PyYAML ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

**Le contrat** de `config.py` (il remplace la version de départ de B7.10 ; `cli.py` ne change pas) :

- `TrainConfig` est une dataclass **figée** (`frozen=True`) aux dix champs et aux valeurs par défaut de la version de départ (`seed=0`, `epochs=1`, `lr=0.05`, `batch_size=128`, `n_train=2000`, `arch="cnn"`, `widths=(16, 32)`, `data="data/cifar10_2000.npz"`, `deterministic=True`, `num_workers=0`) ;
- à la construction (`__post_init__`), chaque réglage a **exactement** le type de sa valeur par défaut (`True` n'est pas un entier, la chaîne `"1e-3"` n'est pas un float), sauf deux conversions permises : un entier pour un float (`lr=1` devient `1.0`) et une liste pour un tuple (`widths=[8, 16]` devient `(8, 16)`) ; puis les bornes : `seed` ≥ 0, `epochs` ≥ 1, $0 <$ `lr` $< 1$, `batch_size` ≥ 1, `n_train` ≥ 1, `num_workers` ≥ 0, `arch` dans `("cnn", "vgg")`, `widths` non vide et fait d'entiers ≥ 1. Toute violation lève une `ValueError` dont le message **nomme le réglage** ;
- `to_dict()` rend un dictionnaire que `json.dumps` accepte (`widths` en liste) ;
- `load_config(path=None, overrides=())` part des valeurs par défaut, puis du fichier YAML `path` (`yaml.safe_load` ; un fichier vide donne les valeurs par défaut ; un fichier qui n'est pas un dictionnaire, `ValueError` ; un fichier absent, `FileNotFoundError`), puis des surcharges `"KEY=VALUE"` (chaque valeur lue par `yaml.safe_load`, donc `widths=[8, 16]` est une liste ; une surcharge sans `=`, `ValueError`) ; une clé inconnue, du fichier ou d'une surcharge, lève une `ValueError` qui la nomme.

Deux fichiers de configuration aussi : `configs/fast.yaml`, l'entraînement fumée du notebook, dont les réglages sont **exactement** les valeurs par défaut (écrits dans le fichier, pour qu'il se suffise) ; `configs/smoke.yaml`, l'essai de bout en bout de l'intégration continue : les mêmes, sauf `n_train: 256`, `batch_size: 64` et `widths: [8, 16]`.

La cellule de vérification essaie ta configuration dans le notebook : valeurs par défaut et `to_dict` en JSON, dataclass figée, conversions, seize surcharges invalides (chacune doit lever une `ValueError` qui nomme le réglage), l'ordre défauts → fichier → surcharges, trois fichiers pièges (`lr: 1e-3`, une faute de frappe, une liste), un fichier vide et un fichier absent, puis tes deux fichiers YAML ; enfin ta commande avec `--set epochs=0` (code 2, avant toute seconde de calcul) et `--config configs/smoke.yaml` (code 0).

Dans tes notes : que lit PyYAML dans `lr: 1e-3`, et pourquoi la validation des types est-elle la seule chose qui te sauve ici ? Pourquoi refuser une clé inconnue plutôt que l'ignorer ? Quels réglages ne mettrais-tu **jamais** dans ce fichier, et où iraient-ils (207.9) ?

In [25]:
# Ex B7.11: the typed configuration of the reference package, and its two configuration files

# src/cifarprod/config.py
CONFIG_PY = r'''"""The configuration of cifarprod: a typed, frozen dataclass, read from a YAML file and overridden on the command line.

    config = load_config("configs/fast.yaml", ["lr=0.01", "widths=[8, 16]"])
    config.lr, config.to_dict()

Every setting is checked when the configuration is built: its type, its range, and no unknown key (a typo fails
instead of being ignored). A secret never goes here: a configuration is versioned and copied into the run journal.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, fields
from pathlib import Path

import yaml

ARCHS = ("cnn", "vgg")


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    epochs: int = 1
    lr: float = 0.05
    batch_size: int = 128
    n_train: int = 2000
    arch: str = "cnn"
    widths: tuple = (16, 32)
    data: str = "data/cifar10_2000.npz"
    deterministic: bool = True
    num_workers: int = 0

    def __post_init__(self) -> None:
        for f in fields(self):                          # a dataclass does not check the types: we do
            value, kind = getattr(self, f.name), type(f.default)
            if kind is float and type(value) is int:
                object.__setattr__(self, f.name, float(value))          # 1 -> 1.0 is fine
            elif kind is tuple and type(value) is list:
                object.__setattr__(self, f.name, tuple(value))          # a YAML list -> a tuple
            elif type(value) is not kind:                               # True is not an int here
                raise ValueError(f"{f.name}: expected {kind.__name__}, got {value!r}")
        if not self.widths or any(type(w) is not int or w < 1 for w in self.widths):
            raise ValueError(f"widths: expected positive whole numbers, got {self.widths!r}")
        if self.arch not in ARCHS:
            raise ValueError(f"arch: expected one of {ARCHS}, got {self.arch!r}")
        if self.seed < 0 or self.epochs < 1 or self.batch_size < 1 or self.n_train < 1 or self.num_workers < 0:
            raise ValueError(f"need seed >= 0, epochs >= 1, batch_size >= 1, n_train >= 1 and num_workers >= 0, got "
                             f"{self.seed}, {self.epochs}, {self.batch_size}, {self.n_train} and {self.num_workers}")
        if not 0 < self.lr < 1:
            raise ValueError(f"lr: expected 0 < lr < 1, got {self.lr}")

    def to_dict(self) -> dict:
        """The settings as a plain dictionary that json.dumps accepts (widths as a list)."""
        out = asdict(self)
        out["widths"] = list(self.widths)
        return out


def load_config(path: str | Path | None = None, overrides=()) -> TrainConfig:
    """The TrainConfig of the YAML file ``path`` (None: the defaults), then of the overrides "KEY=VALUE" (each VALUE
    read by yaml.safe_load). ValueError for a file that is not a mapping, an override without "=", an unknown key or a
    setting refused by TrainConfig; FileNotFoundError for a missing file."""
    raw = {}
    if path is not None:
        loaded = yaml.safe_load(Path(path).read_text(encoding="utf-8"))
        if loaded is not None and not isinstance(loaded, dict):
            raise ValueError(f"{path}: expected a mapping of settings, got {type(loaded).__name__}")
        raw.update(loaded or {})
    for item in overrides:
        key, sep, value = str(item).partition("=")
        if not sep or not key.strip():
            raise ValueError(f"--set expects KEY=VALUE, got {item!r}")
        raw[key.strip()] = yaml.safe_load(value)
    unknown = set(raw) - {f.name for f in fields(TrainConfig)}
    if unknown:
        raise ValueError(f"unknown settings: {sorted(unknown)}")
    return TrainConfig(**raw)
'''

# configs/fast.yaml
FAST_YAML = r'''# configs/fast.yaml: the smoke training of the notebook (2 000 images, 1 epoch, < 1 min on a CPU)
seed: 0
epochs: 1
lr: 0.05
batch_size: 128
n_train: 2000
arch: cnn
widths: [16, 32]
data: data/cifar10_2000.npz
'''

# configs/smoke.yaml
SMOKE_YAML = r'''# configs/smoke.yaml: the end-to-end test of the continuous integration (a few seconds: it judges no accuracy)
seed: 0
epochs: 1
lr: 0.05
batch_size: 64
n_train: 256
arch: cnn
widths: [8, 16]
data: data/cifar10_2000.npz
'''


import dataclasses

ok_11 = {}
DEFAULTS_11 = DEFAULTS
with checked("B7.11"):
    wbp.write_files(PROJECT, {"src/cifarprod/config.py": CONFIG_PY, "configs/fast.yaml": FAST_YAML,
                              "configs/smoke.yaml": SMOKE_YAML})
    config_11 = wbp.import_module(PROJECT, "config")
    default_11 = config_11.TrainConfig()
    ok_11["defaults"] = default_11.to_dict() == DEFAULTS_11 and json.loads(json.dumps(default_11.to_dict())) == DEFAULTS_11
    verdict("B7.11", ok_11["defaults"], "TrainConfig() et to_dict() : les valeurs par défaut du contrat, en JSON.",
            f"TrainConfig().to_dict() doit valoir {DEFAULTS_11} ; reçu {default_11.to_dict()!r:.300}.")
    try:
        default_11.lr = 0.1
        ok_11["frozen"] = False
    except dataclasses.FrozenInstanceError:
        ok_11["frozen"] = True
    verdict("B7.11", ok_11["frozen"], "la configuration est figée (frozen=True).",
            "config.lr = 0.1 doit lever FrozenInstanceError : un run ne modifie pas sa configuration en route.")
    cast_11 = config_11.TrainConfig(lr=1 / 2 ** 4, epochs=2, widths=[8, 16])
    ok_11["cast"] = type(config_11.TrainConfig(seed=0, lr=0.5).lr) is float and cast_11.widths == (8, 16) \
        and type(config_11.TrainConfig(**{"lr": 1e-1}).lr) is float
    lr_int_11 = config_11.load_config(None, ["batch_size=64", "lr=0.25"])
    verdict("B7.11", ok_11["cast"] and lr_int_11.batch_size == 64, "un entier accepté pour un float, une liste YAML "
            "devient un tuple.", "attendu lr en float, widths=[8, 16] devenu (8, 16), et --set batch_size=64.")
    for override_11, name_11 in [("epochs=0", "epochs"), ("epochs=1.5", "epochs"), ("lr=0", "lr"), ("lr=1", "lr"),
                                 ("lr=1e-3", "lr"), ("batch_size=0", "batch_size"), ("batch_size=true", "batch_size"),
                                 ("n_train=0", "n_train"), ("seed=-1", "seed"), ("arch=resnet", "arch"),
                                 ("widths=[]", "widths"), ("widths=[8, 0]", "widths"), ("deterministic=yes2", "deterministic"),
                                 ("num_workers=-1", "num_workers"), ("lerning_rate=0.01", "lerning_rate"), ("lr", "lr")]:
        try:
            config_11.load_config(None, [override_11])
            ok_11[override_11] = False
            print(f"❌ Ex B7.11 : --set {override_11} : attendu ValueError, mais rien n'a été levé.")
        except ValueError as err_11:
            ok_11[override_11] = name_11 in str(err_11)
            if not ok_11[override_11]:
                print(f"❌ Ex B7.11 : --set {override_11} : ValueError levée, mais son message ne nomme pas {name_11!r} : "
                      f"« {str(err_11)[:120]} ».")
        except NotImplementedError:
            raise
        except Exception as err_11:  # noqa: BLE001 - the learner's code
            ok_11[override_11] = False
            print(f"❌ Ex B7.11 : --set {override_11} : attendu ValueError, reçu {type(err_11).__name__} : {str(err_11)[:120]}")
    refused_11 = [k for k in ok_11 if "=" in k or k == "lr"]
    if all(ok_11[k] for k in refused_11):
        print(f"✅ Ex B7.11 : {len(refused_11)} réglages invalides refusés, chacun par une ValueError qui le nomme "
              "(types, bornes, clé inconnue, --set sans =).")
    yaml_11 = scratch("yaml")
    (yaml_11 / "a.yaml").write_text("lr: 0.1\nepochs: 3\nwidths: [4, 8]\n", encoding="utf-8")
    both_11 = config_11.load_config(yaml_11 / "a.yaml", ["lr=0.01", "epochs=2"])
    ok_11["precedence"] = (both_11.lr, both_11.epochs, both_11.widths, both_11.seed) == (0.01, 2, (4, 8), 0)
    verdict("B7.11", ok_11["precedence"], "les valeurs par défaut, puis le fichier, puis les --set.",
            f"fichier lr 0.1, epochs 3, widths [4, 8] puis --set lr=0.01 epochs=2 : attendu (0.01, 2, (4, 8), seed 0), "
            f"reçu {(both_11.lr, both_11.epochs, both_11.widths, both_11.seed)}.")
    for text_11, what_11 in [("lr: 1e-3\n", "lr: 1e-3 (une chaîne pour PyYAML)"), ("lerning_rate: 0.01\n", "une faute de frappe"),
                             ("- lr\n- 0.1\n", "une liste au lieu d'un dictionnaire")]:
        (yaml_11 / "bad.yaml").write_text(text_11, encoding="utf-8")
        ok_11[what_11] = refuses("B7.11", f"un fichier avec {what_11}", lambda: config_11.load_config(yaml_11 / "bad.yaml"))
    (yaml_11 / "empty.yaml").write_text("# nothing\n", encoding="utf-8")
    ok_11["empty"] = config_11.load_config(yaml_11 / "empty.yaml").to_dict() == DEFAULTS_11
    verdict("B7.11", ok_11["empty"], "un fichier vide : les valeurs par défaut.", "un fichier vide (un commentaire) doit "
            "donner les valeurs par défaut (yaml.safe_load rend None).")
    ok_11["absent"] = refuses("B7.11", "un fichier absent", lambda: config_11.load_config(yaml_11 / "absent.yaml"),
                              error=FileNotFoundError)
    want_files_11 = {"configs/fast.yaml": DEFAULTS_11,
                     "configs/smoke.yaml": {**DEFAULTS_11, "n_train": 256, "batch_size": 64, "widths": [8, 16]}}
    for rel_11, want_11 in want_files_11.items():
        got_11 = config_11.load_config(PROJECT / rel_11).to_dict()
        ok_11[rel_11] = got_11 == want_11
        verdict("B7.11", ok_11[rel_11], f"{rel_11} : les réglages du contrat.",
                f"{rel_11} : attendu {want_11}, reçu {got_11}.")
    out_11 = scratch("out")
    r_11 = wbp.cli(PROJECT, "train", "--set", "epochs=0", "--out", out_11 / "a")
    ok_11["cli epochs"] = run_line("B7.11", r_11, r_11.returncode == 2 and any("epochs" in l for l in r_11.stderr_lines("ERROR")),
                                   "code 2 et une ligne ERROR qui nomme epochs, avant toute seconde de calcul.",
                                   "attendu le code 2 et une ligne ERROR qui nomme epochs")
    r_11 = wbp.cli(PROJECT, "train", "--config", "configs/smoke.yaml", "--out", out_11 / "b")
    ok_11["cli smoke"] = run_line("B7.11", r_11, r_11.ok, f"code 0 en {fr(r_11.seconds, 1)} s.", "attendu le code 0")
if ok_11 and all(ok_11.values()):
    print("✅ Ex B7.11 : ta configuration est typée, validée au chargement et surchargeable.")

✅ Ex B7.11 : TrainConfig() et to_dict() : les valeurs par défaut du contrat, en JSON.
✅ Ex B7.11 : la configuration est figée (frozen=True).
✅ Ex B7.11 : un entier accepté pour un float, une liste YAML devient un tuple.
✅ Ex B7.11 : 16 réglages invalides refusés, chacun par une ValueError qui le nomme (types, bornes, clé inconnue, --set sans =).
✅ Ex B7.11 : les valeurs par défaut, puis le fichier, puis les --set.
✅ Ex B7.11 : un fichier avec lr: 1e-3 (une chaîne pour PyYAML) : ValueError levée (lr: expected float, got '1e-3').
✅ Ex B7.11 : un fichier avec une faute de frappe : ValueError levée (unknown settings: ['lerning_rate']).
✅ Ex B7.11 : un fichier avec une liste au lieu d'un dictionnaire : ValueError levée (/tmp/wb_b7_yaml_8up8pqa5/bad.yaml: expected a mapping of settings, got list).
✅ Ex B7.11 : un fichier vide : les valeurs par défaut.
✅ Ex B7.11 : un fichier absent : FileNotFoundError levée ([Errno 2] No such file or directory: '/tmp/wb_b7_yaml_8up8pqa5/absent.yaml').
✅ Ex B

✅ Ex B7.11 : cifarprod train --config configs/smoke.yaml --out b → code 0 en 3,6 s.
✅ Ex B7.11 : ta configuration est typée, validée au chargement et surchargeable.


### Ex B7.12 — Prédire : même graine, puis num_workers = 4, puis GPU — résultats identiques ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que les workers d'un DataLoader et un GPU changent à un entraînement à graine fixée.  
**Prérequis :** Ex B7.11 · ch. 24 (workers) · fiche §207.4 (« Les graines et le déterminisme ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

L'expérience entraîne six fois un petit CNN (`wbp.workers_run`, 512 images fixes, 2 epochs, graine 0) et compare les poids **au bit près** (`torch.equal` sur chaque tenseur). Chaque run fait `torch.manual_seed(0)`, puis lit ses images par un `DataLoader(..., shuffle=True, generator=torch.Generator().manual_seed(0), num_workers=...)`, sans `worker_init_fn` ni `persistent_workers`. Le dataset applique une **augmentation** : dans son `__getitem__`, il retourne l'image (gauche-droite) quand `torch.rand(1) < 0.5`. Avec `num_workers = 0`, `__getitem__` tourne dans le processus du notebook ; avec `num_workers = 4`, dans quatre processus fils (chaque epoch en lance de nouveaux). Avec 4 workers sur une machine à 2 cœurs, PyTorch avertit qu'il en propose au plus 2 : l'avertissement est tu, il ne change rien aux nombres. **Avant d'exécuter**, écris tes quatre prédictions (`True` ou `False`) :
- a) `replay0_12` : deux runs avec `num_workers = 0` donnent-ils exactement les mêmes poids ?
- b) `replay4_12` : deux runs avec `num_workers = 4` donnent-ils exactement les mêmes poids ?
- c) `zero_four_12` : le run à `num_workers = 0` et le run à `num_workers = 4` donnent-ils exactement les mêmes poids ?
- d) `plain_12` : **sans** l'augmentation (aucun tirage dans `__getitem__`), `num_workers = 0` et `num_workers = 4` donnent-ils exactement les mêmes poids ?

Et, dans ta cellule 📝 seulement (aucune vérification) : e) sur un GPU, avec la même graine et `num_workers = 0`, les poids seraient-ils ceux du CPU ? Ceux d'un second run sur le même GPU ? La cellule d'expérience le mesure si un GPU est là (sur Colab), et le dit sinon.

La cellule d'expérience n'affiche rien tant que tes quatre prédictions ne sont pas écrites ; elle les vérifie à la fin. Rien n'y dépend de ton paquet.

In [26]:
replay0_12, replay4_12, zero_four_12, plain_12 = True, True, False, True

In [27]:
wb.record("B7.12a", True, mistakes={"relis le tableau : num_workers 0, essai 1 contre essai 2": False}, prediction=True)
wb.record("B7.12b", True, mistakes={"relis le tableau : num_workers 4, essai 1 contre essai 2": False}, prediction=True)
wb.record("B7.12c", False, mistakes={"relis le tableau : num_workers 0 contre 4, avec l'augmentation": True}, prediction=True)
wb.record("B7.12d", True, mistakes={"relis le tableau : num_workers 0 contre 4, sans l'augmentation": False}, prediction=True)

WB_ANSWER {"hash": "f16bcf6d70697ff594f9b6580ff41366f50a145f01c2acf964577cc02516abe5", "id": "B7.12a", "kind": "bool", "mistakes": {"b14b5132eb7a5e5efb37d8a26d462124c1e124e21110ab184eef34cbf94f899d": "relis le tableau : num_workers 0, essai 1 contre essai 2"}, "prediction": true}
📝 Ex B7.12a : réponse enregistrée (bool).
WB_ANSWER {"hash": "29da372ac53649a14b2e94ebfe99536627fb5b3a1a65161d16509725b8eb3765", "id": "B7.12b", "kind": "bool", "mistakes": {"93ec2b5bc5cb6a82a041e745156737a9fc2e4dcd9a400aa7703eeb2c79bf1c1b": "relis le tableau : num_workers 4, essai 1 contre essai 2"}, "prediction": true}
📝 Ex B7.12b : réponse enregistrée (bool).
WB_ANSWER {"hash": "eaba67d85f7c5be6a54d7edee45ba9f96738aa2a661958397bdd270b5869cfcb", "id": "B7.12c", "kind": "bool", "mistakes": {"9a09612a4cce5888e907970e91d78afd0f0d1e3b995e45dcc2d871251769543d": "relis le tableau : num_workers 0 contre 4, avec l'augmentation"}, "prediction": true}
📝 Ex B7.12c : réponse enregistrée (bool).
WB_ANSWER {"hash": "149b4

In [28]:
if any(answer is ... or answer is None for answer in [replay0_12, replay4_12, zero_four_12, plain_12]):
    print("⏳ Ex B7.12 : écris d'abord tes quatre prédictions dans la cellule précédente.")
else:
    start_12 = time.perf_counter()
    RUNS_12 = {"0, essai 1": wbp.workers_run(0, seed=0), "0, essai 2": wbp.workers_run(0, seed=0),
               "4, essai 1": wbp.workers_run(4, seed=0), "4, essai 2": wbp.workers_run(4, seed=0),
               "0, sans retournement": wbp.workers_run(0, seed=0, augment=False),
               "4, sans retournement": wbp.workers_run(4, seed=0, augment=False)}
    PAIRS_12 = {"a": ("0, essai 1", "0, essai 2"), "b": ("4, essai 1", "4, essai 2"), "c": ("0, essai 1", "4, essai 1"),
                "d": ("0, sans retournement", "4, sans retournement")}
    print(f"Six entraînements courts en {fr(time.perf_counter() - start_12, 1)} s (512 images, 2 epochs, graine 0) :")
    for letter_12, (first_12, second_12) in PAIRS_12.items():
        a_12, b_12 = RUNS_12[first_12], RUNS_12[second_12]
        print(f"  {letter_12}) num_workers {first_12} contre {second_12} : "
              + ("mêmes poids au bit près" if wbp.same_weights(a_12, b_12) else f"poids différents (écart maximal "
                 f"{wbp.max_gap(a_12, b_12):.2e})"))
    if torch.cuda.is_available():
        gpu_12 = [wbp.workers_run(0, seed=0, device="cuda") for _ in range(2)]
        print(f"  e) sur le GPU ({torch.cuda.get_device_name(0)}) : deux runs "
              + ("identiques" if wbp.same_weights(*gpu_12) else f"différents ({wbp.max_gap(*gpu_12):.2e})")
              + " ; contre le CPU : " + ("identiques" if wbp.same_weights(gpu_12[0], RUNS_12["0, essai 1"])
                                         else f"différents ({wbp.max_gap(gpu_12[0], RUNS_12['0, essai 1']):.2e})"))
    else:
        print("  e) pas de GPU ici : ta prédiction e) se compare à 05_solutions.md (ou relance cette cellule sur un GPU "
              "de Colab).")
    wb.check("B7.12a", replay0_12)
    wb.check("B7.12b", replay4_12)
    wb.check("B7.12c", zero_four_12)
    wb.check("B7.12d", plain_12)

Six entraînements courts en 3,0 s (512 images, 2 epochs, graine 0) :
  a) num_workers 0, essai 1 contre 0, essai 2 : mêmes poids au bit près
  b) num_workers 4, essai 1 contre 4, essai 2 : mêmes poids au bit près
  c) num_workers 0, essai 1 contre 4, essai 1 : poids différents (écart maximal 2.72e-02)
  d) num_workers 0, sans retournement contre 4, sans retournement : mêmes poids au bit près
  e) pas de GPU ici : ta prédiction e) se compare à 05_solutions.md (ou relance cette cellule sur un GPU de Colab).
✅ Ex B7.12a : C'est ça !
✅ Ex B7.12b : Parfait !
✅ Ex B7.12c : Bravo !
✅ Ex B7.12d : Exact !


Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? D'où chaque worker tient-il son générateur, et pourquoi la graine du `DataLoader` suffit-elle à le rejouer ? Que faudrait-il pour qu'un run à 0 worker et un run à 4 workers tirent les mêmes retournements ? Que ferait `worker_init_fn` (207.4) pour les tirages de NumPy et de `random` dans un worker ? Pourquoi ces résultats ne disent-ils rien d'un GPU ?

### Ex B7.13 — Mesurer le non-déterminisme : graines fixées ou non, algorithmes déterministes, cinq exécutions 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer, sur ton propre entraînement, ce que changent la graine et les algorithmes déterministes, et rapporter la dispersion d'un score.  
**Prérequis :** Ex B7.12 · ✏️ B7.2 · fiche §207.4 (« Les graines et le déterminisme », « Rapporter un score ») · **Fil rouge :** CIFAR-10 · **Parcours :** C

Tu lances **ta** commande cinq fois et tu compares les poids qu'elle écrit. **Contrat** des trois fonctions de la cellule suivante :
- `train_run_13(seed, deterministic=True)` lance une fois, par `wbp.cli(PROJECT, ...)`, la commande `train --config configs/fast.yaml --set seed=<seed> --set deterministic=<true|false> --out <un nouveau dossier temporaire>` ; elle rend `{"weights": le dictionnaire de tenseurs du fichier de poids écrit, "val_acc": la val_acc de la ligne JSON, "seconds": la durée de la commande}`, et lève une `RuntimeError` avec la fin de stderr si le code de sortie n'est pas 0 ;
- `same_weights_13(a, b)` rend `True` quand les deux dictionnaires ont les mêmes noms et exactement les mêmes tenseurs, `False` sinon (un booléen) ;
- `max_gap_13(a, b)` rend le plus grand écart absolu entre les tenseurs de même nom, un float Python.

La cellule de vérification essaie `same_weights_13` et `max_gap_13` sur trois petits cas (des copies, un poids qui diffère de $10^{-6}$, un tenseur en moins). L'expérience lance ensuite cinq runs de 2 000 images (≈ 4 s chacun) : A et B, graine 0 ; C, graine 0 sans les algorithmes déterministes ; D et E, graines 1 et 2. Elle affiche les écarts A-B, A-C, A-D et D-E, l'accuracy des graines 0, 1 et 2 avec son intervalle à 95 % (la formule de ✏️ B7.2, $t_{0{,}975;\,2} = 4{,}303$, donné par le tableau de 207.4), et vérifie par `wb.check` si A et B ont exactement les mêmes poids (rien à taper : la valeur vient de ton code).

Dans tes notes : que donne A contre C sur ce processeur, et pourquoi ce résultat ne vaut-il que pour **ce** matériel et **ce** logiciel (207.4) ? De quel ordre est l'écart d'une graine à l'autre, et l'écart d'accuracy ? Avec trois graines, que vaut l'intervalle, et que dirais-tu d'un concurrent annoncé 1 point au-dessus sur une seule graine ? Qu'est-ce qui, dans ton paquet, rend A et B identiques : liste les générateurs que la graine fixe.

In [29]:
from safetensors.torch import load_file


def train_run_13(seed, deterministic=True):
    """Run YOUR command line once in PROJECT: python -m cifarprod train --config configs/fast.yaml --set seed=<seed>
    --set deterministic=<true|false> --out <a new temporary folder>, through wbp.cli. Returns {"weights": the dict of
    tensors of the model.safetensors it wrote, "val_acc": the val_acc of its JSON line, "seconds": its duration};
    RuntimeError with the end of stderr if the command does not end with the code 0."""
    out = tempfile.mkdtemp(prefix="wb_b7_13_")
    result = wbp.cli(PROJECT, "train", "--config", "configs/fast.yaml", "--set", f"seed={seed}",
                     "--set", f"deterministic={str(deterministic).lower()}", "--out", out)
    if not result.ok:
        raise RuntimeError(f"cifarprod train exited with {result.returncode}: {result.tail(3)}")
    summary = result.last_json()
    return {"weights": load_file(summary["weights"]), "val_acc": summary["val_acc"], "seconds": result.seconds}


def same_weights_13(a, b):
    """True when the two dicts of tensors have the same names and exactly the same values (torch.equal), else False."""
    return a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a)


def max_gap_13(a, b):
    """The largest absolute difference between the tensors of the same name of two dicts, as a Python float."""
    return max(float((a[k].double() - b[k].double()).abs().max()) for k in a)


tools_ok_13 = False
with checked("B7.13"):
    base_13 = {"w": torch.zeros(2, 3), "b": torch.ones(4), "n": torch.tensor(7)}
    twin_13 = {k: v.clone() for k, v in base_13.items()}
    tiny_13 = {**twin_13, "w": torch.tensor([[0.0, 0.0, 1e-6], [0.0, 0.0, 0.0]])}
    cases_13 = [("deux copies identiques", base_13, twin_13, True), ("un poids qui diffère de 1e-6", base_13, tiny_13, False),
                ("un tenseur en moins", base_13, {k: v for k, v in twin_13.items() if k != "b"}, False)]
    all_13 = True
    for label_13, a_13, b_13, want_13 in cases_13:
        got_13 = same_weights_13(a_13, b_13)
        good_13 = isinstance(got_13, (bool, np.bool_)) and bool(got_13) == want_13
        all_13 &= good_13
        verdict("B7.13", good_13, f"same_weights_13 sur {label_13} : {want_13}.",
                f"same_weights_13 sur {label_13} : attendu {want_13}, reçu {got_13!r}.")
    gap_13 = max_gap_13(base_13, tiny_13)
    ok_gap_13 = type(gap_13) is float and abs(gap_13 - float(torch.tensor(1e-6))) < 1e-12
    verdict("B7.13", ok_gap_13, "max_gap_13 : 1e-6 (un float Python).",
            f"max_gap_13 sur un poids qui diffère de 1e-6 : reçu {gap_13!r}, attendu le float 1e-6 (à l'arrondi du float32 près).")
    tools_ok_13 = bool(all_13 and ok_gap_13)
probe_13 = wbp.cli(PROJECT, "--version")
if tools_ok_13 and not probe_13.ok:
    tools_ok_13 = False
    print(f"⏳ Ex B7.13 : ta commande ne répond pas encore (cifarprod --version : code {probe_13.returncode}, "
          f"{probe_13.tail(1)[:160]}) : fais d'abord passer B7.10 et B7.11.")

✅ Ex B7.13 : same_weights_13 sur deux copies identiques : True.
✅ Ex B7.13 : same_weights_13 sur un poids qui diffère de 1e-6 : False.
✅ Ex B7.13 : same_weights_13 sur un tenseur en moins : False.
✅ Ex B7.13 : max_gap_13 : 1e-6 (un float Python).


In [30]:
wb.record("B7.13", True, mistakes={"tes deux runs à même graine n'ont pas exactement les mêmes poids : qu'est-ce qui, "
                                   "dans ton entraînement, ne dépend pas de la graine de la configuration ?": False})

WB_ANSWER {"hash": "9ee1286ad49598e6d89727bdb3553a017c38793bb5ad5a446b419f821629750c", "id": "B7.13", "kind": "bool", "mistakes": {"9ddf73ff008fff2f5f8e441f451dea4b026faac62622283265fbab0df9fb66c4": "tes deux runs à même graine n'ont pas exactement les mêmes poids : qu'est-ce qui, dans ton entraînement, ne dépend pas de la graine de la configuration ?"}}
📝 Ex B7.13 : réponse enregistrée (bool).


In [31]:
RUNS_13 = {}
if not tools_ok_13:
    print("⏳ Ex B7.13 : l'expérience attend same_weights_13, max_gap_13 et train_run_13 justes (cellule précédente).")
else:
    with checked("B7.13"):
        PLAN_13 = {"A": (0, True), "B": (0, True), "C": (0, False), "D": (1, True), "E": (2, True)}
        for key_13, (seed_13, det_13) in PLAN_13.items():
            RUNS_13[key_13] = train_run_13(seed_13, det_13)
            print(f"run {key_13} : graine {seed_13}, algorithmes déterministes {'oui' if det_13 else 'non'} → val_acc "
                  f"{fr(RUNS_13[key_13]['val_acc'], 3)} en {fr(RUNS_13[key_13]['seconds'], 1)} s")
        for first_13, second_13, what_13 in [("A", "B", "même graine, deux fois"), ("A", "C", "déterministe ou non"),
                                             ("A", "D", "graine 0 contre graine 1"), ("D", "E", "graine 1 contre graine 2")]:
            wa_13, wb_13 = RUNS_13[first_13]["weights"], RUNS_13[second_13]["weights"]
            print(f"  {first_13} contre {second_13} ({what_13}) : " + ("poids identiques au bit près"
                  if same_weights_13(wa_13, wb_13) else f"écart maximal {max_gap_13(wa_13, wb_13):.3g}"))
        accs_13 = np.array([RUNS_13[k]["val_acc"] for k in ("A", "D", "E")])
        half_13 = 4.303 * accs_13.std(ddof=1) / np.sqrt(3)                # t(0.975; 2), the table of 207.4
        print(f"accuracy sur 3 graines : {fr(100 * accs_13.mean(), 1)} % ± {fr(100 * half_13, 1)} (intervalle à 95 %, "
              f"t = 4,303 pour n = 3 ; valeurs {', '.join(fr(100 * a, 1) for a in accs_13)})")
        wb.check("B7.13", same_weights_13(RUNS_13["A"]["weights"], RUNS_13["B"]["weights"]))

run A : graine 0, algorithmes déterministes oui → val_acc 0,302 en 3,7 s


run B : graine 0, algorithmes déterministes oui → val_acc 0,302 en 3,4 s


run C : graine 0, algorithmes déterministes non → val_acc 0,302 en 3,7 s


run D : graine 1, algorithmes déterministes oui → val_acc 0,332 en 4,1 s


run E : graine 2, algorithmes déterministes oui → val_acc 0,326 en 4,3 s
  A contre B (même graine, deux fois) : poids identiques au bit près
  A contre C (déterministe ou non) : poids identiques au bit près
  A contre D (graine 0 contre graine 1) : écart maximal 0.402
  D contre E (graine 1 contre graine 2) : écart maximal 0.403
accuracy sur 3 graines : 32,0 % ± 3,9 (intervalle à 95 %, t = 4,303 pour n = 3 ; valeurs 30,2, 33,2, 32,6)
✅ Ex B7.13 : Bien joué !


### Ex B7.14 — Suivi d'expériences maison : identifiant de run, configuration, métriques et artefacts en JSONL 📦 ★★ ⏱️ 30 min
**Objectif :** faire laisser à chaque run une ligne qui dit ce qui le définit, ce qu'il mesure et ce qu'il produit.  
**Prérequis :** Ex B7.11 · ch. 24 (journal d'expériences) · 🔁 R1 · fiche §207.4 (versions, empreinte), §207.5 · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

> **Mode d'emploi du paquet** : chaque fichier de ton paquet est une chaîne de la cellule à compléter (`DATA_PY = r'''...'''`) ; la cellule de vérification l'**écrit** dans ton projet (`mon_travail/b7_production/cifar-prod/`, à côté de ce notebook), puis l'importe ou le lance dans un autre processus. Pour corriger un fichier, corrige sa chaîne et réexécute les deux cellules. Si tu préfères un éditeur, modifie le fichier sur le disque : la cellule le voit et ne l'écrase plus (elle le dit).

**Le contrat.** Un nouveau fichier, `tracking.py` (la cellule suivante : exécute-la d'abord), et `cli.py` à **compléter dans la cellule de B7.10** ; réexécute ensuite la cellule de vérification de B7.10 (qui écrit aussi `tracking.py`, et vérifie que rien n'a cassé), puis celle-ci :
- `tracking.py` : `new_run_id()` (la date et l'heure UTC puis six caractères hexadécimaux au hasard : `20261104-161731-e06339`, comme 207.5) ; `file_sha256(path)` ; `artifact(path)` rend `{"path": str(path), "bytes": sa taille, "sha256": file_sha256(path)}` ; `environment()` rend `{"python", "numpy", "torch", "platform", "commit", "dirty"}` (`platform.python_version()`, `importlib.metadata.version(...)`, `platform.platform()`, le commit court `git rev-parse --short HEAD` et si `git status --porcelain` n'est pas vide, `None` hors d'un dépôt git) ; `log_run(path, record)` ajoute **une** ligne JSON (clés triées) à la fin du journal, dont il crée le dossier ; `read_runs(path)` rend la liste des runs, du plus ancien au plus récent (`[]` sans fichier) ;
- `cli.py` : une option de plus, `--runs FILE` (défaut `runs/runs.jsonl`) ; sans `--out`, les poids vont dans `runs/<run_id>/` ; **chaque** run qui a commencé (après une configuration valide) laisse une ligne, réussi ou non (un `finally`), avec les champs `run_id`, `started` (la date ISO 8601 UTC), `status` (`"ok"` ou `"failed"`), `seconds`, `config` (`to_dict()`), `environment` ; un run réussi ajoute `data` (`{"path", "n_train", "sha256"}` : l'empreinte `fingerprint(X, y)` des images **et** des labels d'entraînement utilisés), `metrics` (les trois courbes de `history` et `val_acc_final`), `normalization` (`{"mean", "std"}`) et `artifacts` (la liste des `artifact(...)` des fichiers écrits : ici, les poids) ; un run raté garde son `error` (le type et le message de l'exception) ; le résumé JSON de stdout ajoute le `run_id`.

La cellule de vérification essaie `tracking.py` dans le notebook (trois identifiants, l'empreinte d'un fichier de 300 000 octets contre `hashlib`, l'environnement, deux lignes ajoutées puis relues), puis lance trois runs dans un même journal : 200 images, puis `n_train=99999` (plus que les données : code 1), puis 200 images avec la graine 1 sans `--out`. Elle relit le journal (trois lignes, la première jamais réécrite) et compare chaque champ à ce qu'elle sait : la configuration demandée, l'empreinte des 200 premières images et de leurs labels, le fichier de poids (taille et SHA-256), la val_acc de stdout, l'erreur du run raté ; elle affiche enfin le journal par `pandas.json_normalize`.

Dans tes notes : six mois plus tard, quelle ligne du journal te dit avec quelles données, quel code et quels réglages le modèle livré a été entraîné ? Pourquoi garder les runs ratés ? Pourquoi l'empreinte porte-t-elle aussi sur les labels ? Que manque-t-il encore pour rejouer un run sur une autre machine (207.4) ?

In [32]:
# Ex B7.14: tracking.py (the solutions notebook has written it with cli.py in Ex B7.10: the same text again)

# src/cifarprod/tracking.py
TRACKING_PY = r'''"""The run journal of cifarprod: one JSON line per run, appended to a JSONL file, never rewritten.

    run_id = new_run_id()                              # "20261104-161731-e06339"
    log_run(Path("runs/runs.jsonl"), {"run_id": run_id, "status": "ok", ...})
    runs = read_runs(Path("runs/runs.jsonl"))          # a list of dictionaries, oldest first

A run records what defines it (configuration, environment, data fingerprint), what it measures (metrics per epoch,
duration) and what it produces (its artifacts, with their path, size and fingerprint).
"""

from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import uuid
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path


def new_run_id() -> str:
    """A unique run identifier: the UTC date and time, then six random hexadecimal characters."""
    return f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:6]}"


def file_sha256(path: str | Path) -> str:
    """The SHA-256 of the bytes of a file (64 hexadecimal characters)."""
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def artifact(path: str | Path) -> dict:
    """{"path", "bytes", "sha256"} of a file written by the run."""
    path = Path(path)
    return {"path": str(path), "bytes": path.stat().st_size, "sha256": file_sha256(path)}


def _git(*args: str, cwd=None) -> str | None:
    try:
        done = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, timeout=10)
    except (OSError, subprocess.SubprocessError):
        return None
    return done.stdout.strip() if done.returncode == 0 else None


def environment(cwd=None) -> dict:
    """What runs the code: the versions of Python, NumPy and PyTorch, the platform, the git commit (None outside a
    repository) and whether the working tree has uncommitted changes."""
    versions = {}
    for name in ("numpy", "torch"):
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    status = _git("status", "--porcelain", cwd=cwd)
    return {"python": platform.python_version(), **versions, "platform": platform.platform(),
            "commit": _git("rev-parse", "--short", "HEAD", cwd=cwd), "dirty": bool(status) if status is not None else None}


def log_run(path: str | Path, record: dict) -> None:
    """Append one JSON line (keys sorted) to the journal ``path``; its folder is created if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, sort_keys=True) + "\n")


def read_runs(path: str | Path) -> list[dict]:
    """The runs of the journal, oldest first ([] if the file does not exist)."""
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
'''


import hashlib
import platform
from importlib import metadata

import pandas as pd

ok_14 = {}
with checked("B7.14"):
    wbp.write_files(PROJECT, {"src/cifarprod/tracking.py": TRACKING_PY, "src/cifarprod/cli.py": CLI_PY})
    tracking_14 = wbp.import_module(PROJECT, "tracking")
    ids_14 = [tracking_14.new_run_id() for _ in range(3)]
    today_14 = time.strftime("%Y%m%d", time.gmtime())
    ok_14["id"] = (all(isinstance(i, str) and re.fullmatch(r"\d{8}-\d{6}-[0-9a-f]{6}", i) for i in ids_14)
                   and len(set(ids_14)) == 3 and ids_14[0][:8] == today_14)
    verdict("B7.14", ok_14["id"], f"new_run_id : {ids_14[0]}, {ids_14[1]}… (date UTC, heure, six caractères au hasard).",
            f"new_run_id : attendu trois identifiants distincts de la forme {today_14}-HHMMSS-xxxxxx (date UTC), "
            f"reçu {ids_14!r:.120}.")
    blob_14 = scratch("blob") / "weights.bin"
    blob_14.write_bytes(np.random.default_rng(0).bytes(300_000))
    want_art_14 = {"path": str(blob_14), "bytes": 300_000, "sha256": hashlib.sha256(blob_14.read_bytes()).hexdigest()}
    ok_14["artifact"] = tracking_14.file_sha256(blob_14) == want_art_14["sha256"] and tracking_14.artifact(blob_14) == want_art_14
    verdict("B7.14", ok_14["artifact"], "file_sha256 et artifact d'un fichier de 300 000 octets : comme hashlib.",
            f"artifact(fichier de 300 000 octets) : attendu {{'path', 'bytes', 'sha256'}} = {want_art_14['bytes']} octets "
            f"et le SHA-256 de hashlib, reçu {tracking_14.artifact(blob_14)!r:.200}.")
    env_14 = tracking_14.environment()
    ok_14["environment"] = (isinstance(env_14, dict) and {"python", "numpy", "torch", "platform", "commit", "dirty"} <= set(env_14)
                            and env_14["python"] == platform.python_version() and env_14["torch"] == metadata.version("torch"))
    verdict("B7.14", ok_14["environment"], f"environment : Python {env_14.get('python')}, torch {env_14.get('torch')}, "
            f"commit {env_14.get('commit')!r}.", "environment doit rendre {python, numpy, torch, platform, commit, dirty} "
            f"avec platform.python_version() et importlib.metadata.version ; reçu {env_14!r:.200}.")
    journal_14 = scratch("journal") / "sub" / "runs.jsonl"
    tracking_14.log_run(journal_14, {"run_id": "a", "config": {"lr": 0.1}})
    first_14 = journal_14.read_text(encoding="utf-8")
    tracking_14.log_run(journal_14, {"run_id": "b", "metrics": [1, 2]})
    lines_14 = journal_14.read_text(encoding="utf-8").splitlines()
    ok_14["append"] = (len(lines_14) == 2 and journal_14.read_text(encoding="utf-8").startswith(first_14)
                       and [json.loads(l) for l in lines_14] == [{"run_id": "a", "config": {"lr": 0.1}}, {"run_id": "b", "metrics": [1, 2]}]
                       and tracking_14.read_runs(journal_14) == [json.loads(l) for l in lines_14]
                       and tracking_14.read_runs(journal_14.with_name("absent.jsonl")) == [])
    verdict("B7.14", ok_14["append"], "log_run ajoute une ligne JSON par run (dossier créé), read_runs les relit ([] sans fichier).",
            f"log_run puis read_runs : attendu deux lignes JSON, la première inchangée, relues dans l'ordre, et [] pour "
            f"un journal absent ; le fichier contient {lines_14!r:.200}.")

ready_14 = (written(lambda: wbp.import_module(PROJECT, "tracking").new_run_id())
            and written(lambda: wbp.import_module(PROJECT, "cli").build_parser())
            and written(lambda: wbp.import_module(PROJECT, "config").TrainConfig()))
if not ready_14:
    print("⏳ Ex B7.14 : le journal des runs attend tracking.py, et ta commande et ta configuration (B7.10, B7.11).")
else:
    with checked("B7.14"):                                 # the command line logs every run
        out_14 = scratch("runs")
        journal_14 = out_14 / "runs.jsonl"
        r1_14 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--out", out_14 / "a", "--runs", journal_14)
        ok_14["run 1"] = run_line("B7.14", r1_14, r1_14.ok, "code 0.", "attendu le code 0")
        first_lines_14 = journal_14.read_text(encoding="utf-8").splitlines() if journal_14.is_file() else []
        first_line_14 = first_lines_14[0] if first_lines_14 else None
        r2_14 = wbp.cli(PROJECT, "train", "--set", "n_train=99999", "--runs", journal_14)
        ok_14["run 2"] = run_line("B7.14", r2_14, r2_14.returncode == 1, "code 1 (n_train plus grand que les données).",
                                  "attendu le code 1 : n_train = 99 999 dépasse les 2 000 images")
        r3_14 = wbp.cli(PROJECT, "train", "--set", "n_train=200", "--set", "seed=1", "--runs", journal_14)
        ok_14["run 3"] = run_line("B7.14", r3_14, r3_14.ok, "code 0, sans --out.", "attendu le code 0")
        lines_14 = journal_14.read_text(encoding="utf-8").splitlines() if journal_14.is_file() else []
        runs_14 = []
        for line_14 in lines_14:
            try:
                record_14 = json.loads(line_14)
            except ValueError:
                record_14 = None
            runs_14.append(record_14 if isinstance(record_14, dict) else {"status": f"ligne illisible : {line_14[:60]!r}"})
        ok_14["three"] = [r.get("status") for r in runs_14] == ["ok", "failed", "ok"] and lines_14[0] == first_line_14
        verdict("B7.14", ok_14["three"], "trois lignes dans le journal, statuts ok, failed, ok ; la première jamais réécrite.",
                f"attendu trois lignes de statuts ['ok', 'failed', 'ok'], la première inchangée ; reçu "
                f"{[r.get('status') for r in runs_14]}.")
        if not ok_14["three"]:
            raise StopCheck
        keys_14 = {"run_id", "started", "status", "seconds", "config", "environment", "data", "metrics", "normalization",
                   "artifacts"}
        ok_rec_14 = all(keys_14 <= set(r) for r in (runs_14[0], runs_14[2])) and {"run_id", "status", "seconds", "config",
                                                                                   "error"} <= set(runs_14[1])
        verdict("B7.14", ok_rec_14, "chaque ligne a les champs du contrat (un run raté : son erreur).",
                f"champs manquants : run réussi {sorted(keys_14 - set(runs_14[0]))}, run raté "
                f"{sorted({'run_id', 'status', 'seconds', 'config', 'error'} - set(runs_14[1]))}.")
        if not ok_rec_14:
            raise StopCheck
        rec_14 = runs_14[0]
        ok_14["ids"] = len({r["run_id"] for r in runs_14}) == 3 and r1_14.last_json().get("run_id") == rec_14["run_id"] \
            and r3_14.last_json().get("run_id") == runs_14[2]["run_id"]
        verdict("B7.14", ok_14["ids"], "trois identifiants distincts, ceux des lignes JSON de stdout.",
                "attendu trois run_id distincts, et sur stdout le run_id de la ligne du journal.")
        ok_14["config"] = rec_14["config"] == {**DEFAULTS, "n_train": 200} and runs_14[2]["config"] == {**DEFAULTS, "n_train": 200, "seed": 1}
        verdict("B7.14", ok_14["config"], "la configuration complète du run (to_dict), surcharges comprises.",
                f"config du premier run : attendu {{**DEFAULTS, 'n_train': 200}}, reçu {rec_14['config']!r:.200}.")
        want_sha_14 = wbp.array_fingerprint(SUBSET["x_train"][:200], SUBSET["y_train"][:200])
        ok_14["data"] = rec_14["data"].get("sha256") == want_sha_14 and rec_14["data"].get("n_train") == 200
        verdict("B7.14", ok_14["data"], f"l'empreinte des 200 images d'entraînement et de leurs labels : {want_sha_14[:12]}….",
                f"data : attendu n_train 200 et sha256 = fingerprint(X[:200], y[:200]) des images ET des labels utilisés, "
                f"reçu {rec_14['data']!r:.200}.")
        arts_14 = rec_14["artifacts"]
        art_path_14 = (PROJECT / arts_14[0]["path"]).resolve() if arts_14 else None
        ok_14["artifacts"] = (len(arts_14) >= 1 and art_path_14 == (out_14 / "a" / "model.safetensors").resolve()
                              and arts_14[0].get("sha256") == hashlib.sha256(art_path_14.read_bytes()).hexdigest()
                              and arts_14[0].get("bytes") == art_path_14.stat().st_size)
        verdict("B7.14", ok_14["artifacts"], "artifacts : le fichier des poids, sa taille et son SHA-256.",
                f"artifacts : attendu [{{path, bytes, sha256}}] du fichier <out>/model.safetensors, reçu {arts_14!r:.200}.")
        default_out_14 = (PROJECT / runs_14[2]["artifacts"][0]["path"]).resolve() if runs_14[2]["artifacts"] else None
        ok_14["default out"] = default_out_14 == (PROJECT / "runs" / runs_14[2]["run_id"] / "model.safetensors").resolve()
        verdict("B7.14", ok_14["default out"], "sans --out, les poids vont dans runs/<run_id>/.",
                f"sans --out, attendu runs/<run_id>/model.safetensors, reçu {default_out_14}.")
        metrics_14 = rec_14["metrics"]
        ok_14["metrics"] = (len(metrics_14.get("val_acc", [])) == 1 and metrics_14.get("val_acc_final") == r1_14.last_json().get("val_acc")
                            and np.allclose(rec_14["normalization"]["mean"], (SUBSET["x_train"][:200] / 255).mean(axis=(0, 1, 2))))
        verdict("B7.14", ok_14["metrics"], "metrics (une valeur par epoch, val_acc_final) et la normalisation apprise.",
                "metrics : attendu train_loss, val_loss, val_acc (une valeur par epoch) et val_acc_final, la val_acc de "
                "stdout ; normalization : la moyenne et l'écart-type des images d'entraînement.")
        ok_14["failed"] = "n_train" in runs_14[1].get("error", "")
        verdict("B7.14", ok_14["failed"], f"le run raté garde son erreur : « {runs_14[1].get('error', '')[:70]} ».",
                f"le run raté doit garder son erreur (qui nomme n_train) ; reçu {runs_14[1].get('error')!r}.")
        TABLE_14 = pd.json_normalize(runs_14)[["run_id", "status", "config.seed", "config.n_train", "metrics.val_acc_final",
                                               "seconds"]]
        print(TABLE_14.to_string(index=False))
if ok_14 and all(ok_14.values()) and len(ok_14) >= 12:
    print("✅ Ex B7.14 : chaque run laisse sa ligne : ce qui le définit, ce qu'il mesure, ce qu'il produit.")

✅ Ex B7.14 : new_run_id : 20261104-201911-17c059, 20261104-201911-e390b4… (date UTC, heure, six caractères au hasard).
✅ Ex B7.14 : file_sha256 et artifact d'un fichier de 300 000 octets : comme hashlib.
✅ Ex B7.14 : environment : Python 3.13.18, torch 2.11.0+cpu, commit '5a87002'.
✅ Ex B7.14 : log_run ajoute une ligne JSON par run (dossier créé), read_runs les relit ([] sans fichier).


✅ Ex B7.14 : cifarprod train --set n_train=200 --out a --runs runs.jsonl → code 0.


✅ Ex B7.14 : cifarprod train --set n_train=99999 --runs runs.jsonl → code 1 (n_train plus grand que les données).


✅ Ex B7.14 : cifarprod train --set n_train=200 --set seed=1 --runs runs.jsonl → code 0, sans --out.
✅ Ex B7.14 : trois lignes dans le journal, statuts ok, failed, ok ; la première jamais réécrite.
✅ Ex B7.14 : chaque ligne a les champs du contrat (un run raté : son erreur).
✅ Ex B7.14 : trois identifiants distincts, ceux des lignes JSON de stdout.
✅ Ex B7.14 : la configuration complète du run (to_dict), surcharges comprises.
✅ Ex B7.14 : l'empreinte des 200 images d'entraînement et de leurs labels : 52c0dc2f0fb4….
✅ Ex B7.14 : artifacts : le fichier des poids, sa taille et son SHA-256.
✅ Ex B7.14 : sans --out, les poids vont dans runs/<run_id>/.
✅ Ex B7.14 : metrics (une valeur par epoch, val_acc_final) et la normalisation apprise.
✅ Ex B7.14 : le run raté garde son erreur : « ValueError: n_train = 99999, but data/cifar10_2000.npz has 2000 traini ».
                run_id status  config.seed  config.n_train  metrics.val_acc_final  seconds
20261104-201911-969fd0     ok            0     

### Ex B7.15 — Tests pytest pour le ML : formes, overfitting d'un mini-batch, prétraitement, schéma des données 📦 ★★★ ⏱️ 45 min
**Objectif :** écrire les tests qui protègent un projet de ML, et vérifier qu'ils attrapent de vraies erreurs.  
**Prérequis :** Ex B7.11 (et B7.14) · ch. 21 (tests paramétrés, 21.16) · fiche §207.6 (« Quatre familles de tests ») · **Fil rouge :** CIFAR-10 · **Parcours :** R, C

La fiche donne en entier deux tests du modèle et sa fixture : ils sont dans `TEST_MODEL_PY`, avec les fixtures de `CONFTEST_PY` (`model`, le CNN de la fiche ; `data_file`, le chemin de `data/cifar10_2000.npz`, qui fait sauter le test sans données, comme dans l'intégration continue ; `synthetic_file`, un petit `.npz` au même schéma, 96 images d'entraînement et 32 de validation). **À toi** `tests/test_data.py` et `tests/test_cli.py`, avec **au moins 8 tests** (les cas d'un `parametrize` comptent chacun) dans ces deux fichiers, qui couvrent :
- le **prétraitement** d'une image constante (tout à 0, tout à 255 : la valeur attendue de chaque canal se calcule à la main) et l'ordre des canaux ;
- le **schéma des données** : types, formes, labels de 0 à 9, chaque classe présente, et le refus d'un fichier mal formé ;
- **aucun recouvrement** entre l'entraînement et la validation, par l'empreinte de chaque image ;
- une **configuration invalide refusée** (au moins `epochs=0`, un learning rate hors de $]0, 1[$, une clé inconnue) ;
- le **code de sortie** de `main(argv)`, appelé dans le test (2 pour une configuration invalide, 1 sans données) ;
- un **entraînement fumée** de bout en bout, marqué `@pytest.mark.slow`, sur `synthetic_file` : code 0, le fichier de poids écrit, et le run dans le journal si tu as fait B7.14.

Chaque `assert` a un message dont la **première ligne dit ce qui était attendu** (`f"expected ..., got ..."`), comme dans la fiche : quand l'intégration continue échoue, c'est tout ce que tu lis. Passe `--runs` et `--out` vers `tmp_path` (la fixture de pytest) pour ne rien écrire dans le projet. Si tu reviens après B7.18, ajoute un test de l'API (une réponse 422 pour une image de 28 × 28).

La cellule de vérification écrit tes tests dans `tests/`, les lance par pytest sur ton paquet, fichier par fichier (le nom de chaque test en échec, puis sa raison), et compte tes tests et ceux marqués `slow`. Puis elle les lance sur le **paquet de référence** du chapitre (`solutions/b7_cifarprod`, qui suit le contrat de B7.9 à B7.14 : un test qui y échoue exige autre chose que le contrat) et sur **six versions fautives** de ce paquet, chacune avec une erreur que tes tests doivent voir : le prétraitement qui ne divise plus par 255, `load_npz` qui rend des floats sans vérifier le schéma, une validation qui recouvre l'entraînement, une configuration qui accepte un learning rate hors de $]0, 1[$, `main` qui rend 0 sur une configuration invalide, et une commande `train` qui rend 0 sans écrire ses poids (vue seulement par un test `slow`). Pour chacune, elle dit quel test l'a attrapée, et si la première ligne de son message dit ce qui était attendu.

Dans tes notes : lequel de tes tests attrape le plus de versions fautives, et pourquoi ? Pourquoi un test qui ne fait que « ça ne plante pas » n'en attrape-t-il aucune ? Que verrait l'overfitting d'un mini-batch que les autres ne voient pas (pense à un `zero_grad` oublié) ? Pourquoi l'entraînement fumée est-il marqué `slow`, et quand le lancerais-tu quand même ?

In [33]:
# Ex B7.15: the tests of the reference package

# tests/conftest.py
CONFTEST_PY = r'''"""Fixtures of the tests of cifarprod: a fresh model, the data file of the project, a small synthetic data file."""

from pathlib import Path

import numpy as np
import pytest
import torch

from cifarprod.data import synthetic
from cifarprod.model import build_model

DATA = Path(__file__).resolve().parents[1] / "data" / "cifar10_2000.npz"


@pytest.fixture
def model():
    torch.manual_seed(0)
    return build_model(widths=(12, 24))


@pytest.fixture
def data_file():
    """The CIFAR-10 subset of the project; the tests that need it are skipped without it (in the CI, for example)."""
    if not DATA.is_file():
        pytest.skip(f"no data file {DATA}")
    return DATA


@pytest.fixture
def synthetic_file(tmp_path):
    """A small .npz file with the schema of the data (96 training and 32 validation random images)."""
    X, y = synthetic(96, seed=0)
    X_val, y_val = synthetic(32, seed=1)
    path = tmp_path / "synthetic.npz"
    np.savez(path, x_train=X, y_train=y, x_val=X_val, y_val=y_val)
    return path
'''

# tests/test_model.py
TEST_MODEL_PY = r'''"""Tests of the model: the shape of the output, and a mini-batch learned by heart (fiche, 207.6)."""

import pytest
import torch
import torch.nn.functional as F

from cifarprod.model import build_model


@pytest.mark.parametrize("batch", [1, 7])
def test_output_shape(model, batch):
    out = model(torch.zeros(batch, 3, 32, 32))
    assert out.shape == (batch, 10), f"expected ({batch}, 10), got {tuple(out.shape)}"


def test_overfits_one_batch(model):
    g = torch.Generator().manual_seed(0)
    x, y = torch.randn(16, 3, 32, 32, generator=g), torch.arange(16) % 10
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(100):
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
    assert loss.item() < 0.05, f"expected a loss < 0.05 on 16 memorised images, got {loss.item():.3f}"


@pytest.mark.parametrize("arch", ["cnn", "vgg"])
def test_both_archs_give_ten_logits(arch):
    out = build_model(arch, widths=(8, 16))(torch.zeros(3, 3, 32, 32))
    assert out.shape == (3, 10), f"expected (3, 10) for {arch}, got {tuple(out.shape)}"
'''

# tests/test_data.py
TEST_DATA_PY = r'''"""Tests of the data: the preprocessing on a constant image, the schema of the data file, no image shared by the
training and the validation parts, the fingerprint."""

import numpy as np
import pytest
import torch

from cifarprod.data import channel_stats, fingerprint, load_npz, preprocess

MEAN, STD = (0.5, 0.4, 0.3), (0.25, 0.2, 0.1)


@pytest.mark.parametrize("value", [0, 255])
def test_constant_image_preprocessed(value):
    x = preprocess(np.full((32, 32, 3), value, dtype=np.uint8), MEAN, STD)
    want = torch.tensor([(value / 255 - m) / s for m, s in zip(MEAN, STD)]).view(1, 3, 1, 1).expand(1, 3, 32, 32)
    assert x.shape == (1, 3, 32, 32), f"expected the shape (1, 3, 32, 32), got {tuple(x.shape)}"
    assert x.dtype == torch.float32, f"expected float32, got {x.dtype}"
    assert torch.allclose(x, want, atol=1e-6), f"expected {want[0, :, 0, 0].tolist()} per channel, got {x[0, :, 0, 0].tolist()}"


def test_channels_stay_in_rgb_order():
    image = np.zeros((32, 32, 3), dtype=np.uint8)
    image[..., 0] = 255                                   # a red image
    x = preprocess(image, (0, 0, 0), (1, 1, 1))
    assert x[0, 0].min() == 1 and x[0, 1:].max() == 0, f"expected red in channel 0 only, got the means {x[0].mean(dim=(1, 2)).tolist()}"


@pytest.mark.parametrize("part, n", [("train", 2000), ("val", 500)])
def test_data_schema(data_file, part, n):
    X, y = load_npz(data_file, part)
    assert X.dtype == np.uint8, f"expected uint8 images, got {X.dtype}"
    assert X.shape == (n, 32, 32, 3), f"expected the shape ({n}, 32, 32, 3), got {X.shape}"
    assert y.shape == (n,) and np.issubdtype(y.dtype, np.integer), f"expected {n} integer labels, got {y.shape} {y.dtype}"
    assert set(np.unique(y)) == set(range(10)), f"expected the labels 0 to 9, got {sorted(set(np.unique(y)))}"


def test_no_image_in_both_parts(data_file):
    train = {fingerprint(image) for image in load_npz(data_file, "train")[0]}
    shared = sum(fingerprint(image) in train for image in load_npz(data_file, "val")[0])
    assert shared == 0, f"expected no validation image in the training part, got {shared}"


@pytest.mark.parametrize("bad", ["float", "label", "shape"])
def test_load_npz_refuses_a_bad_schema(tmp_path, bad):
    X, y = np.zeros((4, 32, 32, 3), np.uint8), np.arange(4)
    X = X.astype(np.float32) if bad == "float" else X[:, :28] if bad == "shape" else X
    y = y + 7 if bad == "label" else y
    np.savez(tmp_path / "bad.npz", x_train=X, y_train=y, x_val=X, y_val=y)
    with pytest.raises(ValueError):
        load_npz(tmp_path / "bad.npz", "train")


def test_fingerprint_sees_one_pixel():
    X = np.zeros((2, 32, 32, 3), dtype=np.uint8)
    Y = X.copy()
    Y[1, 5, 5, 2] = 1
    assert fingerprint(X) != fingerprint(Y), "expected another fingerprint when one pixel changes"
    assert fingerprint(X) != fingerprint(X.astype(np.int64)), "expected another fingerprint for another dtype"


def test_channel_stats_on_training_images():
    rng = np.random.default_rng(0)
    X = rng.integers(0, 256, (10, 32, 32, 3), dtype=np.uint8)
    mean, std = channel_stats(X)
    want = (X / 255).mean(axis=(0, 1, 2)), (X / 255).std(axis=(0, 1, 2))
    assert np.allclose(mean, want[0]) and np.allclose(std, want[1]), f"expected {want}, got {(mean, std)}"
'''

# tests/test_cli.py
TEST_CLI_PY = r'''"""Tests of the configuration and of the command line: invalid settings refused, exit codes, a smoke training."""

import json

import pytest
from safetensors.torch import load_file

from cifarprod.cli import main
from cifarprod.config import load_config


@pytest.mark.parametrize("override", ["epochs=0", "lr=0", "lr=5", "lr=1e-3", "lerning_rate=0.01", "widths=[]",
                                      "arch=resnet", "batch_size=true", "seed"])
def test_invalid_config_refused(override):
    with pytest.raises(ValueError):
        load_config(None, [override])


def test_overrides_win_over_the_file(tmp_path):
    path = tmp_path / "c.yaml"
    path.write_text("lr: 0.1\nepochs: 3\nwidths: [4, 8]\n", encoding="utf-8")
    config = load_config(path, ["lr=0.01", "epochs=2"])
    assert (config.lr, config.epochs, config.widths) == (0.01, 2, (4, 8)), f"expected (0.01, 2, (4, 8)), got {(config.lr, config.epochs, config.widths)}"


def test_exit_code_of_an_invalid_config(tmp_path):
    code = main(["train", "--set", "epochs=0", "--runs", str(tmp_path / "runs.jsonl")])
    assert code == 2, f"expected the exit code 2 for epochs=0, got {code}"


def test_exit_code_without_data(tmp_path):
    code = main(["train", "--set", f"data={tmp_path / 'absent.npz'}", "--runs", str(tmp_path / "runs.jsonl")])
    assert code == 1, f"expected the exit code 1 without data file, got {code}"


@pytest.mark.slow
def test_smoke_train(tmp_path, synthetic_file):
    runs = tmp_path / "runs.jsonl"
    code = main(["train", "--set", f"data={synthetic_file}", "--set", "n_train=96", "--set", "widths=[4, 8]",
                 "--out", str(tmp_path / "run"), "--runs", str(runs)])
    assert code == 0, f"expected the exit code 0, got {code}"
    weights = tmp_path / "run" / "model.safetensors"
    assert weights.is_file(), f"expected the weights in {weights}"
    assert "0.weight" in load_file(weights), f"expected the weights of the model, got {sorted(load_file(weights))[:4]}"
    record = json.loads(runs.read_text(encoding="utf-8").splitlines()[-1])
    assert record["status"] == "ok", f"expected a run logged with the status ok, got {record['status']!r}"
'''


ok_15 = {}
PYTEST_ARGS_15 = ["-c", str(PROJECT / "pyproject.toml"), "--rootdir", str(PROJECT), "--strict-markers"]
ENV_15 = {"PYTHONPATH": str(PROJECT / "src")}
TESTS_15 = {"tests/conftest.py": CONFTEST_PY, "tests/test_model.py": TEST_MODEL_PY, "tests/test_data.py": TEST_DATA_PY,
            "tests/test_cli.py": TEST_CLI_PY}
if TEST_DATA_PY is ... or TEST_CLI_PY is ...:
    print("⏳ Ex B7.15 : écris d'abord TEST_DATA_PY et TEST_CLI_PY.")
else:
    with checked("B7.15"):
        wbp.write_files(PROJECT, TESTS_15)
        own_15 = {}
        for rel_15 in ("tests/test_model.py", "tests/test_data.py", "tests/test_cli.py"):
            print(f"— {rel_15}, sur ton paquet :")
            own_15[rel_15] = run_pytest_group(rel_15, root=PROJECT, env=ENV_15, args=PYTEST_ARGS_15)
        ok_15["own"] = all(r.ok for r in own_15.values())
        verdict("B7.15", ok_15["own"], "tes tests passent sur ton paquet.", "tes tests doivent d'abord passer sur ton "
                "paquet (les lignes ci-dessus) ; si c'est ton paquet qui a tort, corrige-le dans sa cellule.")
        collected_15 = {}
        for marker_15 in ("", "slow"):
            listing_15 = wbp.run([sys.executable, "-m", "pytest", "tests/test_data.py", "tests/test_cli.py", "--collect-only",
                                  "-q", "-p", "no:cacheprovider", *PYTEST_ARGS_15, *(["-m", marker_15] if marker_15 else [])],
                                 PROJECT, src=[PROJECT / "src"], env={"COLUMNS": "1000"})
            collected_15[marker_15] = sum("::" in line for line in listing_15.stdout.splitlines())
        ok_15["count"] = collected_15[""] >= 8 and collected_15["slow"] >= 1
        verdict("B7.15", ok_15["count"], f"{collected_15['']} tests dans test_data.py et test_cli.py (paramètres compris), "
                f"dont {collected_15['slow']} marqué(s) slow.", f"{collected_15['']} tests dans test_data.py et test_cli.py, "
                f"{collected_15['slow']} marqué(s) slow : attendu au moins 8 tests (paramètres compris), dont un marqué slow.")
        if not ok_15["own"]:
            raise StopCheck
        base_15 = scratch("mutants")
        reference_15 = wbp.run_tests(wbp.mutant_package(None, base_15 / "reference"), PROJECT / "tests", data=DATA_FILE)
        ok_15["reference"] = reference_15.ok
        verdict("B7.15", reference_15.ok, f"tes tests passent aussi sur le paquet de référence ({reference_15.summary}).",
                f"tes tests doivent passer sur le paquet de référence, qui suit le contrat ({reference_15.summary}) :")
        for name_15, reason_15 in reference_15.failures[:6]:
            print(f"   ❌ {name_15}\n      {reason_15[:300]}")
        if not reference_15.ok:
            print("   un test qui échoue ici exige plus (ou autre chose) que le contrat de B7.9 à B7.14.")
            raise StopCheck
        for k_15, mutant_15 in enumerate(wbp.MUTANTS, start=1):
            package_15 = wbp.mutant_package(mutant_15.key, base_15 / mutant_15.key)
            run_15 = wbp.run_tests(package_15, PROJECT / "tests", data=DATA_FILE,
                                   marker="slow" if mutant_15.slow else "not slow", maxfail=1)
            if run_15.failed + run_15.errors == 0:
                ok_15[mutant_15.key] = False
                print(f"❌ Ex B7.15 : version fautive {k_15} ({mutant_15.what}) : elle passe tous tes tests"
                      + (" marqués slow" if mutant_15.slow else "") + f" ({run_15.summary}). Quel test aurait dû la voir ?")
                continue
            name_15, reason_15 = run_15.failures[0] if run_15.failures else ("?", "")
            ok_15[mutant_15.key] = wbp.says_expected(reason_15)
            verdict("B7.15", ok_15[mutant_15.key], f"version fautive {k_15} ({mutant_15.what}) : attrapée par {name_15} "
                    f"(« {reason_15[:110]} »).", f"version fautive {k_15} ({mutant_15.what}) : attrapée par {name_15}, "
                    f"mais la première ligne de son message ne dit pas ce qui était attendu : « {reason_15[:160]} ».")
if ok_15 and all(ok_15.values()) and len(ok_15) == 3 + len(wbp.MUTANTS):
    print(f"✅ Ex B7.15 : tes tests passent sur un paquet juste et attrapent les {len(wbp.MUTANTS)} versions fautives.")

— tests/test_model.py, sur ton paquet :


pytest: 5 passed in 1.29s
— tests/test_data.py, sur ton paquet :


pytest: 11 passed in 0.09s
— tests/test_cli.py, sur ton paquet :


pytest: 13 passed in 0.94s
✅ Ex B7.15 : tes tests passent sur ton paquet.


✅ Ex B7.15 : 24 tests dans test_data.py et test_cli.py (paramètres compris), dont 1 marqué(s) slow.


✅ Ex B7.15 : tes tests passent aussi sur le paquet de référence (29 passed in 1.47s).


... (2 outputs left out)

✅ Ex B7.15 : version fautive 3 (load_npz(path, "val") rend des images d'entraînement : la validation recouvre l'entraînement) : attrapée par test_no_image_in_both_parts (« AssertionError: expected no validation image in the training part, got 500 »).


✅ Ex B7.15 : version fautive 4 (TrainConfig accepte un learning rate hors de ]0, 1[ (lr = 0 ou lr = 5)) : attrapée par test_invalid_config_refused[lr=0] (« Failed: DID NOT RAISE <class 'ValueError'> »).


✅ Ex B7.15 : version fautive 5 (main rend 0 (un succès) quand la configuration est invalide) : attrapée par test_exit_code_of_an_invalid_config (« AssertionError: expected the exit code 2 for epochs=0, got 0 »).


✅ Ex B7.15 : version fautive 6 (la commande train rend 0 sans écrire model.safetensors) : attrapée par test_smoke_train (« AssertionError: expected the weights in /tmp/pytest-of-root/pytest-1233/test_smoke_train0/run/model.safetensor »).
✅ Ex B7.15 : tes tests passent sur un paquet juste et attrapent les 6 versions fautives.


## Partie B · Le service : les écarts entre entraînement et production, l'artefact, l'API, la latence et l'export

Le modèle sort maintenant du projet pour servir. Tu commences par réparer le service d'un collègue, dont le modèle marche dans son notebook et pas en production (B7.16) : cinq écarts entre la préparation de l'entraînement et celle du service, le training-serving skew du ch. 12. Le modèle du collègue est le VGG 16-32-64 livré par la solution du MP5 (`model.pt` et `model.json`, versionnés avec le dépôt), entraîné sur 45 000 images : la partie B sert un vrai modèle, pas le petit CNN de la partie A.

### Ex B7.16 — Le modèle qui marche dans le notebook et pas en production : cinq écarts à retrouver 🐛 ★★ ⏱️ 30 min
**Objectif :** retrouver, par des diagnostics enchaînés, les écarts entre la préparation de l'entraînement et celle d'un service.  
**Prérequis :** Ex B7.15 · ch. 12 (training-serving skew) · ch. 20 (`eval()`) · fiche §207.7 (« L'artefact du modèle »), §207.10 · **Fil rouge :** CIFAR-10 · **Parcours :** C

Un collègue a entraîné le VGG du MP5 dans son notebook : 85 % sur ces images de test. Il l'a ensuite emballé en service : `save_artifact_16` écrit l'artefact à la fin de l'entraînement ; le service le recharge (`load_service_16`), lit chaque image envoyée par un client (`read_upload_16`, sur un fichier PNG), la prépare (`preprocess_16`) et répond le nom français de la classe (`predict_16`). **Ce que faisait son notebook**, la référence : des images `uint8` RGB `(N, 32, 32, 3)`, divisées par 255, normalisées canal par canal par la moyenne et l'écart-type de **ses images d'entraînement** (`INFO_16["mean"]`, `INFO_16["std"]`), en `float32`, le modèle en mode `eval()`, et la sortie $k$ du modèle est la classe `CLASSES[k]`. En production, le service répond n'importe quoi. **Répare son code sur place** (mêmes noms, mêmes signatures, les docstrings sont le contrat) : cinq écarts s'y cachent, et un écart peut en cacher un autre.

La cellule donnée prépare le modèle du collègue (`MODEL_16`, `INFO_16`, une copie neuve à chaque appel de `wbp.reference_model()`), 200 images de test (`X_16`, `Y_16`) écrites en PNG comme les enverrait un client (`PATHS_16`), ce que le notebook du collègue répond pour elles (`WANT_16`), et les cinq diagnostics. Chaque diagnostic écrit un **nouvel** artefact avec ton `save_artifact_16`, le recharge avec ton `load_service_16`, et ne regarde qu'une partie du service : 1. le service répond-il, et en quel type ; 2. l'image lue d'un fichier ; 3. le prétraitement, sur des images bien lues ; 4. le modèle rechargé, sur des images bien préparées ; 5. les noms rendus pour les 200 images, comparés à ceux du notebook (un **test de non-régression**, ch. 7), puis pour un modèle dont les sorties sont rangées dans un autre ordre. Ils s'enchaînent : le premier qui échoue arrête les suivants (⏳). La cellule suivante montre le service tel qu'il est sur 8 images, puis son accuracy sur les 200.

Dans tes notes : pour chaque écart, le symptôme, la cause, la ligne corrigée, et le test qui l'aurait attrapé avant la mise en production. Pourquoi certains écarts ne faisaient-ils **pas** planter le service ? Lequel en cachait un autre, et pourquoi (pense à ce que fait une normalisation par les statistiques du batch à des pixels de 0 à 255) ? Qu'aurait-il fallu mettre dans l'artefact dès le départ (207.7) ?

In [34]:
# The colleague's model (the VGG 16-32-64 delivered by the MP5 solution, trained in their notebook on uint8 RGB images
# divided by 255, then normalized with the mean and the std of its training images, in float32; output k = CLASSES[k]),
# 200 test images saved as the PNG files a client uploads, what the evaluation of the notebook gives on them, and the
# five diagnoses, in a row
from safetensors.torch import load_file, save_file

MODEL_16, INFO_16 = wbp.reference_model()               # a new copy at each call: the diagnoses never share it
X_16, Y_16 = wbp.test_images(20)                         # 200 test images (20 per class), uint8 RGB (200, 32, 32, 3)
UPLOADS_16 = scratch("uploads")
PATHS_16 = [wbp.write_png(image, UPLOADS_16 / f"upload_{i:03d}.png") for i, image in enumerate(X_16)]
MEAN_16, STD_16 = np.asarray(INFO_16["mean"], np.float32), np.asarray(INFO_16["std"], np.float32)
FRENCH_16 = {"airplane": "avion", "automobile": "automobile", "bird": "oiseau", "cat": "chat", "deer": "cerf",
             "dog": "chien", "frog": "grenouille", "horse": "cheval", "ship": "bateau", "truck": "camion"}


def offline_input_16(images):
    """The input of the model in the colleague's notebook: uint8 RGB (N, 32, 32, 3) -> float32 (N, 3, 32, 32)."""
    x = (np.asarray(images, np.float32) / np.float32(255) - MEAN_16) / STD_16
    return torch.from_numpy(np.ascontiguousarray(x.transpose(0, 3, 1, 2)))


with torch.inference_mode():
    OFFLINE_16 = MODEL_16(offline_input_16(X_16)).argmax(dim=1).numpy()      # the evaluation of the notebook
WANT_16 = [FRENCH_16[CLASSES[k]] for k in OFFLINE_16]                         # what the service should answer
print(f"Dans le notebook du collègue : accuracy {fr((OFFLINE_16 == Y_16).mean(), 3)} sur ces 200 images de test.")


def fresh_service_16(model=None, classes=CLASSES):
    """(model, meta) of the service, from a NEW artifact written by YOUR save_artifact_16 (the colleague's model, or
    `model` whose outputs are the classes `classes`), loaded by YOUR load_service_16."""
    folder = scratch("artifact")
    save_artifact_16(model if model is not None else wbp.reference_model()[0], folder, list(INFO_16["mean"]),
                     list(INFO_16["std"]), list(classes))
    return load_service_16(folder)


def diagnose_16(ex_id="B7.16"):
    """The five diagnoses of the service, in a row: the first one that fails stops the chain (one gap can hide
    another). Returns True when the five pass."""

    def point1():                                         # does the service answer?
        model, meta = fresh_service_16()
        seen = []
        handle = model.register_forward_pre_hook(lambda module, args: seen.append(args[0].dtype))
        try:
            names = predict_16(model, meta, PATHS_16[:8])
        except Exception as error:  # noqa: BLE001 - the learner's code: the symptom
            return False, f"le service plante sur 8 images envoyées : {type(error).__name__} : {str(error)[:160]}", \
                "lis la dernière ligne de l'erreur : quel tenseur ne va pas avec les poids du modèle ?"
        finally:
            handle.remove()
        if not (isinstance(names, list) and len(names) == 8 and all(isinstance(n, str) for n in names)):
            return False, f"predict_16 rend {type(names).__name__} {str(names)[:80]}", "attendu une liste de 8 noms (str)"
        weights = {p.dtype for p in model.parameters()}
        if weights != {torch.float32} or set(seen) != {torch.float32}:
            return False, (f"le modèle du service a des poids {sorted(map(str, weights))} et reçoit {[str(d) for d in seen]}, "
                           "alors que l'artefact et l'entraînement sont en float32"), "où ce type apparaît-il ?"
        return True, "le service répond 8 noms, en float32 comme l'entraînement", ""

    def point2():                                         # the image read from the file
        image = read_upload_16(PATHS_16[0])
        got = np.asarray(image)
        want = X_16[0].astype(np.float32) / 255
        if got.shape != (32, 32, 3):
            return False, f"l'image lue a la forme {got.shape}", "attendu (32, 32, 3), comme à l'entraînement"
        got = got.astype(np.float64)
        if np.allclose(got, want, atol=1e-6):
            return True, "l'image lue est celle de l'entraînement : RGB, des floats dans [0, 1]", ""
        symptoms, scaled = [], got
        if got.max() > 1.5:
            symptoms.append(f"ses valeurs vont de {got.min():.0f} à {got.max():.0f}, alors que l'entraînement voit des "
                            "pixels dans [0, 1]")
            scaled = got / 255
        if not np.allclose(scaled, want, atol=1e-6) and np.allclose(scaled, want[..., ::-1], atol=1e-6):
            symptoms.append("son canal 0 est le canal 2 de l'image envoyée (le bleu à la place du rouge)")
        elif not np.allclose(scaled, want, atol=1e-6):
            symptoms.append(f"elle s'écarte de l'image envoyée de {np.abs(scaled - want).max():.3f}")
        return False, "l'image lue n'est pas celle que voit l'entraînement : " + " ; ".join(symptoms), \
            "compare-la à X_16[0] / 255 : que rend la fonction de lecture ?"

    def point3():                                         # the preprocessing
        model, meta = fresh_service_16()
        images = X_16[:8].astype(np.float32) / 255
        batch = torch.as_tensor(preprocess_16(images.copy(), meta)).double()
        if tuple(batch.shape) != (8, 3, 32, 32):
            return False, f"preprocess_16 de 8 images rend la forme {tuple(batch.shape)}", "attendu (8, 3, 32, 32)"
        alone = torch.as_tensor(preprocess_16(images[:1].copy(), meta)).double()
        want = offline_input_16(X_16[:8]).double()
        if not torch.allclose(alone[0], batch[0], atol=1e-5):
            return False, (f"une image prétraitée seule n'a pas les mêmes valeurs que dans un batch de 8 (écart "
                           f"{float((alone[0] - batch[0]).abs().max()):.3f}) : son prétraitement dépend des autres images "
                           "envoyées"), "d'où viennent la moyenne et l'écart-type qu'utilise le service ?"
        if not torch.allclose(batch, want, atol=1e-4):
            return False, (f"le prétraitement du service s'écarte de celui de l'entraînement de "
                           f"{float((batch - want).abs().max()):.3f}"), "relis comment le notebook préparait les images"
        return True, "le prétraitement est celui de l'entraînement, image par image", ""

    def point4():                                         # the model of the service
        model, meta = fresh_service_16()
        x = offline_input_16(X_16[:8])
        before = {k: v.clone() for k, v in model.state_dict().items()}
        with torch.inference_mode():
            together, alone = model(x), model(x[:1])
        changed = any(not torch.equal(before[k], v) for k, v in model.state_dict().items())
        if changed or not torch.allclose(together[:1], alone, atol=1e-5):
            return False, ("la même image, bien prétraitée, reçoit d'autres scores seule et dans un batch de 8 (écart "
                           f"{float((together[:1] - alone).abs().max()):.3f})"
                           + (", et chaque appel change les statistiques internes du modèle (running_mean)" if changed else "")), \
                "compare le modèle que rend load_service_16 à celui de l'évaluation hors ligne : qu'est-ce qui les distingue ?"
        with torch.inference_mode():
            want = MODEL_16(x)
        if not torch.allclose(together, want, atol=1e-4):
            return False, f"les scores du service s'écartent de ceux du notebook de {float((together - want).abs().max()):.3f}", \
                "les poids chargés sont-ils ceux de l'artefact ?"
        return True, "le modèle du service donne les scores du notebook, seule ou en batch", ""

    def point5():                                         # the names returned: the non-regression test
        model, meta = fresh_service_16()
        names = predict_16(model, meta, PATHS_16)
        agree = sum(a == b for a, b in zip(names, WANT_16))
        if agree != len(WANT_16):
            pairs = {}
            for got, want in zip(names, WANT_16):
                if got != want:
                    pairs[(want, got)] = pairs.get((want, got), 0) + 1
            (want, got), count = max(pairs.items(), key=lambda item: item[1])
            return False, (f"sur 200 images, le service ne rend le nom du notebook que pour {agree} ; par exemple, "
                           f"{count} images que le notebook classe « {want} » reçoivent « {got} »"), \
                "le nom que rend le service correspond-il à la sortie du modèle ?"
        perm = np.random.default_rng(16).permutation(10)
        shuffled = wbp.reference_model()[0]
        with torch.no_grad():
            shuffled[-1].weight.copy_(shuffled[-1].weight[perm].clone())
            shuffled[-1].bias.copy_(shuffled[-1].bias[perm].clone())
        model, meta = fresh_service_16(shuffled, [CLASSES[k] for k in perm])
        agree = sum(a == b for a, b in zip(predict_16(model, meta, PATHS_16), WANT_16))
        if agree != len(WANT_16):
            return False, (f"le même modèle, ses sorties rangées dans un autre ordre (et classes dans cet ordre), ne "
                           f"reçoit le bon nom que pour {agree} images sur 200"), \
                "d'où le service tient-il l'ordre des classes ?"
        return True, ("les 200 noms sont ceux du notebook, même pour un modèle dont les sorties sont rangées dans un "
                      "autre ordre"), ""

    points = [("le service répond", point1), ("l'image lue", point2), ("le prétraitement", point3),
              ("le modèle du service", point4), ("les noms rendus (test de non-régression)", point5)]
    for k, (name, run) in enumerate(points, start=1):
        try:
            with wbp.torch_state_kept():
                ok, measured, why = run()
        except NotImplementedError:
            raise
        except Exception as error:  # noqa: BLE001 - the learner's code: one ❌ line, then the chain stops
            ok, measured, why = False, f"une erreur {type(error).__name__} : {str(error)[:200]}", "relis la fonction en cause"
        if ok:
            print(f"✅ Ex {ex_id} : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex {ex_id} : point {k}, {name} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex {ex_id} : point {later} : corrige d'abord le point {k}.")
            return False
    print(f"✅ Ex {ex_id} : les cinq diagnostics passent : le service répond comme le notebook.")
    return True

Dans le notebook du collègue : accuracy 0,870 sur ces 200 images de test.


In [35]:
# The colleague's service, repaired: five changes, numbered in the comments
def save_artifact_16(model, folder, mean, std, classes):
    """Training side: the artifact of `model` in `folder`: model.safetensors (the weights) and meta.json (what the
    service needs). mean, std: the normalization learned on the training images; classes: the names of the outputs,
    in their order."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    save_file(model.state_dict(), str(folder / "model.safetensors"))
    meta = {"arch": "vgg", "widths": [16, 32, 64], "mean": list(mean), "std": list(std),
            "classes": list(classes)}                             # 3. the order of the outputs
    (folder / "meta.json").write_text(json.dumps(meta), encoding="utf-8")


def load_service_16(folder):
    """Service side: (model, meta), the model rebuilt from the artifact, ready to predict."""
    meta = json.loads((Path(folder) / "meta.json").read_text(encoding="utf-8"))
    model = wbp.build_reference(meta["arch"], meta["widths"])
    model.load_state_dict(load_file(str(Path(folder) / "model.safetensors")))
    model.eval()                                                  # 2. the evaluation mode
    return model, meta


def read_upload_16(path):
    """Service side: an uploaded image file, as the array the model was trained on."""
    return wbp.imread_bgr(path)[..., ::-1].astype(np.float32) / 255   # 4. RGB, floats in [0, 1]


def preprocess_16(images, meta):
    """Service side: the images (N, 32, 32, 3) read by read_upload_16, as the input tensor (N, 3, 32, 32) of the model."""
    x = np.asarray(images, dtype=np.float32)                      # 5. float32, as in training
    mean = np.asarray(meta["mean"], dtype=x.dtype)                # 1. the statistics of the
    std = np.asarray(meta["std"], dtype=x.dtype)                  #    training, from the artifact
    x = (x - mean) / std
    return torch.from_numpy(x).permute(0, 3, 1, 2)


def predict_16(model, meta, paths):
    """Service side: the French name of the class of each uploaded image (a list of str)."""
    images = np.stack([read_upload_16(p) for p in paths])
    with torch.inference_mode():
        k = model(preprocess_16(images, meta)).argmax(dim=1)
    return [FRENCH_16[meta["classes"][i]] for i in k.tolist()]   # 3. the order of the artifact


repaired_16_ok = False
with checked("B7.16"):
    repaired_16_ok = diagnose_16()

✅ Ex B7.16 : point 1, le service répond (le service répond 8 noms, en float32 comme l'entraînement).
✅ Ex B7.16 : point 2, l'image lue (l'image lue est celle de l'entraînement : RGB, des floats dans [0, 1]).
✅ Ex B7.16 : point 3, le prétraitement (le prétraitement est celui de l'entraînement, image par image).
✅ Ex B7.16 : point 4, le modèle du service (le modèle du service donne les scores du notebook, seule ou en batch).


✅ Ex B7.16 : point 5, les noms rendus (test de non-régression) (les 200 noms sont ceux du notebook, même pour un modèle dont les sorties sont rangées dans un autre ordre).
✅ Ex B7.16 : les cinq diagnostics passent : le service répond comme le notebook.


In [36]:
# The service, as it is now (it shows the symptoms before the repair), on 8 uploads and on the 200 images
try:
    with wbp.torch_state_kept():
        model_16, meta_16 = fresh_service_16()
        names_16 = predict_16(model_16, meta_16, PATHS_16)
except Exception as error_16:  # noqa: BLE001 - the learner's code
    names_16 = None
    print(f"⏳ Ex B7.16 : le service lève {type(error_16).__name__} ({str(error_16)[:160]}) : rien à montrer.")
if isinstance(names_16, list) and len(names_16) == len(PATHS_16):
    print("image   vraie classe   notebook   service")
    for i_16 in range(8):
        print(f"{i_16:5d}   {FRENCH_16[CLASSES[Y_16[i_16]]]:12s}   {WANT_16[i_16]:10s} {names_16[i_16]}")
    truth_16 = [FRENCH_16[CLASSES[k]] for k in Y_16]
    print(f"accuracy sur 200 images : notebook {fr((OFFLINE_16 == Y_16).mean(), 3)}, service "
          f"{fr(np.mean([a == b for a, b in zip(names_16, truth_16)]), 3)} ; accord service-notebook "
          f"{sum(a == b for a, b in zip(names_16, WANT_16))} sur 200.")

image   vraie classe   notebook   service
    0   grenouille     grenouille grenouille
    1   camion         camion     camion
    2   grenouille     grenouille grenouille
    3   grenouille     grenouille grenouille
    4   oiseau         oiseau     oiseau
    5   oiseau         oiseau     oiseau
    6   chien          chien      chien
    7   chat           chat       chat
accuracy sur 200 images : notebook 0,870, service 0,870 ; accord service-notebook 200 sur 200.


## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire où va chaque fichier d'un projet de ML (code, tests, configuration, données) et pourquoi un notebook ne suffit pas ?
2. Sais-tu rapporter une accuracy sur cinq graines avec son intervalle, et comparer deux modèles graine par graine ?
3. Sais-tu calculer un p50, un p95 et un débit, et dimensionner un service à la louche ?
4. Sais-tu calculer un PSI, dire ce qu'il mesure, et ce que tu ferais d'une alerte ?

**Pour aller plus loin** : l'article de Sculley et coll. (📄 B7.8) et le ML Test Score de Breck et coll., cités dans la fiche. La suite : le bonus B8, l'apprentissage par renforcement moderne.

*Les exercices B7.17 à B7.21 et B7.22 à B7.25 seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B7` pour obtenir le notebook complet.*